# v4 + v5 – CAGS: loại bỏ Gaussian bù trừ trắc quang trong quá trình huấn luyện

Notebook **tự vá code trong runtime** và chỉ dùng dữ liệu đã có trên Drive: 15 scan DTU, điểm SfM `sfm_v8.npy`, bề mặt
tham chiếu lọc chặt `ref_points_strict80.npy`. Mọi run dùng pipeline **3DGS chuẩn** của v3 (gsplat, SH bậc 3, khởi tạo
SfM, 10 000 vòng, mật độ hoá 500–5000, 8 view huấn luyện). Run mới có tiền tố `runs/v5_…`, kết quả ở `{STORE}/results/v5`.

**Bốn điều kiện trắc quang** (chỉ áp lên ảnh huấn luyện, trong mặt nạ vật thể):

| Mức | Mô tả | GC-ILA mô hình hoá được? |
|---|---|---|
| `none` | ảnh sạch | – |
| `pa` | P-A: gain, cân bằng trắng, gamma toàn cục theo view (= `strong` của v3) | có (affine trong miền log) |
| `pb` | P-B: trường nhân mượt thay đổi theo không gian (đa thức bậc 2 trung bình 0 + vignetting), ±1 stop | không |
| `pc` | P-C: 3–6 đốm sáng/tối cục bộ (Gaussian 2D), ±0.5–1 stop, có lệch màu nhẹ | không |
| `pac` | P-AC: P-A rồi P-C chồng lên (lệch toàn cục + cục bộ); GC-ILA chỉ khử được phần toàn cục | một phần |

P-B/P-C là **nhiễu tổng hợp thay đổi theo không gian** để kiểm tra độ vững, không phải mô hình vật lý của bóng đổ hay phản xạ.

**CAGS** (tên tạm). Mỗi `cags_every` vòng, một lượt *probe* (một lần render + một backward cho mỗi view huấn luyện) tính
31 đặc trưng không dùng GT cho từng Gaussian, chia 4 nhóm: A thuộc tính, G hỗ trợ hình học đa góc nhìn (kể cả ngữ cảnh
k-NN), P bù trừ trắc quang (phần dư trong footprint, xung đột gradient màu DC/SH theo view), D động học tối ưu
(tuổi, phả hệ clone/split, biến thiên độ mờ/kích thước). Detector (MLP) cho hai đầu ra
p_geo (ngoài bề mặt) và δ̂ (mức bù trừ, đơn vị τ_δ); rủi ro **q = p_geo · σ((δ̂ − 1)/T)**, làm mượt bằng EMA và di truyền
cho Gaussian con. Ba hành động, bật riêng hoặc cùng nhau:
- `gate`: không clone/split Gaussian có rủi ro > τ_gate (giai đoạn *sinh mới*);
- `att`: trần độ mờ α ≤ (1 − m)^γ áp **mỗi vòng** (không tích luỹ, Adam không kéo lên lại được) (giai đoạn *củng cố*);
- `mod`: điều biến **bước cập nhật** sau Adam cho màu/SH/kích thước: θ ← θ_cũ + (1 − m)(θ_Adam − θ_cũ);
  không đụng vị trí.

với m = clip((r − τ_s)/(1 − τ_s), 0, 1), chỉ cho tối đa `cags_budget` phần Gaussian có rủi ro cao nhất.

**Nhãn** (chỉ để huấn luyện detector và cho oracle; GT không bao giờ điều khiển hành động của phương pháp):
- y_geo = 1 nếu d(μ, STL) > 0.1 (như W_float, bất kể kích thước); y_geo = 0 nếu d < 0.05, hoặc d ≤ 0.1 và d / σ_n < 1
  (σ_n: độ lệch chuẩn của Gaussian theo pháp tuyến bề mặt tham chiếu); còn lại là vùng bất định, bỏ khỏi loss phân loại. Đơn vị: hệ toạ độ chuẩn hoá
  của DTU loader (vật thể trong hình cầu đơn vị), cùng ngưỡng 0.1 với W_float.
- δ = ‖ Σ_j ŵ_ij · log φ̄_ij − mean_{j hỗ trợ hình học} log φ_j(π_j(μ_i)) ‖, log φ = log I' − log I (biết vì nhiễu do ta
  đưa vào), log φ̄_ij lấy trung bình có trọng số trên **footprint** của Gaussian; view "hỗ trợ hình học" = tâm Gaussian nằm
  trong ảnh và không bị bề mặt kết xuất che. τ_δ = phân vị 90 của δ trên Gaussian gần bề mặt cùng thời điểm.
  y_joint = y_geo = 1 và δ > τ_δ.

**Oracle** (trần trên, chẩn đoán): `oracle_geo` (rủi ro = y_geo) và `oracle_cmp` (rủi ro = y_joint, bù trừ có ý thức).

| Phần | Nội dung | Lượt (ước lượng) |
|---|---|---|
| A (v5a) | 3DGS gốc có ghi đặc trưng + nhãn tại 10 mốc; 15 scan × 4 mức × 2 seed; lưu checkpoint fork 100/250/500 cho scan63, 83 | 120 |
| B (v4) | Oracle can thiệp: 2 scan × {none, pb, pc} × fork tại 100/250/500 × {ctl, geo: gate/att/mod/full, cmp: full} | 108 |
| C (v5b) | Detector MLP, 5 fold leave-scan-out; ablation A → AG → AGP → AGPD; heuristic; cross-corruption | vài phút |
| D (v5c) | 15 scan × {none, pc, pac} × 2 seed × {GC-ILA, CAGS, GC-ILA + CAGS, oracle_cmp} | 360 |
| E | Ablation trên 4 scan: từng hành động, detector chỉ hình học, bỏ đầu bù trừ, heuristic, oracle_geo, chu kỳ probe 500, mức P-A | 224 |
| F | Một vòng DAgger: huấn luyện lại với trạng thái thu từ run CAGS, chạy lại D cho CAGS | 180 |

Mỗi lượt khoảng 6–9 phút trên GPU (probe thêm khoảng 5–10%). Hàng đợi **chạy lại được**: run đã có `base.pt` được bỏ
qua, nên hết runtime chỉ cần chạy lại từ cell 1, 2, 4 rồi tiếp tục phần đang dở.

**Tiêu chí đặt trước** (mục 7):
- *B → tiếp tục*: `oracle_cmp` (full, mốc tốt nhất) giảm ΔW ≥ 30% so với ctl cùng mốc ở pb **và** pc, và F-score ở pb, pc
  không thấp hơn ctl quá 0.02. Nếu không đạt: CAGS không có trần để đạt, dừng trước D và báo cáo như phát hiện cơ chế.
- *C → tiếp tục*: AP_joint của AGPD cao hơn AG (chỉ hình học) và cao hơn heuristic ở cả pc và pac (trung bình 5 fold).
- *D*: T1 CAGS đạt ≥ 50% mức giảm ΔW của oracle_cmp (trung vị trên scan) ở pc và pac; T2 PSNR mức none giảm ≤ 0.3 dB
  (trung vị); T3 CAGS < geometry-only và < heuristic về ΔW (Wilcoxon cặp theo scan, p < 0.05, ở mục E); T4 GC-ILA + CAGS
  tốt hơn từng thành phần ở pc/pac. (P-B bị loại khỏi D: ở v4, ΔW_ctl ≈ 0.01, không đủ floater để đo.)

Thứ tự: 1 → 2 → 3 → 4 → 5A → 5B → 6 → 7a (chốt tham số ở cell 4b) → 5C → 7b → 5D → 6 → 7c → 5E/5F → 6 → 7d.

## 1. Setup

In [ ]:
import os, subprocess, pathlib, shutil, sys
from google.colab import drive, userdata
drive.mount('/content/drive')
STORE = '/content/drive/MyDrive/faithfulrecon_store'
REPO_URL = 'github.com/Huyrua27/faithfulrecon.git'
BRANCH = 'feature/huy-dtu-multiscan'
REPO = '/content/faithfulrecon'
token = userdata.get('GH_TOKEN')
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@{REPO_URL}', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', f'https://{token}@{REPO_URL}', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-f', '-B', BRANCH, 'FETCH_HEAD'], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
for d in ('runs', 'data'):
    p = pathlib.Path(REPO, d)
    if p.is_symlink(): p.unlink()
    elif p.exists(): shutil.rmtree(p)
    p.symlink_to(f'{STORE}/{d}')
os.chdir(REPO); sys.path.insert(0, REPO); sys.path.insert(0, f'{REPO}/scripts')
RES = f'{STORE}/results/v5'; os.makedirs(f'{RES}/det', exist_ok=True)
!git log --oneline -1
!pip -q install -r requirements.txt scikit-learn ninja gsplat
!nvidia-smi --query-gpu=name,memory.total --format=csv; nproc; free -g | head -2

## 2. Vá code trong runtime (chạy lại sau mỗi lần khởi động lại runtime)
Cell cuối phải in đủ `PATCH 1 OK`, `E5 patch OK`, `E6 patch OK`, `PATCH 4 OK`, `PATCH 5 OK`, `PATCH 6 OK`, `PATCH 7 OK`,
`PATCH 8 OK` (tự kiểm tra trên CPU khoảng 1–2 phút).

In [ ]:
%%writefile scripts/floater_metrics.py
"""Floater and geometry metrics of trained DTU base models (one row per run_dir x seed).

python scripts/floater_metrics.py --run_dirs runs/dtu_scan24_v8 runs/photo_strong_dtu_scan24_v8 \
    --seeds 0 1 2 --out results/photometric/floater_metrics.csv

Metrics (normalized units; d = distance from a Gaussian mean to the nearest reference point):
  off_003, off_01   fraction of Gaussians with d > 0.03 / d > 0.1
  front_01          fraction of Gaussians with d > 0.1 that lie IN FRONT of the reference surface
                    (median over training views of z_gauss - z_surface < -tol)
  wfloat_01         share of held-out blending weight carried by Gaussians with d > 0.1
                    (the floaters that actually render)
  acc, comp         mean distance Gaussian -> reference and reference -> Gaussian
                    (Gaussians with opacity > 0.05)
  psnr, dssim       held-out views, ORIGINAL (unperturbed) appearance
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from frecon.camera import farthest_point_indices  # noqa: E402
from frecon.data.dtu import load_dtu_views, reference_points, split_views  # noqa: E402
from frecon.gaussians import nearest_dist  # noqa: E402
from frecon.measure import heldout_errors, render_views  # noqa: E402
from frecon.render import get_rasterizer  # noqa: E402
from frecon.train import TrainConfig, Trainer  # noqa: E402


def project(P, cam):
    Xc = P @ cam.R.T + cam.t
    z = Xc[:, 2]
    u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
    v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
    ok = (z > 1e-4) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
    return z, u.clamp(0, cam.W - 1), v.clamp(0, cam.H - 1), ok


def zbuffer(ref, cam):
    z, u, v, ok = project(ref, cam)
    zb = torch.full((cam.H * cam.W,), float("inf"), device=ref.device)
    zb = zb.scatter_reduce(0, v[ok] * cam.W + u[ok], z[ok], "amin")
    return -F.max_pool2d(-zb.view(1, 1, cam.H, cam.W), 3, 1, 1)[0, 0]   # close small holes


@torch.no_grad()
def depth_side(means, views, tol):
    diffs = []
    for cam, mask, zb in views:
        z, u, v, ok = project(means, cam)
        zs = zb[v, u]
        ok &= mask[v, u] & torch.isfinite(zs)
        d = torch.full_like(z, float("nan"))
        d[ok] = z[ok] - zs[ok]
        diffs.append(d)
    return torch.nanmedian(torch.stack(diffs, 1), 1).values   # nan where no valid view


def split_eval(cams, n_train: int, n_test: int, test_pool: int = 0):
    """Training views as in training (FPS, nested in N). With test_pool > 0 the held-out views are
    taken outside the first max(n_train, test_pool) FPS cameras, so runs with any N <= test_pool share
    the same held-out views."""
    if test_pool <= 0:
        return split_views(cams, n_train, n_test)
    centers = torch.stack([c.center for c in cams])
    tr = farthest_point_indices(centers, n_train)
    pool = farthest_point_indices(centers, max(n_train, test_pool))
    rest = [i for i in range(len(cams)) if i not in pool]
    step = max(len(rest) // max(n_test, 1), 1)
    return tr, rest[::step][:n_test]


def strict_reference(sc, cams, masks, scale_mat, frac: float, seed: int = 0) -> torch.Tensor:
    """Reference points that lie inside the object mask in at least `frac` of the views whose image
    they project into (the default crop keeps points inside the mask of >= 1 view, which retains
    table / background points). Cached next to the default cache."""
    cache = Path(sc["ref_cache"]).with_name(f"ref_points_strict{int(round(frac * 100))}.npy")
    if cache.exists():
        return torch.from_numpy(np.load(cache)).float()
    from frecon.data.blender import _read_ply_xyz
    pts_w = _read_ply_xyz(Path(sc["stl_ply"])).numpy().astype(np.float64)
    inv = np.linalg.inv(scale_mat)
    pts = (pts_w @ inv[:3, :3].T) + inv[:3, 3]
    pts = pts[np.linalg.norm(pts, axis=1) < 1.5]
    n_in, n_frame = np.zeros(len(pts), np.int32), np.zeros(len(pts), np.int32)
    for cam, m in zip(cams, masks):
        R, t = cam.R.numpy().astype(np.float64), cam.t.numpy().astype(np.float64)
        pc = pts @ R.T + t
        z = pc[:, 2]
        ok = z > 1e-6
        u = np.floor(cam.fx * pc[:, 0] / np.where(ok, z, 1) + cam.cx).astype(np.int64)
        v = np.floor(cam.fy * pc[:, 1] / np.where(ok, z, 1) + cam.cy).astype(np.int64)
        inb = ok & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
        hit = np.zeros(len(pts), dtype=bool)
        hit[inb] = m[v[inb], u[inb]]
        n_frame += inb
        n_in += hit
    keep = (n_frame > 0) & (n_in >= frac * n_frame)
    print(f"strict reference ({frac:.2f}): kept {keep.sum()}/{len(pts)} points", flush=True)
    pts = pts[keep]
    mx = sc.get("max_ref_points", 200_000)
    if len(pts) > mx:
        pts = pts[np.random.default_rng(seed).choice(len(pts), mx, replace=False)]
    pts = pts.astype(np.float32)
    np.save(cache, pts)
    return torch.from_numpy(pts)


def load_eval_scene(cfg, dev, ref_mode: str = "default", strict_frac: float = 0.8, test_pool: int = 0):
    sc = cfg["scene"]
    assert sc.get("type") == "dtu", "floater_metrics.py supports DTU scenes only"
    cams, masks, scale_mat = load_dtu_views(sc["root"], sc.get("downscale", 4))
    tr, te = split_eval(cams, sc["n_train_views"], sc["n_test_views"], test_pool)
    if ref_mode == "strict":
        ref = strict_reference(sc, cams, masks, scale_mat, strict_frac).to(dev)
    else:
        ref = reference_points(sc.get("stl_ply"), scale_mat, cams, masks, sc.get("min_mask_views", 1),
                               sc.get("max_ref_points", 200_000), sc.get("ref_cache")).to(dev)
    views = []
    for i in tr:
        cam = cams[i].to(dev)
        views.append((cam, torch.from_numpy(masks[i]).to(dev), zbuffer(ref, cam)))
    return views, [cams[i].to(dev) for i in te], ref


def tau_tag(tau: float) -> str:
    """0.03 -> '003', 0.1 -> '01', 0.15 -> '015' (keeps the historical column names)."""
    return f"{tau:g}".replace("0.", "0", 1)


@torch.no_grad()
def metrics(model, views, test_cams, ref, rasterize, bg, tol, taus=(0.03, 0.05, 0.1, 0.15)):
    """W_float(tau) = sum_{k,p} sum_{i: d_i > tau} w_ik(p) / sum_{k,p} sum_i w_ik(p) over all pixels of
    the held-out views (torch renderer: exact blending weights, contributions < 1/255 dropped)."""
    means = model.means.detach()
    opac = model.opacities.detach().reshape(-1)
    d = nearest_dist(means, ref)
    med = depth_side(means, views, tol)
    contrib = torch.zeros(model.n, device=means.device)
    for cam in test_cams:
        contrib += model.render(rasterize, cam, bg, return_contrib=True)["contrib"]
    solid = opac > 0.05
    err = heldout_errors(render_views(model, test_cams, rasterize, bg), test_cams)
    out = {"n_gaussians": model.n}
    for tau in taus:
        far = d > tau
        t = tau_tag(tau)
        out[f"off_{t}"] = float(far.float().mean())
        out[f"wfloat_{t}"] = float(contrib[far].sum() / contrib.sum().clamp_min(1e-12))
        out[f"n_off_{t}"] = int(far.sum())                      # count, not a fraction
        out[f"opsum_off_{t}"] = float(opac[far].sum())          # total opacity of far Gaussians
        out[f"contrib_off_{t}"] = float(contrib[far].sum())     # absolute held-out weight
    off01 = d > 0.1
    out.update({
        "front_01": float(((med < -tol) & off01).sum() / off01.sum().clamp_min(1)),
        "contrib_total": float(contrib.sum()),
        "acc": float(d[solid].mean()),
        "comp": float(nearest_dist(ref, means[solid]).mean()),
        "psnr": err["psnr"], "dssim": err["dssim"],
    })
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run_dirs", nargs="+", required=True)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--tol", type=float, default=0.03)
    ap.add_argument("--taus", type=float, nargs="+", default=[0.03, 0.05, 0.1, 0.15],
                    help="distance thresholds (normalized units, scene ~ unit sphere) for the sweep")
    ap.add_argument("--out", default="results/photometric/floater_metrics.csv")
    ap.add_argument("--ref_mode", choices=["default", "strict"], default="default",
                    help="strict: keep reference points inside the mask in >= --strict_frac of the views "
                         "they project into")
    ap.add_argument("--strict_frac", type=float, default=0.8)
    ap.add_argument("--test_pool", type=int, default=0,
                    help="> 0: held-out views outside the first max(N, test_pool) FPS cameras (same views "
                         "for every N <= test_pool)")
    args = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    rows, cache = [], {}
    for rd in map(Path, args.run_dirs):
        cfg = yaml.safe_load((rd / "config.yaml").read_text(encoding="utf-8"))
        rasterize = get_rasterizer(cfg.get("renderer", "torch"))
        bg = torch.ones(3, device=dev)
        key = (cfg["scene"]["root"], cfg["scene"]["n_train_views"])
        if key not in cache:   # evaluation scene ignores `photometric`: original appearance
            cache = {key: load_eval_scene(cfg, dev, args.ref_mode, args.strict_frac, args.test_pool)}
        views, test_cams, ref = cache[key]
        photo = cfg["scene"].get("photometric") or {}
        for seed in args.seeds:
            ckpt = rd / f"seed_{seed}" / "base.pt"
            if not ckpt.exists():
                print(f"skip {ckpt} (missing)")
                continue
            model = Trainer.load(ckpt, TrainConfig(**cfg["train"]), rasterize, bg, dev).model
            tc, ic = cfg.get("train", {}), cfg.get("init", {})
            row = {"run": rd.name, "scan": Path(key[0]).name, "seed": seed, "N": key[1],
                   "exposure_stops": photo.get("exposure_stops", 0.0), "wb": photo.get("wb", 0.0),
                   "gamma": photo.get("gamma", 0.0), "photo_seed": photo.get("seed", 0),
                   "photo_mode": photo.get("mode", "isp"), "appearance": tc.get("appearance", "none"),
                   "lr_appearance": tc.get("lr_appearance", 1e-3), "sh_degree": tc.get("sh_degree", 0),
                   "cap": bool(tc.get("cap_scale_after_densify", False)), "init": ic.get("type", "random"),
                   "ref_mode": args.ref_mode, "test_pool": args.test_pool}
            row.update(metrics(model, views, test_cams, ref, rasterize, bg, args.tol, args.taus))
            rows.append(row)
            print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()}, flush=True)
            del model
            if dev == "cuda":
                torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(args.out, index=False)
    print(f"-> {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/sfm_tri.py
"""Sparse points for SfM-style initialization from the CLEAN training views with known poses.

Replaces COLMAP's point_triangulator (features + matching + triangulation with fixed poses):
SIFT inside the object masks -> ratio-test matching for every view pair -> epipolar check with the
known fundamental matrix -> DLT triangulation -> cheirality, reprojection and triangulation-angle checks
-> voxel de-duplication. Positions are identical for every photometric level of a (scan, N) pair, so
paired comparisons isolate the effect of the perturbation on 3DGS optimization ("clean-SfM" init).

python scripts/sfm_tri.py --root data/dtu/scan63 --n_train 8 --out data/dtu/scan63/sfm_v8.npy
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import cv2
import numpy as np
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))


def _skew(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])


def fundamental(K1, R1, t1, K2, R2, t2):
    R = R2 @ R1.T
    t = t2 - R @ t1
    return np.linalg.inv(K2).T @ (_skew(t) @ R) @ np.linalg.inv(K1)


def sampson(F, x1, x2):
    h1 = np.hstack([x1, np.ones((len(x1), 1))])
    h2 = np.hstack([x2, np.ones((len(x2), 1))])
    Fx1, Ftx2 = h1 @ F.T, h2 @ F
    num = np.sum(h2 * Fx1, 1) ** 2
    return num / (Fx1[:, 0] ** 2 + Fx1[:, 1] ** 2 + Ftx2[:, 0] ** 2 + Ftx2[:, 1] ** 2 + 1e-12)


def triangulate_views(images, masks, Ks, Rs, ts, ratio=0.75, max_epi=1.5, max_reproj=1.5,
                      min_angle_deg=2.0, voxel=0.004, n_features=8000):
    """images: (H, W, 3) float in [0, 1]; masks: (H, W) bool; K, R, t in the convention of frecon.camera
    (pixel centres at index + 0.5). Returns (M, 3) float32 points."""
    sift = cv2.SIFT_create(nfeatures=n_features)
    feats = []
    for img, m in zip(images, masks):
        g = cv2.cvtColor((np.clip(img, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        kp, des = sift.detectAndCompute(g, m.astype(np.uint8) * 255)
        xy = np.float64([k.pt for k in kp]).reshape(-1, 2) + 0.5     # OpenCV pixel centres -> ours
        feats.append((xy, des))
    P = [K @ np.hstack([R, t[:, None]]) for K, R, t in zip(Ks, Rs, ts)]
    C = [-R.T @ t for R, t in zip(Rs, ts)]
    bf = cv2.BFMatcher(cv2.NORM_L2)
    out, n = [], len(images)
    for i in range(n):
        for j in range(i + 1, n):
            (xi, di), (xj, dj) = feats[i], feats[j]
            if di is None or dj is None or len(di) < 2 or len(dj) < 2:
                continue
            good = [a for a, b in (p for p in bf.knnMatch(di, dj, k=2) if len(p) == 2) if a.distance < ratio * b.distance]
            if len(good) < 8:
                continue
            x1 = xi[[g.queryIdx for g in good]]
            x2 = xj[[g.trainIdx for g in good]]
            F = fundamental(Ks[i], Rs[i], ts[i], Ks[j], Rs[j], ts[j])
            ok = sampson(F, x1, x2) < max_epi ** 2
            x1, x2 = x1[ok], x2[ok]
            if len(x1) == 0:
                continue
            X = cv2.triangulatePoints(P[i], P[j], x1.T, x2.T)
            X = (X[:3] / X[3]).T
            keep = np.ones(len(X), bool)
            for Pk, xk, Rk, tk in ((P[i], x1, Rs[i], ts[i]), (P[j], x2, Rs[j], ts[j])):
                z = X @ Rk[2] + tk[2]
                pr = np.hstack([X, np.ones((len(X), 1))]) @ Pk.T
                err = np.linalg.norm(pr[:, :2] / pr[:, 2:3] - xk, axis=1)
                keep &= (z > 1e-6) & (err < max_reproj)
            r1, r2 = X - C[i], X - C[j]
            cosang = np.sum(r1 * r2, 1) / (np.linalg.norm(r1, axis=1) * np.linalg.norm(r2, axis=1) + 1e-12)
            keep &= cosang < np.cos(np.radians(min_angle_deg))
            out.append(X[keep])
    if not out:
        return np.zeros((0, 3), np.float32)
    X = np.concatenate(out)
    X = X[np.linalg.norm(X, axis=1) < 1.5]
    _, idx = np.unique(np.floor(X / voxel).astype(np.int64), axis=0, return_index=True)
    return X[np.sort(idx)].astype(np.float32)


def scan_points(root: str, n_train: int, downscale: int = 2):
    """Triangulate from the FPS training views of a DTU scan (same views as training, clean images)."""
    from frecon.camera import farthest_point_indices
    from frecon.data.dtu import load_dtu_views
    cams, masks, _ = load_dtu_views(root, downscale)
    tr = farthest_point_indices(torch.stack([c.center for c in cams]), n_train)
    Ks = [np.array([[cams[i].fx, 0, cams[i].cx], [0, cams[i].fy, cams[i].cy], [0, 0, 1]]) for i in tr]
    return triangulate_views([cams[i].image.numpy() for i in tr], [masks[i] for i in tr], Ks,
                             [cams[i].R.numpy().astype(np.float64) for i in tr],
                             [cams[i].t.numpy().astype(np.float64) for i in tr])


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--n_train", type=int, default=8)
    ap.add_argument("--out", required=True)
    ap.add_argument("--downscale", type=int, default=2)
    a = ap.parse_args()
    pts = scan_points(a.root, a.n_train, a.downscale)
    np.save(a.out, pts)
    print(f"{a.root}: {len(pts)} points -> {a.out}")

In [ ]:
%%writefile scripts/fm2.py
"""Evaluation v2 (strict reference, 16 shared held-out views), one row per run_dir x seed (x snapshot).

Always renders with the torch rasterizer (exact blending weights), whatever renderer was used for training.
Adds to floater_metrics.metrics:
  wsig_01       W_float with a shape-aware weight: per Gaussian, fraction of 7 sigma points (centre and
                centre +- 1 sigma along the 3 principal axes) farther than 0.1 from the reference
  psnr_aligned  held-out PSNR after a per-view, per-channel affine colour fit (removes a global colour shift)
With --snaps: light metrics (W_float(0.1), off(0.1), #Gaussians, PSNR) for every snap_XXXXX.pt and base.pt.

python scripts/fm2.py --run_dirs runs/v2_* --seeds 0 1 2 --out results/v2/x.csv [--snaps]
"""
from __future__ import annotations

import argparse
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))

from floater_metrics import load_eval_scene, metrics  # noqa: E402
from frecon.gaussians import GaussianModel, nearest_dist  # noqa: E402
from frecon.measure import heldout_errors, render_views  # noqa: E402
from frecon.render import get_rasterizer  # noqa: E402
from frecon.render.torch_rast import quat_to_rotmat  # noqa: E402
from frecon.train import TrainConfig, Trainer  # noqa: E402

NAME = re.compile(r"(photo|v2|v3|v5)_(none|mild|strongng|strongs1|strongs2|strong|pac|pa|pb|pc)(.*)_dtu_scan(\d+)_v(\d+)$")


def parse(name: str) -> dict:
    m = NAME.match(name)
    if not m:
        return {"prefix": "?", "level": "?", "tags": name}
    return {"prefix": m[1], "level": m[2], "tags": m[3]}


@torch.no_grad()
def contrib(model, cams, ras, bg):
    c = torch.zeros(model.n, device=model.means.device)
    for cam in cams:
        c += model.render(ras, cam, bg, return_contrib=True)["contrib"]
    return c


@torch.no_grad()
def sigma_far(model, ref, tau=0.1):
    mu, s = model.means.detach(), model.scales.detach()
    R = quat_to_rotmat(model.quats.detach())
    far = (nearest_dist(mu, ref) > tau).float()
    for m in range(3):
        ax = R[:, :, m] * s[:, m:m + 1]
        for sg in (-1.0, 1.0):
            far += (nearest_dist(mu + sg * ax, ref) > tau).float()
    return far / 7


@torch.no_grad()
def aligned_psnr(model, cams, ras, bg):
    out = []
    for cam in cams:
        x = model.render(ras, cam, bg)["image"].clamp(0, 1).reshape(-1, 3)
        y = cam.image.reshape(-1, 3)
        fit = torch.empty_like(x)
        for ch in range(3):
            A = torch.stack([x[:, ch], torch.ones_like(x[:, ch])], 1)
            sol = torch.linalg.lstsq(A.cpu().double(), y[:, ch:ch + 1].cpu().double()).solution
            fit[:, ch] = (A @ sol.to(A).float())[:, 0]
        mse = ((fit.clamp(0, 1) - y) ** 2).mean()
        out.append(float(-10 * torch.log10(mse.clamp_min(1e-12))))
    return float(np.mean(out))


def light(model, test_cams, ref, ras, bg):
    cb = contrib(model, test_cams, ras, bg)
    d = nearest_dist(model.means.detach(), ref)
    far = d > 0.1
    err = heldout_errors(render_views(model, test_cams, ras, bg), test_cams)
    return {"wfloat_01": float(cb[far].sum() / cb.sum().clamp_min(1e-12)), "off_01": float(far.float().mean()),
            "n_gaussians": model.n, "psnr": err["psnr"]}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run_dirs", nargs="+", required=True)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--out", required=True)
    ap.add_argument("--snaps", action="store_true")
    args = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ras, bg = get_rasterizer("torch"), torch.ones(3, device=dev)
    rows, cache = [], {}
    for rd in map(Path, args.run_dirs):
        cfg = yaml.safe_load((rd / "config.yaml").read_text(encoding="utf-8"))
        sc, tc, ic = cfg["scene"], cfg.get("train", {}), cfg.get("init", {})
        key = (sc["root"], sc["n_train_views"])
        if key not in cache:
            cache = {key: load_eval_scene(cfg, dev, "strict", 0.8, 16)}
        views, test_cams, ref = cache[key]
        photo = sc.get("photometric") or {}
        info = dict(run=rd.name, scan=Path(sc["root"]).name, N=sc["n_train_views"], **parse(rd.name),
                    exposure_stops=photo.get("exposure_stops", 0.0), photo_seed=photo.get("seed", 0),
                    perm=str(photo.get("perm", "")), clip=bool(photo.get("clip", True)),
                    photo_mode=photo.get("mode", "isp"), appearance=tc.get("appearance", "none"),
                    lr_appearance=tc.get("lr_appearance", 1e-3), sh_degree=tc.get("sh_degree", 0),
                    sat_mask=bool(tc.get("sat_mask", False)), warmup=int(tc.get("warmup_geom", 0)),
                    init=ic.get("type", "random"), renderer=cfg.get("renderer", "torch"),
                    iterations=tc.get("iterations", 7000))
        for seed in args.seeds:
            ck = rd / f"seed_{seed}" / "base.pt"
            if not ck.exists():
                continue
            if args.snaps:
                for sp in sorted((rd / f"seed_{seed}").glob("snap_*.pt")) + [ck]:
                    if sp == ck:
                        model, step = Trainer.load(ck, TrainConfig(**tc), ras, bg, dev).model, info["iterations"]
                    else:
                        d = torch.load(sp, map_location=dev, weights_only=False)
                        model, step = GaussianModel.from_state_dict(d["params"], dev), int(d["step"])
                    rows.append(dict(info, seed=seed, step=step, **light(model, test_cams, ref, ras, bg)))
                    del model
                print(rd.name, seed, "snapshots done", flush=True)
            else:
                model = Trainer.load(ck, TrainConfig(**tc), ras, bg, dev).model
                row = dict(info, seed=seed, step=info["iterations"])
                row.update(metrics(model, views, test_cams, ref, ras, bg, 0.03, (0.05, 0.1)))
                cb = contrib(model, test_cams, ras, bg)
                row["wsig_01"] = float((cb * sigma_far(model, ref)).sum() / cb.sum().clamp_min(1e-12))
                row["psnr_aligned"] = aligned_psnr(model, test_cams, ras, bg)
                rows.append(row)
                print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()
                       if k in ("run", "seed", "wfloat_01", "wsig_01", "psnr", "psnr_aligned")}, flush=True)
                del model
            if dev == "cuda":
                torch.cuda.empty_cache()
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(args.out, index=False)
    print(f"-> {args.out} ({len(rows)} rows)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/fm3.py
"""Evaluation v3 = fm2 (W_float, W_float^Sigma, direct / aligned PSNR) + geometry metrics independent of W_float.

Geometry: expected depth is rendered (torch rasterizer) for the N training and 16 held-out views, pixels inside
the object mask with alpha > 0.5 are back-projected and merged (at most 300k points), and compared with the strict
STL reference:
  acc   mean distance prediction -> reference      comp  mean distance reference -> prediction
  chamfer = (acc + comp) / 2                       fscore_XX  F-score at threshold 0.0XX (normalized units)

python scripts/fm3.py --run_dirs runs/v3_* --seeds 0 1 2 --out results/v3/x.csv
"""
from __future__ import annotations

import argparse
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
sys.path.insert(0, str(Path(__file__).resolve().parent))

import fm2  # noqa: E402
from floater_metrics import load_eval_scene, metrics, split_eval  # noqa: E402
from frecon.data.dtu import load_dtu_views  # noqa: E402
from frecon.gaussians import nearest_dist  # noqa: E402
from frecon.render import get_rasterizer  # noqa: E402
from frecon.train import TrainConfig, Trainer  # noqa: E402

NAME = re.compile(r"(photo|v2|v3|v5)_(none|mild|strongng|strongs1|strongs2|strong|pac|pa|pb|pc)(.*)_dtu_scan(\d+)_v(\d+)$")


def parse(name: str) -> dict:
    m = NAME.match(name)
    return {"prefix": m[1], "level": m[2], "tags": m[3]} if m else {"prefix": "?", "level": "?", "tags": name}


@torch.no_grad()
def covered_wfloat(model, cams, ref, ras, taus=(0.05, 0.1), r=5):
    """W_float restricted to pixels onto which the reference projects (dilated by r): the DTU STL has holes on parts of
    the objects (e.g. scan83 hat, scan63 pumpkin rim), where surface Gaussians are counted as floaters by W_float."""
    d = nearest_dist(model.means.detach(), ref)
    far = torch.stack([(d > t).float() for t in taus], 1)
    fa, tot, cov_px, obj_px = torch.zeros(len(taus)), 0.0, 0.0, 0.0
    for cam in cams:
        Xc = ref @ cam.R.T + cam.t
        z = Xc[:, 2]
        u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
        v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
        ok = (z > 0) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
        c = torch.zeros(cam.H, cam.W, device=ref.device)
        c[v[ok], u[ok]] = 1
        cov = torch.nn.functional.max_pool2d(c[None, None], r, 1, r // 2)[0, 0]
        out = model.render(ras, cam, torch.zeros(len(taus), device=ref.device), features=far)
        fa += (out["image"] * cov[..., None]).sum((0, 1)).cpu()
        tot += float((out["alpha"] * cov).sum())
        obj = (cam.image.to(ref.device) < 0.99).any(-1).float()
        cov_px += float((cov * obj).sum())
        obj_px += float(obj.sum())
    res = {f"wfloat_{str(t).replace('0.', '0', 1)}_cov": float(fa[i] / max(tot, 1e-9)) for i, t in enumerate(taus)}
    res["ref_cover"] = cov_px / max(obj_px, 1.0)
    return res


@torch.no_grad()
def fused_points(model, cams, masks, ras, max_points=300_000, seed=0):
    pts = []
    for cam, m in zip(cams, masks):
        z = (model.means.detach() @ cam.R.T + cam.t)[:, 2:3]
        out = model.render(ras, cam, torch.zeros(1, device=z.device), features=z)
        a = out["alpha"]
        ok = (a > 0.5) & m
        if not bool(ok.any()):
            continue
        D = out["image"][..., 0] / a.clamp_min(1e-6)
        v, u = torch.nonzero(ok, as_tuple=True)
        d = D[v, u]
        x = ((u.float() + 0.5) - cam.cx) / cam.fx * d
        y = ((v.float() + 0.5) - cam.cy) / cam.fy * d
        Xc = torch.stack([x, y, d], -1)
        pts.append((Xc - cam.t) @ cam.R)          # camera -> world: R^T (Xc - t)
    P = torch.cat(pts) if pts else torch.zeros(0, 3, device=model.means.device)
    if len(P) > max_points:
        P = P[torch.randperm(len(P), generator=torch.Generator().manual_seed(seed))[:max_points].to(P.device)]
    return P


@torch.no_grad()
def geometry(model, cams, masks, ref, ras, taus=(0.01, 0.02)):
    P = fused_points(model, cams, masks, ras)
    if len(P) == 0:
        return {"geo_acc": np.nan, "geo_comp": np.nan, "chamfer": np.nan, "n_fused": 0, **{f"fscore_{int(t * 1000):03d}": 0.0 for t in taus}}
    d_pr = nearest_dist(P, ref)
    d_rp = nearest_dist(ref, P)
    out = {"geo_acc": float(d_pr.mean()), "geo_comp": float(d_rp.mean()), "n_fused": len(P)}
    out["chamfer"] = 0.5 * (out["geo_acc"] + out["geo_comp"])
    for t in taus:
        p, r = float((d_pr < t).float().mean()), float((d_rp < t).float().mean())
        out[f"fscore_{int(t * 1000):03d}"] = 2 * p * r / max(p + r, 1e-12)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run_dirs", nargs="+", required=True)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ras, bg = get_rasterizer("torch"), torch.ones(3, device=dev)
    rows, cache = [], {}
    for rd in map(Path, args.run_dirs):
        cfg = yaml.safe_load((rd / "config.yaml").read_text(encoding="utf-8"))
        sc, tc, ic = cfg["scene"], cfg.get("train", {}), cfg.get("init", {})
        key = (sc["root"], sc["n_train_views"])
        if key not in cache:
            views, test_cams, ref = load_eval_scene(cfg, dev, "strict", 0.8, 16)
            cams, masks, _ = load_dtu_views(sc["root"], sc.get("downscale", 4))
            tr_i, te_i = split_eval(cams, sc["n_train_views"], 16, 16)
            gcams = [cams[i].to(dev) for i in list(tr_i) + list(te_i)]
            gmasks = [torch.from_numpy(masks[i]).to(dev) for i in list(tr_i) + list(te_i)]
            cache = {key: (views, test_cams, ref, gcams, gmasks)}
        views, test_cams, ref, gcams, gmasks = cache[key]
        photo = sc.get("photometric") or {}
        info = dict(run=rd.name, scan=Path(sc["root"]).name, N=sc["n_train_views"], **parse(rd.name),
                    appearance=tc.get("appearance", "none"), init=ic.get("type", "random"),
                    renderer=cfg.get("renderer", "torch"), sh_degree=tc.get("sh_degree", 0),
                    interv=tc.get("interv", "none"), interv_target=tc.get("interv_target", "off"),
                    freeze="+".join(tc.get("freeze", []) or []), topo_freeze_from=tc.get("topo_freeze_from", -1),
                    parent=ic.get("ckpt", ""), exposure_stops=photo.get("exposure_stops", 0.0))
        for seed in args.seeds:
            ck = rd / f"seed_{seed}" / "base.pt"
            if not ck.exists():
                continue
            model = Trainer.load(ck, TrainConfig(**tc), ras, bg, dev).model
            row = dict(info, seed=seed)
            row.update(metrics(model, views, test_cams, ref, ras, bg, 0.03, (0.05, 0.1)))
            cb = fm2.contrib(model, test_cams, ras, bg)
            row["wsig_01"] = float((cb * fm2.sigma_far(model, ref)).sum() / cb.sum().clamp_min(1e-12))
            row["psnr_aligned"] = fm2.aligned_psnr(model, test_cams, ras, bg)
            row.update(covered_wfloat(model, test_cams, ref, ras))   # W_COV
            row.update(geometry(model, gcams, gmasks, ref, ras))
            rows.append(row)
            print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()
                   if k in ("run", "seed", "wfloat_01", "chamfer", "fscore_010", "psnr")}, flush=True)
            del model
            if dev == "cuda":
                torch.cuda.empty_cache()
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(args.out, index=False)
    print(f"-> {args.out} ({len(rows)} rows)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile frecon/cags.py
"""CAGS - compensation-aware Gaussian suppression inside 3DGS optimization (working name).

Per-Gaussian risk q_i = p_geo_i * sigmoid((delta_hat_i - 1) / T) is estimated every `cags_every` steps from
GT-free features computed by a probe pass over the training views, smoothed by an EMA (inherited by clone / split
children) and used by three actions:
  gate  densification: no clone / split for Gaussians with risk > cags_tau_gate
  att   opacity attenuation as a per-step cap  alpha_i <= (1 - m_i) ** gamma  (not cumulative, Adam cannot regrow it)
  mod   update-level modulation of colour / SH / scale: theta <- theta_old + (1 - m_i) (theta_adam - theta_old)
with m_i = clip((r_i - tau_s) / (1 - tau_s), 0, 1), limited to the top `cags_budget` fraction of Gaussians.

Modes (TrainConfig.cags_mode): none | log (features + labels only) | oracle_geo | oracle_cmp (risk from GT labels,
oracle diagnostics) | det (frozen detector, GT-free) | rule (hand-made score on the same features, GT-free).
GT (STL points) and the clean images are used ONLY for labels / oracle runs and for logging.

Probe: one render per training view with features [view colour (3), camera depth (1), probe channels (K)].
The probe channels are zero vectors with requires_grad; for a pixel weight map c_k, the gradient of
sum_p c_k(p) R_k(p) w.r.t. probe_i,k is sum_p c_k(p) w_ip (w_ip = blending weight), i.e. footprint-weighted
statistics of any per-pixel map, for every Gaussian, from a single backward.
"""
from __future__ import annotations

import math

import numpy as np
import torch

from .gaussians import inverse_sigmoid, nearest_dist
from .render.torch_rast import quat_to_rotmat

# ---- feature table: (name, group, bounded in [0, 1]?) ------------------------------------------------------------
FEATS = [
    ("opacity", "A", True), ("ls_max", "A", False), ("ls_min", "A", False), ("aniso", "A", False),
    ("col_r", "A", True), ("col_g", "A", True), ("col_b", "A", True), ("sh_energy", "A", False),
    ("dist_center", "A", False), ("depth_ratio", "A", False),
    ("n_vis", "G", True), ("entropy", "G", True), ("max_share", "G", True), ("front_frac", "G", True),
    ("log_contrib", "G", False), ("knn_dist", "G", False), ("knn_opacity", "G", True), ("knn_front", "G", True),
    ("knn_share", "G", True),
    ("res_abs", "P", False), ("res_sgn_std", "P", False), ("conf_dc", "P", True), ("conf_sh", "P", True),
    ("knn_conf", "P", True),
    ("age", "D", True), ("gen", "D", False), ("d_opacity", "D", False), ("d_ls", "D", False),
    ("grad_acc", "D", False), ("parent_risk", "D", True), ("t_norm", "D", True),
]
FEAT_NAMES = [f[0] for f in FEATS]
GROUPS = {g: [i for i, f in enumerate(FEATS) if f[1] == g] for g in "AGPD"}
N_PROBE = 6          # probe channels: 1, |res|, signed log res, log phi (3)
EPS = 1e-8


# ---- per-Gaussian auxiliary state (kept in sync with clone / split / prune) --------------------------------------
def _aux_new(tr, n, par=None):
    dev = tr.model.means.device
    a = tr._aux
    if par is not None and len(par) == n and len(a["birth"]):
        return {"birth": torch.full((n,), float(tr.step), device=dev), "gen": a["gen"][par] + 1,
                "op_last": a["op_last"][par], "ls_last": a["ls_last"][par], "risk": a["risk"][par],
                "parent_risk": a["risk"][par]}
    z = torch.zeros(n, device=dev)
    return {"birth": torch.full((n,), float(tr.step), device=dev), "gen": z.clone(), "op_last": torch.full((n,), -1.0, device=dev),
            "ls_last": torch.full((n,), -99.0, device=dev), "risk": z.clone(), "parent_risk": z.clone()}


def ensure_aux(tr):
    if getattr(tr.cfg, "cags_mode", "none") == "none":
        return False
    n = tr.model.n
    if getattr(tr, "_aux", None) is None:
        tr._aux = {k: torch.zeros(0, device=tr.model.means.device) for k in
                   ("birth", "gen", "op_last", "ls_last", "risk", "parent_risk")}
        tr._aux = _aux_new(tr, n)
    elif tr._aux["birth"].numel() != n:          # out of sync (should not happen): reset
        tr._aux = _aux_new(tr, n)
    return True


def on_append(tr, n_new):
    par = tr.__dict__.pop("_par", None)
    if getattr(tr, "_aux", None) is None:
        return
    new = _aux_new(tr, n_new, par)
    tr._aux = {k: torch.cat([v, new[k]]) for k, v in tr._aux.items()}


def on_prune(tr, keep):
    if getattr(tr, "_aux", None) is not None and tr._aux["birth"].numel() == keep.numel():
        tr._aux = {k: v[keep] for k, v in tr._aux.items()}


# ---- GT labels (oracle / training targets only) -------------------------------------------------------------------
def _gt(tr):
    if not hasattr(tr, "_cags_gt"):
        P = torch.from_numpy(np.load(tr.cfg.gt_ref)).float()
        if len(P) > 50000:
            P = P[torch.randperm(len(P), generator=torch.Generator().manual_seed(0))[:50000]]
        P = P.to(tr.model.means.device)
        nrm = torch.zeros_like(P)                    # PCA normals from 16 nearest neighbours
        for i in range(0, len(P), 2048):
            d = torch.cdist(P[i:i + 2048], P)
            idx = torch.topk(d, 16, largest=False).indices
            Q = P[idx] - P[idx].mean(1, keepdim=True)
            C = Q.transpose(1, 2) @ Q
            _, vec = torch.linalg.eigh(C)
            nrm[i:i + 2048] = vec[..., 0]
        tr._cags_gt = (P, nrm)
    return tr._cags_gt


@torch.no_grad()
def gt_labels(tr):
    """d: centre distance to the STL reference; d_norm: d / std of the Gaussian along the reference normal.
    y_geo: 1 off-surface (d > tau, as in W_float, whatever the size), 0 on-surface (d < tau/2, or d <= tau and
    d_norm < 1), -1 ambiguous. A wide Gaussian whose centre is farther than tau counts as off-surface: in v4 the
    oracle moved the compensation onto such Gaussians when they were labelled on-surface by d_norm < 1."""
    P, N = _gt(tr)
    m = tr.model
    mu = m.means.detach()
    d = torch.empty(m.n, device=mu.device)
    nn = torch.empty(m.n, dtype=torch.long, device=mu.device)
    for i in range(0, m.n, 2048):
        dd = torch.cdist(mu[i:i + 2048], P)
        v, j = dd.min(1)
        d[i:i + 2048], nn[i:i + 2048] = v, j
    n = N[nn]
    R = quat_to_rotmat(m.quats.detach() / m.quats.detach().norm(dim=-1, keepdim=True))
    v = torch.einsum("nji,nj->ni", R, n)                       # R^T n
    sig = (m.scales.detach() ** 2 * v ** 2).sum(-1).clamp_min(1e-12).sqrt()
    dn = d / sig
    tau = float(getattr(tr.cfg, "gt_tau", 0.1))
    y = torch.full((m.n,), -1.0, device=mu.device)
    y[(d < tau / 2) | ((dn < 1) & (d <= tau))] = 0.0   # LABEL_V2
    y[d > tau] = 1.0
    return d, dn, y


# ---- probe ----------------------------------------------------------------------------------------------------------
def _ro_appearance(tr, img, cam):
    """GC-ILA correction with the current coefficients, without updating them (read-only)."""
    if getattr(tr.cfg, "appearance", "none") != "gc_log" or cam.name not in getattr(tr, "gc", {}):
        return img
    a, b = tr.gc[cam.name]
    obj = (cam.image < 0.99).any(-1)
    return torch.where(obj[..., None], (a * img.clamp_min(1e-3).log() + b).exp(), img)


def _rank(x):
    n = x.numel()
    if n < 2:
        return torch.zeros_like(x)
    r = torch.empty(n, device=x.device)
    r[torch.argsort(x)] = torch.arange(n, device=x.device, dtype=torch.float32)
    return r / (n - 1)


def probe(tr, cams, with_labels=False):
    """Returns (F (N, len(FEATS)) normalized features, raw dict, labels dict or None)."""
    from frecon.data import photometric as ph
    cfg, m = tr.cfg, tr.model
    n, dev, V = m.n, m.means.device, len(cams)
    ext = float(tr.extent)
    mu = m.means.detach()
    W = torch.zeros(V, n, device=dev)
    Rabs = torch.zeros(V, n, device=dev)
    Rsgn = torch.zeros(V, n, device=dev)
    Lphi = torch.zeros(n, 3, device=dev)
    inb_c = torch.zeros(n, device=dev)
    front_c = torch.zeros(n, device=dev)
    ref_phi = torch.zeros(n, 3, device=dev)
    ref_cnt = torch.zeros(n, device=dev)
    ratio_l = []
    gdc_sum = torch.zeros(n, 3, device=dev)
    gdc_nrm = torch.zeros(n, device=dev)
    has_sh = "sh_rest" in m.params
    gsh_sum = torch.zeros(n, m.params["sh_rest"][0].numel() if has_sh else 1, device=dev)
    gsh_nrm = torch.zeros(n, device=dev)
    zmin = torch.full((n,), float("inf"), device=dev)
    have_phi = False
    thr = 0.5                                    # blending weight (pixels) for "the view sees the Gaussian"
    for j, cam in enumerate(cams):
        for p in m.params.values():
            p.grad = None
        Xc = mu @ cam.R.T + cam.t
        z = Xc[:, 2]
        cols = m.view_colors(cam)
        pr = torch.zeros(n, N_PROBE, device=dev, requires_grad=True)
        feats = torch.cat([cols, z[:, None], pr], -1)
        bg = torch.cat([tr.bg.to(dev).float(), torch.zeros(1 + N_PROBE, device=dev)])
        out = m.render(tr.rasterize, cam, bg, features=feats)
        img, Z, A = out["image"][..., :3], out["image"][..., 3], out["alpha"]
        aux = out["image"][..., 4:]
        tgt = cam.image.to(dev)
        pred = _ro_appearance(tr, img, cam)
        obj = (tgt < 0.99).any(-1)
        loss = (pred - tgt).abs().mean()
        with torch.no_grad():
            res = (pred - tgt).abs().mean(-1) * obj
            sgn = (tgt.clamp_min(1e-3).log() - pred.clamp_min(1e-3).log()).mean(-1) * obj
            clean = ph.__dict__.get("CLEAN", {}).get(cam.name)
            if clean is not None:
                clean = clean.to(dev)
                lphi = (tgt.clamp_min(1e-3).log() - clean.clamp_min(1e-3).log()) * obj[..., None]
                have_phi = True
            else:
                lphi = torch.zeros_like(tgt)
            cmap = torch.stack([torch.ones_like(res), res, sgn, lphi[..., 0], lphi[..., 1], lphi[..., 2]], -1)
        (loss + (cmap * aux).sum()).backward()
        with torch.no_grad():
            g = pr.grad
            w = g[:, 0].clamp_min(0)
            W[j], Rabs[j], Rsgn[j] = w, g[:, 1], g[:, 2]
            Lphi += g[:, 3:6]
            seen = w > thr
            gdc = m.params["color_logits"].grad
            if gdc is not None:
                gh = gdc / (w[:, None] + EPS) * seen[:, None]
                gdc_sum += gh
                gdc_nrm += gh.norm(dim=-1)
            if has_sh and m.params["sh_rest"].grad is not None:
                gs = m.params["sh_rest"].grad.reshape(n, -1) / (w[:, None] + EPS) * seen[:, None]
                gsh_sum += gs
                gsh_nrm += gs.norm(dim=-1)
            # centre projection: geometric visibility / "floating in front of the rendered surface"
            D = Z / A.clamp_min(1e-6)
            u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
            v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
            inb = (z > 1e-3) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
            uu, vv = u.clamp(0, cam.W - 1), v.clamp(0, cam.H - 1)
            Dp, Ap = D[vv, uu], A[vv, uu]
            dlt = 0.02 * ext
            surf = inb & (Ap > 0.5)
            front = surf & (z < Dp - dlt)
            visg = inb & (~surf | (z <= Dp + dlt))     # not occluded by the rendered surface
            inb_c += inb.float()
            front_c += front.float()
            ref_phi += lphi[vv, uu] * visg[:, None]
            ref_cnt += visg.float()
            rr = torch.where(surf, z / Dp.clamp_min(1e-6), torch.full_like(z, float("nan")))
            ratio_l.append(rr)
            zmin = torch.minimum(zmin, torch.where(inb, z, torch.full_like(z, float("inf"))))
    for p in m.params.values():
        p.grad = None
    with torch.no_grad():
        wsum = W.sum(0)
        wh = W / (wsum + EPS)
        ent = -(wh * (wh + EPS).log()).sum(0) / math.log(max(V, 2))
        seenV = (W > thr).float()
        nvis = seenV.mean(0)
        maxs = wh.max(0).values
        res_abs = Rabs.sum(0) / (wsum + EPS)
        s_j = Rsgn / (W + EPS) * seenV
        cnt = seenV.sum(0).clamp_min(1)
        s_mean = s_j.sum(0) / cnt
        s_std = (((s_j - s_mean) ** 2) * seenV).sum(0).div(cnt).sqrt()
        conf_dc = torch.where(cnt > 1, 1 - gdc_sum.norm(dim=-1) / (gdc_nrm + EPS), torch.zeros_like(wsum))
        conf_sh = torch.where(cnt > 1, 1 - gsh_sum.norm(dim=-1) / (gsh_nrm + EPS), torch.zeros_like(wsum))
        front_frac = front_c / inb_c.clamp_min(1)
        ratio = torch.nanmedian(torch.stack(ratio_l), 0).values
        ratio = torch.nan_to_num(ratio, nan=1.0)
        op = m.opacities.detach().reshape(-1)
        ls = m.params["log_scales"].detach()
        ls_max, ls_min = ls.max(1).values, ls.min(1).values
        col = torch.sigmoid(m.params["color_logits"].detach())
        she = m.params["sh_rest"].detach().pow(2).flatten(1).sum(1) if has_sh else torch.zeros(n, device=dev)
        cen = mu.median(0).values
        dcen = (mu - cen).norm(dim=-1) / ext
        # k-NN context against a random subset of Gaussians
        g = torch.Generator(device="cpu").manual_seed(int(tr.step))
        sub = torch.randperm(n, generator=g)[:min(n, 20000)].to(dev)
        kd = torch.zeros(n, device=dev)
        ko, kf, ks, kc = (torch.zeros(n, device=dev) for _ in range(4))
        k = min(8, len(sub) - 1)
        if k >= 1:
            for i in range(0, n, 4096):
                dd = torch.cdist(mu[i:i + 4096], mu[sub])
                val, idx = torch.topk(dd, k + 1, largest=False)
                val, idx = val[:, 1:], sub[idx[:, 1:]]
                kd[i:i + 4096] = val.mean(1) / ext
                ko[i:i + 4096] = op[idx].mean(1)
                kf[i:i + 4096] = front_frac[idx].mean(1)
                ks[i:i + 4096] = maxs[idx].mean(1)
                kc[i:i + 4096] = conf_dc[idx].mean(1)
        a = tr._aux
        age = (tr.step - a["birth"]) / max(cfg.iterations, 1)
        d_op = torch.where(a["op_last"] < 0, torch.zeros_like(op), op - a["op_last"])
        d_ls = torch.where(a["ls_last"] < -50, torch.zeros_like(ls_max), ls_max - a["ls_last"])
        a["op_last"], a["ls_last"] = op.clone(), ls_max.clone()
        gacc = tr.grad_accum / tr.grad_denom.clamp_min(1) if tr.grad_accum.numel() == n else torch.zeros(n, device=dev)
        raw = {"opacity": op, "ls_max": ls_max, "ls_min": ls_min, "aniso": ls_max - ls_min, "col_r": col[:, 0],
               "col_g": col[:, 1], "col_b": col[:, 2], "sh_energy": she, "dist_center": dcen, "depth_ratio": ratio,
               "n_vis": nvis, "entropy": ent, "max_share": maxs, "front_frac": front_frac,
               "log_contrib": (wsum + 1).log(), "knn_dist": kd, "knn_opacity": ko, "knn_front": kf, "knn_share": ks,
               "res_abs": res_abs, "res_sgn_std": s_std, "conf_dc": conf_dc.clamp(0, 1), "conf_sh": conf_sh.clamp(0, 1),
               "knn_conf": kc.clamp(0, 1), "age": age.clamp(0, 1), "gen": a["gen"], "d_opacity": d_op, "d_ls": d_ls,
               "grad_acc": gacc, "parent_risk": a["parent_risk"],
               "t_norm": torch.full((n,), tr.step / max(cfg.iterations, 1), device=dev)}
        F = torch.stack([raw[nm] if bnd else _rank(raw[nm]) for nm, _, bnd in FEATS], 1)
        F = torch.nan_to_num(F, nan=0.0, posinf=1.0, neginf=0.0)
        lab = None
        if with_labels and getattr(cfg, "gt_ref", ""):
            d, dn, y = gt_labels(tr)
            if have_phi:
                obs = Lphi / (wsum[:, None] + EPS)
                ref = ref_phi / ref_cnt.clamp_min(1)[:, None]
                delta = (obs - ref).norm(dim=-1) * (wsum > thr)
            else:
                delta = torch.zeros(n, device=dev)
            on = (y == 0) & (wsum > thr)
            tau_d = float(torch.quantile(delta[on], 0.9)) if have_phi and int(on.sum()) > 10 else float("inf")
            lab = {"d": d, "d_norm": dn, "y_geo": y, "delta": delta, "tau_delta": tau_d, "wsum": wsum,
                   "y_cmp": (delta > tau_d).float(), "y_joint": ((y == 1) & (delta > tau_d)).float()}
    return F, raw, lab


# ---- detector -----------------------------------------------------------------------------------------------------
class Detector(torch.nn.Module):
    def __init__(self, idx, hidden=128, depth=3):
        super().__init__()
        self.register_buffer("idx", torch.as_tensor(idx, dtype=torch.long))
        layers, d = [], len(idx)
        for _ in range(depth):
            layers += [torch.nn.Linear(d, hidden), torch.nn.LayerNorm(hidden), torch.nn.GELU()]
            d = hidden
        self.body = torch.nn.Sequential(*layers)
        self.head = torch.nn.Linear(hidden, 2)        # logit p_geo, delta_hat (in units of tau_delta)

    def forward(self, F):
        h = self.body(F[:, self.idx])
        o = self.head(h)
        return o[:, 0], o[:, 1]


def joint_risk(logit, dhat, T=0.25, use_cmp=True):
    p = torch.sigmoid(logit)
    return p * torch.sigmoid((dhat - 1.0) / T) if use_cmp else p


def load_detector(path, dev):
    ck = torch.load(path, map_location=dev, weights_only=False)
    det = Detector(ck["idx"], ck.get("hidden", 128), ck.get("depth", 3)).to(dev)
    det.load_state_dict(ck["state"])
    det.eval()
    det.use_cmp = ck.get("use_cmp", True)
    det.tau = ck.get("tau")          # calibration threshold on q (cags_train), None for old checkpoints
    return det


def rule_score(F):
    """GT-free hand-made baseline: mean of the ranks of max_share, front_frac, conf_dc and res_abs."""
    cols = [FEAT_NAMES.index(c) for c in ("max_share", "front_frac", "conf_dc", "res_abs")]
    return torch.stack([_rank(F[:, c]) for c in cols], 1).mean(1)


# ---- hooks called from Trainer --------------------------------------------------------------------------------------
def _strength(tr):
    cfg, r = tr.cfg, tr._aux["risk"]
    ts = float(getattr(cfg, "cags_tau_s", 0.5))
    m = ((r - ts) / (1 - ts)).clamp(0, 1)
    b = float(getattr(cfg, "cags_budget", 1.0))
    if b < 1.0 and r.numel():
        k = max(int(b * r.numel()), 1)
        thr = torch.topk(r, k).values.min()
        m = m * (r >= thr)
    return m


def _acts(cfg):
    a = getattr(cfg, "cags_actions", "full")
    return {"gate", "att", "mod"} if a == "full" else set(a.split("+"))


def _active(tr):
    cfg = tr.cfg
    return (getattr(cfg, "cags_mode", "none") not in ("none", "log") and tr.step >= getattr(cfg, "cags_from", 0)
            and getattr(tr, "_aux", None) is not None and tr._aux["risk"].numel() == tr.model.n)


def gate(tr):
    if not _active(tr) or "gate" not in _acts(tr.cfg):
        return None
    return tr._aux["risk"] > float(getattr(tr.cfg, "cags_tau_gate", 0.5))


def pre_step(tr):
    """Before the optimizer step: rows to modulate and their current values."""
    if not _active(tr) or "mod" not in _acts(tr.cfg):
        return None
    m = _strength(tr)
    sel = m > 0
    if not bool(sel.any()):
        return None
    P = tr.model.params
    keep = {k: P[k].detach()[sel].clone() for k in ("color_logits", "sh_rest", "log_scales") if k in P}
    return sel, m[sel], keep


def post_step(tr, pre):
    if not _active(tr):
        return
    P = tr.model.params
    with torch.no_grad():
        if pre is not None:
            sel, ms, keep = pre
            for k, old in keep.items():
                new = P[k].detach()[sel]
                f = (1 - ms).reshape(-1, *([1] * (old.dim() - 1)))
                P[k][sel] = old + f * (new - old)
        if "att" in _acts(tr.cfg):
            m = _strength(tr)
            sel = m > 0
            if bool(sel.any()):
                gam = float(getattr(tr.cfg, "cags_gamma", 2.0))
                cap = inverse_sigmoid(((1 - m[sel]) ** gam).clamp(1e-4, 1.0))
                lg = P["opacity_logits"]
                lg[sel] = torch.minimum(lg[sel], cap.reshape(lg[sel].shape))


def tick(tr):
    """After a training step (and densification): probe when due, log features / labels, update the risk EMA."""
    cfg = tr.cfg
    mode = getattr(cfg, "cags_mode", "none")
    if mode == "none" or not ensure_aux(tr):
        return
    s = tr.step
    due_log = s in set(int(x) for x in (getattr(cfg, "cags_log_steps", ()) or ()))
    t0, K = int(getattr(cfg, "cags_from", 0)), int(getattr(cfg, "cags_every", 100))
    first = not getattr(tr, "_cags_init", False)
    due_act = mode != "log" and s >= t0 and (first or (s - t0) % K == 0)
    if not (due_log or due_act):
        return
    cams = getattr(tr, "_cams", None)
    if not cams:
        return
    need_lab = due_log or mode.startswith("oracle")
    F, raw, lab = probe(tr, cams, with_labels=need_lab)
    if due_log and lab is not None:
        _log(tr, F, lab)
        if mode != "log" and getattr(tr, "_cags_init", False):   # diagnostics against GT (logging only)
            act, yg = _strength(tr) > 0, lab["y_geo"]
            tr.cags_hist.append({"step": s, "diag": 1, "n": tr.model.n, "n_off": int((yg == 1).sum()),
                                 "act_off": int((act & (yg == 1)).sum()), "act_on": int((act & (yg == 0)).sum()),
                                 "act_joint": int((act & (lab["y_joint"] > 0)).sum()), "n_joint": int(lab["y_joint"].sum())})
    if not due_act:
        return
    with torch.no_grad():
        if mode == "oracle_geo":
            q = (lab["y_geo"] == 1).float()
        elif mode == "oracle_cmp":
            q = lab["y_joint"]
        elif mode == "rule":
            q = rule_score(F)
        elif mode == "det":
            if not hasattr(tr, "_det"):
                tr._det = load_detector(cfg.cags_model, F.device)
            lg, dh = tr._det(F)
            q = joint_risk(lg, dh, use_cmp=tr._det.use_cmp)
            if getattr(tr._det, "tau", None):    # calibrated: q = tau maps to 0.5 (= tau_s), full strength at 2 tau
                q = (0.5 * q / tr._det.tau).clamp(0, 1)
        else:
            raise ValueError(mode)
        b = float(getattr(cfg, "cags_ema", 0.5))
        r = tr._aux["risk"]
        tr._aux["risk"] = q.float() if first else b * r + (1 - b) * q.float()
        tr._cags_init = True
        row = {"step": s, "n": tr.model.n, "risk_mean": float(tr._aux["risk"].mean()),
               "n_active": int((_strength(tr) > 0).sum()),
               "n_gate": int((tr._aux["risk"] > float(getattr(cfg, "cags_tau_gate", 0.5))).sum())}
        tr.__dict__.setdefault("cags_hist", []).append(row)


def _log(tr, F, lab):
    """Stratified subsample of (features, labels) rows kept on the CPU (float16)."""
    n = F.shape[0]
    g = torch.Generator().manual_seed(int(tr.step) * 31 + 7)
    y = lab["y_geo"].cpu()
    pos = torch.nonzero(y == 1).flatten()
    oth = torch.nonzero(y != 1).flatten()
    cap = int(getattr(tr.cfg, "cags_log_max", 20000))
    pos = pos[torch.randperm(len(pos), generator=g)[:cap]]
    oth = oth[torch.randperm(len(oth), generator=g)[:cap]]
    idx = torch.cat([pos, oth]).sort().values
    wpos = max(int((y == 1).sum()), 1) / max(len(pos), 1)
    woth = max(int((y != 1).sum()), 1) / max(len(oth), 1)
    sw = torch.where(y[idx] == 1, torch.tensor(wpos), torch.tensor(woth))   # inverse sampling rate
    rows = {"F": F[idx.to(F.device)].half().cpu(), "step": torch.full((len(idx),), tr.step, dtype=torch.int32),
            "sw": sw.float()}
    for k in ("d", "d_norm", "y_geo", "delta", "wsum", "y_cmp", "y_joint"):
        rows[k] = lab[k][idx.to(F.device)].float().cpu()
    rows["tau_delta"] = torch.full((len(idx),), lab["tau_delta"])
    tr.__dict__.setdefault("cags_rows", []).append(rows)
    tr.__dict__.setdefault("cags_hist", []).append(
        {"step": tr.step, "n": n, "n_off": int((y == 1).sum()), "n_amb": int((y == -1).sum()),
         "tau_delta": lab["tau_delta"], "frac_joint": float(lab["y_joint"].mean())})


def write_log(tr, path):
    rows = getattr(tr, "cags_rows", None)
    if rows:
        out = {k: torch.cat([r[k] for r in rows]).numpy() for k in rows[0]}
        np.savez_compressed(path.parent / "cags_log.npz", feat_names=np.array(FEAT_NAMES), **out)
    hist = getattr(tr, "cags_hist", None)
    if hist:
        import csv
        keys = sorted({k for r in hist for k in r})
        with open(path.parent / "cags_hist.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(hist)

In [ ]:
%%writefile scripts/cags_train.py
"""Train / evaluate the CAGS detector on logged (features, labels) rows, leave-scan-out.

python scripts/cags_train.py --logs "runs/v5_*_std_van_dtu_scan*_v8/seed_*/cags_log.npz" --out results/v5/det \
    --folds 24,63,97 37,65,105 ... --sets A AG AGP AGPD --train_levels none pa pb pc --tag r0

For every fold f and feature set S a detector is trained on the scans outside f and evaluated on the scans of f.
Saved: <out>/<tag>_<S>_fold<f>.pt (used by cags_mode=det) and <out>/<tag>_metrics.csv. Labels: y_geo (off-surface,
-1 ambiguous rows are excluded from the classification loss), delta / tau_delta (compensation, regression head).
"""
from __future__ import annotations

import argparse
import glob
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from frecon.cags import FEATS, GROUPS, Detector, joint_risk  # noqa: E402

RUN = re.compile(r"v5_(none|pac|pa|pb|pc)_std_(.+)_dtu_scan(\d+)_v8")


def load(pattern):
    parts = []
    for f in sorted(glob.glob(pattern)):
        m = RUN.search(f)
        if not m:
            continue
        z = np.load(f)
        n = len(z["y_geo"])
        d = {k: z[k] for k in ("F", "y_geo", "delta", "tau_delta", "wsum", "y_joint", "sw", "step", "d", "d_norm")}
        d.update(level=np.full(n, m[1]), tag=np.full(n, m[2]), scan=np.full(n, int(m[3])),
                 seed=np.full(n, int(Path(f).parent.name.split("_")[1])))
        parts.append(d)
    if not parts:
        raise SystemExit(f"no logs match {pattern}")
    D = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}
    # LABEL_V2: labels recomputed from the logged distances (d > 0.1 is off-surface whatever d_norm);
    # sw (inverse sampling rate of the logged strata) stays valid
    y = np.full(len(D['d']), -1.0, np.float32)
    y[(D['d'] < 0.05) | ((D['d_norm'] < 1) & (D['d'] <= 0.1))] = 0.0
    y[D['d'] > 0.1] = 1.0
    D['y_geo'] = y
    D['y_joint'] = ((y == 1) & (D['delta'] > D['tau_delta'])).astype(np.float32)
    return D


def feat_idx(s):
    return sorted(i for g in s for i in GROUPS[g])


def targets(D, rows):
    tau = D["tau_delta"][rows]
    t = np.where(np.isfinite(tau) & (tau > 0), D["delta"][rows] / np.where(np.isfinite(tau) & (tau > 0), tau, 1), 0.0)
    return np.clip(t, 0, 5).astype(np.float32)


def train_one(D, rows, idx, dev, epochs=8, bs=16384, lr=2e-3, max_rows=4_000_000, seed=0):
    rng = np.random.default_rng(seed)
    if len(rows) > max_rows:
        rows = rng.choice(rows, max_rows, replace=False)
    F = torch.from_numpy(D["F"][rows]).to(dev)
    y = torch.from_numpy(D["y_geo"][rows]).float().to(dev)
    t = torch.from_numpy(targets(D, rows)).to(dev)
    w = torch.from_numpy((D["sw"][rows] * np.log1p(D["wsum"][rows])).astype(np.float32)).to(dev)
    valid = y >= 0
    pos, neg = (w * (y == 1)).sum(), (w * (y == 0)).sum()
    wc = torch.where(y == 1, w / pos.clamp_min(1e-9), w / neg.clamp_min(1e-9)) * valid * 0.5 * len(y)
    wr = w / w.mean().clamp_min(1e-9)
    torch.manual_seed(seed)
    det = Detector(idx).to(dev)
    opt = torch.optim.AdamW(det.parameters(), lr=lr, weight_decay=1e-4)
    steps = epochs * max(1, len(y) // bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    det.train()
    for _ in range(steps):
        b = torch.randint(0, len(y), (bs,), device=dev)
        lg, dh = det(F[b].float())
        p = torch.sigmoid(lg)
        yb = y[b].clamp_min(0)
        bce = torch.nn.functional.binary_cross_entropy_with_logits(lg, yb, reduction="none")
        pt = torch.where(yb > 0, p, 1 - p)
        cls = (wc[b] * (1 - pt) ** 2 * bce).mean()
        reg = (wr[b] * torch.nn.functional.huber_loss(dh, t[b], reduction="none", delta=1.0)).mean()
        loss = cls + 0.5 * reg
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        sched.step()
    det.eval()
    return det


@torch.no_grad()
def predict(det, F, dev, bs=262144, use_cmp=True):
    lg, dh = [], []
    for i in range(0, len(F), bs):
        a, b = det(torch.from_numpy(F[i:i + bs]).to(dev).float())
        lg.append(a.cpu())
        dh.append(b.cpu())
    lg, dh = torch.cat(lg), torch.cat(dh)
    return torch.sigmoid(lg).numpy(), dh.numpy(), joint_risk(lg, dh, use_cmp=use_cmp).numpy()


def calibrate(D, rows, det, dev, prec=0.8, use_cmp=True, n=1_000_000, seed=0):
    """Lowest threshold tau on the risk q at which the flagged training Gaussians are off-surface (y_geo = 1) with
    weighted precision >= prec, i.e. the largest recall at that precision. Falls back to the max-F1 threshold."""
    rng = np.random.default_rng(seed)
    rows = rows[D["y_geo"][rows] >= 0]
    if len(rows) > n:
        rows = rng.choice(rows, n, replace=False)
    _, _, q = predict(det, D["F"][rows], dev, use_cmp=use_cmp)
    y = D["y_geo"][rows] == 1
    w = D["sw"][rows].astype(np.float64)
    o = np.argsort(-q)
    tp, fl = np.cumsum((w * y)[o]), np.cumsum(w[o])
    pr, rc = tp / fl, tp / max(tp[-1], 1e-12)
    ok = np.nonzero(pr >= prec)[0]
    k = ok.max() if len(ok) else int(np.argmax(2 * pr * rc / np.maximum(pr + rc, 1e-12)))
    tau = float(max(q[o][k], 1e-6))
    info = dict(tau=tau, prec=float(pr[k]), recall_off=float(rc[k]), flag_rate=float(fl[k] / fl[-1]), fallback=not len(ok))
    return tau, info


def rule(F):
    names = [f[0] for f in FEATS]
    cols = [names.index(c) for c in ("max_share", "front_frac", "conf_dc", "res_abs")]
    r = np.stack([pd.Series(F[:, c].astype(np.float32)).rank(pct=True).to_numpy() for c in cols], 1)
    return r.mean(1)


def scores(D, rows, s_geo, s_joint, s_delta, name):
    from scipy.stats import spearmanr
    from sklearn.metrics import average_precision_score, roc_auc_score
    y = D["y_geo"][rows]
    v = y >= 0
    w = D["sw"][rows] * D["wsum"][rows]
    out = {"model": name, "n": int(v.sum()), "pos_rate": float((y[v] == 1).mean()) if v.any() else np.nan}
    if v.any() and 0 < (y[v] == 1).mean() < 1:
        out["auroc_geo"] = roc_auc_score(y[v] == 1, s_geo[v], sample_weight=D["sw"][rows][v])
        out["ap_geo"] = average_precision_score(y[v] == 1, s_geo[v], sample_weight=D["sw"][rows][v])
        out["ap_geo_w"] = average_precision_score(y[v] == 1, s_geo[v], sample_weight=w[v])
    yj = D["y_joint"][rows] > 0.5
    if 0 < yj.mean() < 1:
        out["ap_joint"] = average_precision_score(yj, s_joint, sample_weight=D["sw"][rows])
        out["ap_joint_w"] = average_precision_score(yj, s_joint, sample_weight=w)
        out["joint_rate"] = float(yj.mean())
    pert = D["level"][rows] != "none"
    if s_delta is not None and pert.sum() > 100:
        out["spearman_delta"] = spearmanr(s_delta[pert], D["delta"][rows][pert]).correlation
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--logs", nargs="+", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--folds", nargs="+", required=True, help="comma-separated scans per fold")
    ap.add_argument("--sets", nargs="+", default=["A", "AG", "AGP", "AGPD"])
    ap.add_argument("--train_levels", nargs="+", default=["none", "pa", "pb", "pc", "pac"])
    ap.add_argument("--test_levels", nargs="+", default=["none", "pa", "pb", "pc", "pac"])
    ap.add_argument("--tag", default="r0")
    ap.add_argument("--epochs", type=int, default=8)
    ap.add_argument("--baselines", action="store_true")
    ap.add_argument("--cal_prec", type=float, default=0.8, help="precision (off-surface) of the calibration threshold")
    ap.add_argument("--calibrate_only", action="store_true", help="recalibrate existing <tag>_<set>_fold<f>.pt in place")
    a = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    parts = [load(p) for p in a.logs]
    D = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}
    print(f"{len(D['y_geo'])} rows, scans {sorted(set(D['scan']))}, levels {sorted(set(D['level']))}", flush=True)
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    folds = [[int(s) for s in f.split(",")] for f in a.folds]
    res = []
    for fi, fs in enumerate(folds):
        tr_rows = np.nonzero(~np.isin(D["scan"], fs) & np.isin(D["level"], a.train_levels))[0]
        te_rows = np.nonzero(np.isin(D["scan"], fs) & np.isin(D["level"], a.test_levels))[0]
        if len(tr_rows) == 0 or len(te_rows) == 0:
            print(f"fold {fi}: no rows, skipped")
            continue
        for s in a.sets:
            f = out / f"{a.tag}_{s}_fold{fi}.pt"
            if a.calibrate_only:
                ck = torch.load(f, map_location=dev, weights_only=False)
                det = Detector(ck["idx"], ck.get("hidden", 128), ck.get("depth", 3)).to(dev)
                det.load_state_dict(ck["state"])
                det.eval()
            else:
                idx = feat_idx(s)
                det = train_one(D, tr_rows, idx, dev, epochs=a.epochs, seed=fi)
                ck = {"idx": idx, "state": det.state_dict(), "hidden": 128, "depth": 3, "use_cmp": True,
                      "set": s, "fold": fs, "train_levels": a.train_levels}
            tau, info = calibrate(D, tr_rows, det, dev, a.cal_prec, ck.get("use_cmp", True), seed=fi)
            ck.update(tau=tau, cal=info)
            torch.save(ck, f)
            print(f"fold {fi} set {s}: tau {tau:.4f}, precision {info['prec']:.3f}, recall_off {info['recall_off']:.3f}, "
                  f"flag rate {info['flag_rate']:.4f}" + (" (max-F1 fallback)" if info["fallback"] else ""), flush=True)
            if a.calibrate_only:
                continue
            for lv in a.test_levels + ["all"]:
                rr = te_rows if lv == "all" else te_rows[D["level"][te_rows] == lv]
                if len(rr) == 0:
                    continue
                pg, dh, q = predict(det, D["F"][rr], dev)
                res.append(dict(scores(D, rr, pg, q, dh, s), fold=fi, level=lv, tag=a.tag))
            print(f"fold {fi} set {s}: done", flush=True)
        if a.baselines:
            names = [f[0] for f in FEATS]
            for lv in a.test_levels + ["all"]:
                rr = te_rows if lv == "all" else te_rows[D["level"][te_rows] == lv]
                if len(rr) == 0:
                    continue
                F = D["F"][rr].astype(np.float32)
                for bn, sc in (("rule", rule(F)), ("opacity", F[:, names.index("opacity")]),
                               ("max_share", F[:, names.index("max_share")]), ("front_frac", F[:, names.index("front_frac")])):
                    res.append(dict(scores(D, rr, sc, sc, None, bn), fold=fi, level=lv, tag=a.tag))
    df = pd.DataFrame(res)
    if df.empty:          # --calibrate_only: metrics unchanged
        return
    df.to_csv(out / f"{a.tag}_metrics.csv", index=False)
    cols = [c for c in ("auroc_geo", "ap_geo", "ap_geo_w", "ap_joint", "ap_joint_w", "spearman_delta") if c in df]
    print(df[df.level == "all"].groupby("model")[cols].mean().round(3).to_string())


if __name__ == "__main__":
    main()

In [ ]:
# === PATCH 1: tập kiểm định cố định (FRECON_TEST_POOL) và bù affine theo ảnh (train.appearance) ===
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

patch('frecon/data/dtu.py',
      '    rest = [i for i in range(len(cams)) if i not in train_idx]',
      '    import os   # PATCH_FIXED_TEST\n'
      '    ref_idx = farthest_point_indices(centers, max(n_train, int(os.environ.get("FRECON_TEST_POOL", 8))))\n'
      '    rest = [i for i in range(len(cams)) if i not in ref_idx]',
      'PATCH_FIXED_TEST')

t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n'
      '    appearance: str = "none"        # PATCH_AFFINE: affine | oracle | cf_lin | cf_log (loss only)\n'
      '    lr_appearance: float = 1e-3\n', 'PATCH_AFFINE')
patch(t, '        self._reset_stats()\n\n    # ---- statistics used by densification',
      '        self._reset_stats()\n        self.app, self.app_opt = {}, None   # PATCH_APP_INIT\n\n'
      '    # ---- statistics used by densification', 'PATCH_APP_INIT')
patch(t, '        loss = photometric_loss(out["image"], cam.image, cfg.lambda_dssim)\n'
         '        self.opt.zero_grad(set_to_none=True)\n',
      '        loss = photometric_loss(self._appearance(out["image"], cam), cam.image, cfg.lambda_dssim)\n'
      '        self.opt.zero_grad(set_to_none=True)\n'
      '        if self.app_opt is not None:   # PATCH_APP_STEP\n'
      '            self.app_opt.zero_grad(set_to_none=True)\n', 'PATCH_APP_STEP')
patch(t, '        self.opt.step()\n        with torch.no_grad():  # keep quaternions',
      '        self.opt.step()\n        if self.app_opt is not None:   # PATCH_APP_OPT\n'
      '            self.app_opt.step()\n        with torch.no_grad():  # keep quaternions', 'PATCH_APP_OPT')
patch(t, '    def fit(self, cams,',
      '    def _appearance(self, img, cam):   # PATCH_APP_FN\n'
      '        if getattr(self.cfg, "appearance", "none") != "affine":\n'
      '            return img\n'
      '        if cam.name not in self.app:\n'
      '            A = torch.nn.Parameter(torch.eye(3, device=img.device))\n'
      '            b = torch.nn.Parameter(torch.zeros(3, device=img.device))\n'
      '            self.app[cam.name] = (A, b)\n'
      '            if self.app_opt is None:\n'
      '                self.app_opt = torch.optim.Adam([A, b], lr=self.cfg.lr_appearance)\n'
      '            else:\n'
      '                self.app_opt.add_param_group({"params": [A, b]})\n'
      '        A, b = self.app[cam.name]\n'
      '        return img @ A.T + b\n\n'
      '    def fit(self, cams,', 'PATCH_APP_FN')
print('PATCH 1 OK')


# === E5 PATCH: SH bậc 1-3 (màu phụ thuộc hướng nhìn), bật bằng train.sh_degree trong config ===
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

g = 'frecon/gaussians.py'
patch(g, '    def __init__(self, means, log_scales, quats, opacity_logits, color_logits):\n',
      '    def __init__(self, means, log_scales, quats, opacity_logits, color_logits, sh_rest=None):  # PATCH_SH_INIT\n', 'PATCH_SH_INIT')
patch(g, '            "color_logits": torch.nn.Parameter(color_logits.contiguous()),\n        }\n',
      '            "color_logits": torch.nn.Parameter(color_logits.contiguous()),\n        }\n'
      '        if sh_rest is not None:   # PATCH_SH_PARAM\n'
      '            self.params["sh_rest"] = torch.nn.Parameter(sh_rest.contiguous())\n', 'PATCH_SH_PARAM')
patch(g, '    def render(self, rasterize, cam, bg, features=None, opacities=None, **kw):\n'
         '        return rasterize(self.means, self.quats, self.scales,\n'
         '                         self.opacities if opacities is None else opacities,\n'
         '                         self.colors if features is None else features, cam, bg, **kw)\n',
      '    def enable_sh(self, degree: int):   # PATCH_SH_RENDER\n'
      '        if degree > 0 and "sh_rest" not in self.params:\n'
      '            k = (degree + 1) ** 2 - 1\n'
      '            self.params["sh_rest"] = torch.nn.Parameter(torch.zeros(self.n, k, 3, device=self.means.device))\n'
      '        return self\n\n'
      '    def view_colors(self, cam):\n'
      '        if "sh_rest" not in self.params:\n'
      '            return self.colors\n'
      '        d = self.means - cam.center.to(self.means)\n'
      '        d = d / d.norm(dim=-1, keepdim=True).clamp_min(1e-9)\n'
      '        B = _sh_basis(d, self.params["sh_rest"].shape[1])          # (N, K)\n'
      '        return torch.sigmoid(self.params["color_logits"] + (B[..., None] * self.params["sh_rest"]).sum(1))\n\n'
      '    def render(self, rasterize, cam, bg, features=None, opacities=None, **kw):\n'
      '        return rasterize(self.means, self.quats, self.scales,\n'
      '                         self.opacities if opacities is None else opacities,\n'
      '                         self.view_colors(cam) if features is None else features, cam, bg, **kw)\n', 'PATCH_SH_RENDER')
patch(g, '        return cls(*(sd[k] for k in PARAM_NAMES))\n',
      '        return cls(*(sd[k] for k in PARAM_NAMES), sh_rest=sd.get("sh_rest"))   # PATCH_SH_IO\n', 'PATCH_SH_IO')
patch(g, '        return GaussianModel(*(self.params[k].detach()[keep].clone() for k in PARAM_NAMES))\n',
      '        sh = self.params.get("sh_rest")   # PATCH_SH_SUBSET\n'
      '        return GaussianModel(*(self.params[k].detach()[keep].clone() for k in PARAM_NAMES),\n'
      '                             sh_rest=None if sh is None else sh.detach()[keep].clone())\n', 'PATCH_SH_SUBSET')
s = Path(g).read_text(encoding='utf-8')
if 'PATCH_SH_BASIS' not in s:
    s += '''

# PATCH_SH_BASIS: real SH basis without the DC term (constants as in 3DGS sh_utils)
def _sh_basis(d, k):
    x, y, z = d[:, 0], d[:, 1], d[:, 2]
    C1 = 0.4886025119029199
    C2 = (1.0925484305920792, -1.0925484305920792, 0.31539156525252005, -1.0925484305920792, 0.5462742152960396)
    C3 = (-0.5900435899266435, 2.890611442640554, -0.4570457994644658, 0.3731763325901154,
          -0.4570457994644658, 1.445305721320277, -0.5900435899266435)
    xx, yy, zz = x * x, y * y, z * z
    b = [-C1 * y, C1 * z, -C1 * x]
    if k > 3:
        b += [C2[0] * x * y, C2[1] * y * z, C2[2] * (2 * zz - xx - yy), C2[3] * x * z, C2[4] * (xx - yy)]
    if k > 8:
        b += [C3[0] * y * (3 * xx - yy), C3[1] * x * y * z, C3[2] * y * (4 * zz - xx - yy),
              C3[3] * z * (2 * zz - 3 * xx - 3 * yy), C3[4] * x * (4 * zz - xx - yy),
              C3[5] * z * (xx - yy), C3[6] * x * (xx - 3 * yy)]
    return torch.stack(b[:k], -1)
'''
    Path(g).write_text(s, encoding='utf-8'); print(g, ': patched (PATCH_SH_BASIS)')

t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n    sh_degree: int = 0   # PATCH_SH_CFG (model built with enable_sh in phase0)\n', 'PATCH_SH_CFG')
patch(t, '            [{"params": [model.params[k]], "lr": lrs[k], "name": k} for k in PARAM_NAMES], lr=0.0, eps=1e-15)',
      '            [{"params": [model.params[k]], "lr": lrs.get(k, cfg.lr_colors / 20), "name": k}   # PATCH_SH_OPT\n'
      '             for k in model.params], lr=0.0, eps=1e-15)', 'PATCH_SH_OPT')
patch(t, '                self._append({k: m.params[k].detach()[clone] for k in PARAM_NAMES})',
      '                self._append({k: m.params[k].detach()[clone] for k in m.params})   # PATCH_SH_CLONE', 'PATCH_SH_CLONE')
patch(t, '                for k in ("quats", "opacity_logits", "color_logits"):',
      '                for k in [k for k in m.params if k not in ("means", "log_scales")]:   # PATCH_SH_SPLIT', 'PATCH_SH_SPLIT')

patch('scripts/phase0.py',
      '    model = GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev)\n',
      '    model = GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev)\n'
      '    model.enable_sh(int(cfg["train"].get("sh_degree", 0)))   # PATCH_SH_PHASE0\n', 'PATCH_SH_PHASE0')

# self-test (CPU): SH0 unchanged, SH3 trains, densifies, save/load round trip
import importlib, torch, tempfile, os
import frecon.gaussians as G, frecon.train as T
importlib.reload(G); importlib.reload(T)
from frecon.render import get_rasterizer
from frecon.camera import Camera
ras, bg = get_rasterizer('torch'), torch.ones(3)
cams = [Camera(R=torch.eye(3), t=torch.tensor([0.1 * i, 0., 3.]), fx=40., fy=40., cx=16., cy=16., W=32, H=32,
               image=torch.rand(32, 32, 3) * 0.5 + 0.25, name=f'c{i}') for i in range(3)]
for deg in (0, 3):
    m = G.GaussianModel.random_in_box(400, 1.0, torch.Generator().manual_seed(0), 'cpu').enable_sh(deg)
    tr = T.Trainer(m, T.TrainConfig(iterations=30, densify_from=5, densify_until=25, densify_every=5,
                                    log_every=0, sh_degree=deg), 1.0, ras, bg)
    tr.fit(cams, iterations=30, log=None)
    p = os.path.join(tempfile.gettempdir(), f'sh{deg}.pt'); tr.save(p)
    tr2 = T.Trainer.load(p, T.TrainConfig(sh_degree=deg), ras, bg, 'cpu')
    same = torch.allclose(tr.model.render(ras, cams[0], bg)['image'], tr2.model.render(ras, cams[0], bg)['image'])
    has = 'sh_rest' in tr2.model.params
    print(f'SH{deg}: n={tr.model.n}, sh_rest={has}, reload identical={same}')
    assert same and has == (deg > 0)
print('E5 patch OK')


# === E6 PATCH: giu gioi han kich thuoc Gaussian sau pha mat do hoa (train.cap_scale_after_densify) ===
from pathlib import Path
def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')
t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n    cap_scale_after_densify: bool = False   # PATCH_CAP_CFG\n', 'PATCH_CAP_CFG')
patch(t, '        return float(loss)\n',
      '        if (densify and getattr(cfg, "cap_scale_after_densify", False)   # PATCH_CAP_SCALE\n'
      '                and self.step >= cfg.densify_until and self.step % cfg.densify_every == 0):\n'
      '            with torch.no_grad():\n'
      '                big = self.model.scales.max(dim=1).values > 0.1 * self.extent\n'
      '                if big.any():\n'
      '                    self.prune(~big)\n'
      '        return float(loss)\n', 'PATCH_CAP_SCALE')
# tu kiem tra (CPU)
import importlib, torch
import frecon.train as T; importlib.reload(T)
from frecon.gaussians import GaussianModel
from frecon.render import get_rasterizer
from frecon.camera import Camera
ras0, bg0 = get_rasterizer('torch'), torch.ones(3)
cams = [Camera(R=torch.eye(3), t=torch.tensor([0.1 * i, 0., 3.]), fx=40., fy=40., cx=16., cy=16., W=32, H=32,
               image=torch.rand(32, 32, 3), name=f'c{i}') for i in range(3)]
m = GaussianModel.random_in_box(300, 1.0, torch.Generator().manual_seed(0), 'cpu')
tr = T.Trainer(m, T.TrainConfig(iterations=40, densify_from=5, densify_until=20, densify_every=5, log_every=0,
                                cap_scale_after_densify=True), 5.0, ras0, bg0)
with torch.no_grad():
    tr.model.params['log_scales'][:5] = 0.0   # 5 Gaussian scale 1.0 > nguong 0.5
tr.step = 20
tr.fit(cams, iterations=10, log=None)
n_big = int((tr.model.scales.max(1).values > 0.5).sum())
assert n_big == 0 and tr.model.n > 0, n_big
print('E6 patch OK, max scale', round(float(tr.model.scales.max()), 4))


# === PATCH 4: nhiễu toàn cục (E1b), bù chính xác (oracle, E4b), căn chỉnh dạng đóng (M1), khởi tạo visual hull (E9) ===
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

def append(path, code, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    Path(path).write_text(s + code, encoding='utf-8'); print(f'{path}: patched ({tag})')

# (a) photometric.py: mode isp | pixel | global; lưu tham số nhiễu của từng ảnh cho oracle
append('frecon/data/photometric.py', '''

# PATCH_MODES_V2 (runtime patch from colab_rq1_final.ipynb)
ORACLE = {}   # camera name -> (params, mask) of the perturbation applied to that training view


def perturb_views(cams, masks, cfg: dict):
    params = sample_params(len(cams), cfg.get("exposure_stops", 0.0), cfg.get("wb", 0.0),
                           cfg.get("gamma", 0.0), cfg.get("seed", 0))
    mode = cfg.get("mode", "isp")
    if mode == "global":   # one transform for every view: the draw with the largest exposure change
        k0 = max(range(len(params)), key=lambda k: abs(np.log2(params[k]["gain"])))
        params = [params[k0]] * len(params)
    out = []
    for k, (cam, m, p) in enumerate(zip(cams, masks, params)):
        m = torch.as_tensor(np.asarray(m), device=cam.image.device).bool()
        img = apply(cam.image, m, p)
        if mode == "pixel":   # same per-pixel magnitude, random sign -> zero-mean, no per-view component
            g = torch.Generator().manual_seed(int(cfg.get("seed", 0)) * 1000 + k)
            sign = (torch.randint(0, 2, img.shape[:2], generator=g) * 2 - 1).to(img)[..., None]
            img = torch.where(m[..., None], (cam.image + sign * (img - cam.image)).clamp(0, 1), cam.image)
        ORACLE[cam.name] = (p, m.cpu())
        out.append(replace(cam, image=img))
    return out, params
''', 'PATCH_MODES_V2')

# (b) dtu.py: giữ mặt nạ của ảnh huấn luyện cho khởi tạo visual hull
patch('frecon/data/dtu.py', '    train = [cams[i] for i in tr]\n',
      '    train = [cams[i] for i in tr]\n    LAST_TRAIN_MASKS[:] = [masks[i] for i in tr]   # PATCH_HULL_MASKS\n',
      'PATCH_HULL_MASKS')
append('frecon/data/dtu.py', '\n\nLAST_TRAIN_MASKS = []   # PATCH_HULL_GLOBAL\n', 'PATCH_HULL_GLOBAL')

# (c) train.py: appearance = oracle | cf_lin | cf_log (chỉ áp trong hàm mất mát)
t = 'frecon/train.py'
patch(t, '        if getattr(self.cfg, "appearance", "none") != "affine":\n            return img\n',
      '        mode = getattr(self.cfg, "appearance", "none")   # PATCH_APP_MODES\n'
      '        if mode == "oracle":\n'
      '            return _oracle_appearance(img, cam)\n'
      '        if mode in ("cf_lin", "cf_log"):\n'
      '            return _closed_form_appearance(img, cam.image, mode)\n'
      '        if mode != "affine":\n'
      '            return img\n', 'PATCH_APP_MODES')
append(t, '''

# PATCH_APP_FUNCS (runtime patch from colab_rq1_final.ipynb)
def _oracle_appearance(img, cam):
    """Exact forward model of the perturbation of this training view (known parameters)."""
    from frecon.data.photometric import ORACLE
    if cam.name not in ORACLE:
        return img
    p, m = ORACLE[cam.name]
    m = m.to(img.device)
    scale = p["gain"] * torch.tensor(p["wb"], dtype=img.dtype, device=img.device)
    out = (img * scale).clamp(1e-4, 1.0) ** p["gamma"]
    return torch.where(m[..., None], out, img)


def _closed_form_appearance(img, target, mode):
    """Per-view, per-channel affine alignment of the render to the target, solved in closed form at
    every step (least squares, no gradient through the coefficients). cf_log works on log intensity,
    where gain, white balance and gamma of the ISP model are exactly affine per channel."""
    sel = (target < 0.99).any(-1)          # object pixels (background is composited white)
    if int(sel.sum()) < 64:
        return img
    if mode == "cf_log":
        x, y = img.clamp_min(1e-3).log(), target.clamp_min(1e-3).log()
    else:
        x, y = img, target
    with torch.no_grad():
        xs, ys = x[sel], y[sel]
        xm, ym = xs.mean(0), ys.mean(0)
        xc = xs - xm
        a = ((xc * (ys - ym)).mean(0) / xc.pow(2).mean(0).clamp_min(1e-6)).clamp(0.25, 4.0)
        b = ym - a * xm
    out = a * x + b
    if mode == "cf_log":
        out = out.exp()
    return torch.where(sel[..., None], out, img)
''', 'PATCH_APP_FUNCS')

# (d) phase0.py: init.type = hull (điểm ngẫu nhiên trong visual hull của mặt nạ huấn luyện)
ph0 = 'scripts/phase0.py'
patch(ph0, '    model = GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev)\n',
      '    model = (_hull_init(cfg, train_cams, gen, dev) if cfg["init"].get("type") == "hull" else   # PATCH_HULL\n'
      '             GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev))\n',
      'PATCH_HULL')
patch(ph0, 'def train_base(cfg, seed, train_cams, rasterize, bg, dev, path: Path):\n',
      '''def _hull_init(cfg, cams, gen, dev):   # PATCH_HULL_FN
    """Random points inside the visual hull of the training masks: inside the mask of every training
    view that sees them, and inside the image of at least half of the views."""
    import frecon.data.dtu as D
    masks = [torch.as_tensor(m, device=dev) for m in D.LAST_TRAIN_MASKS]
    assert len(masks) == len(cams), "hull init needs the DTU loader (masks of the training views)"
    n, h = cfg["init"]["n_points"], cfg["init"]["half_extent"]
    kept, total = [], 0
    for _ in range(200):
        P = (torch.rand(200_000, 3, generator=gen, device=dev) * 2 - 1) * h
        ok = torch.ones(len(P), dtype=torch.bool, device=dev)
        n_frame = torch.zeros(len(P), device=dev)
        for cam, m in zip(cams, masks):
            Xc = P @ cam.R.to(dev).T + cam.t.to(dev)
            z = Xc[:, 2]
            u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
            v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
            inb = (z > 1e-6) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
            hit = torch.zeros_like(ok)
            hit[inb] = m[v[inb], u[inb]]
            ok &= hit | ~inb
            n_frame += inb
        sel = P[ok & (n_frame >= max(2, len(cams) // 2))]
        kept.append(sel)
        total += len(sel)
        if total >= n:
            break
    P = torch.cat(kept)[:n]
    log(f"hull init: {len(P)} points")
    return GaussianModel.from_points(P, torch.rand(len(P), 3, generator=gen, device=dev))


def train_base(cfg, seed, train_cams, rasterize, bg, dev, path: Path):
''', 'PATCH_HULL_FN')

# ---- tự kiểm tra (CPU, vài giây) ----
import importlib, sys, torch, numpy as np
import frecon.data.photometric as ph, frecon.data.dtu as dtu, frecon.train as T, frecon.gaussians as G
for mod in (ph, dtu, G, T):
    importlib.reload(mod)
sys.path.insert(0, 'scripts')
import phase0; importlib.reload(phase0)
from frecon.camera import Camera
from frecon.render import get_rasterizer
ras, bg = get_rasterizer('torch'), torch.ones(3)
def look(i):   # camera on a circle looking at the origin
    a = 2 * np.pi * i / 4
    c = torch.tensor([3 * np.sin(a), 0., 3 * np.cos(a)], dtype=torch.float32)
    f = -c / c.norm(); r = torch.cross(torch.tensor([0., 1., 0.]), f, dim=0); r = r / r.norm()
    R = torch.stack([r, torch.cross(f, r, dim=0), f]); return R, -R @ c
cams, masks = [], []
for i in range(4):
    R, t = look(i)
    img = torch.ones(32, 32, 3); img[8:24, 8:24] = torch.tensor([0.6, 0.4, 0.3])
    m = np.zeros((32, 32), bool); m[8:24, 8:24] = True
    cams.append(Camera(R=R, t=t, fx=40., fy=40., cx=16., cy=16., W=32, H=32, image=img, name=f'c{i}')); masks.append(m)
cfgp = dict(exposure_stops=1.0, wb=0.15, gamma=0.2, seed=0)
glob_, _ = ph.perturb_views(cams, masks, dict(cfgp, mode='global'))
assert all(torch.equal(glob_[0].image, c.image) for c in glob_), 'global mode must give identical transforms'
isp, _ = ph.perturb_views(cams, masks, cfgp)
orc = T._oracle_appearance(torch.stack([c.image for c in cams])[0], cams[0])
assert torch.allclose(orc, isp[0].image, atol=1e-3), 'oracle must reproduce the perturbed image'
for mode in ('cf_lin', 'cf_log'):
    al = T._closed_form_appearance(cams[0].image, isp[0].image, mode)
    assert (al - isp[0].image).abs().max() < 0.02, (mode, float((al - isp[0].image).abs().max()))
for app in ('oracle', 'cf_log', 'cf_lin', 'affine'):
    m0 = G.GaussianModel.random_in_box(300, 1.0, torch.Generator().manual_seed(0), 'cpu')
    tr = T.Trainer(m0, T.TrainConfig(iterations=20, densify_from=5, densify_until=15, densify_every=5,
                                     log_every=0, appearance=app), 1.0, ras, bg)
    tr.fit(isp, iterations=20, log=None)
    print(f'{app}: 20 steps OK, n={tr.model.n}')
dtu.LAST_TRAIN_MASKS[:] = masks
hm = phase0._hull_init({'init': {'n_points': 500, 'half_extent': 1.0}}, cams, torch.Generator().manual_seed(0), 'cpu')
assert hm.n == 500 and float(hm.means.abs().max()) < 0.8, float(hm.means.abs().max())
print('PATCH 4 OK (global, oracle, closed-form, hull)')


# === PATCH 5 (v2): hoán vị nhiễu, nhiễu không clip, oracle theo clip, GC-ILA, mặt nạ bão hoà, warm-up, snapshot,
# ===               khởi tạo SfM, sửa gradient densification của backend gsplat
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

def append(path, code, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    Path(path).write_text(s + code, encoding='utf-8'); print(f'{path}: patched ({tag})')

# (a) photometric.py: perturb_views V3 = V2 + hoán vị tham số giữa các ảnh (perm) + nhiễu không clip (clip: false)
append('frecon/data/photometric.py', '''

# PATCH_MODES_V3 (runtime patch from colab_v2_mechanism.ipynb); overrides the V2 definition above
def perturb_views(cams, masks, cfg: dict):
    params = sample_params(len(cams), cfg.get("exposure_stops", 0.0), cfg.get("wb", 0.0),
                           cfg.get("gamma", 0.0), cfg.get("seed", 0))
    perm = cfg.get("perm")
    if perm is not None:          # same set of transforms, different assignment to the cameras
        assert sorted(perm) == list(range(len(params))), perm
        params = [params[j] for j in perm]
    mode = cfg.get("mode", "isp")
    clip = bool(cfg.get("clip", True))
    if mode == "global":
        k0 = max(range(len(params)), key=lambda k: abs(np.log2(params[k]["gain"])))
        params = [params[k0]] * len(params)
    out = []
    for k, (cam, m, p) in enumerate(zip(cams, masks, params)):
        m = torch.as_tensor(np.asarray(m), device=cam.image.device).bool()
        scale = p["gain"] * torch.tensor(p["wb"], dtype=cam.image.dtype, device=cam.image.device)
        v = cam.image * scale
        v = v.clamp(0, 1) if clip else v.clamp_min(0)
        img = torch.where(m[..., None], v ** p["gamma"], cam.image)
        if mode == "pixel":
            g = torch.Generator().manual_seed(int(cfg.get("seed", 0)) * 1000 + k)
            sign = (torch.randint(0, 2, img.shape[:2], generator=g) * 2 - 1).to(img)[..., None]
            img = torch.where(m[..., None], (cam.image + sign * (img - cam.image)).clamp(0, 1), cam.image)
        ORACLE[cam.name] = (p, m.cpu(), clip)
        out.append(replace(cam, image=img))
    return out, params
''', 'PATCH_MODES_V3')

# (b) dtu.py: mặt nạ vật thể theo tên camera (để mặt nạ bão hoà không loại nền trắng)
patch('frecon/data/dtu.py', '    LAST_TRAIN_MASKS[:] = [masks[i] for i in tr]   # PATCH_HULL_MASKS\n',
      '    LAST_TRAIN_MASKS[:] = [masks[i] for i in tr]   # PATCH_HULL_MASKS\n'
      '    MASK_BY_NAME.update({cams[i].name: masks[i] for i in tr})   # PATCH_MASK_NAME\n', 'PATCH_MASK_NAME')
append('frecon/data/dtu.py', '\n\nMASK_BY_NAME = {}   # PATCH_MASK_NAME_GLOBAL\n', 'PATCH_MASK_NAME_GLOBAL')

# (c) train.py
t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n'
      '    sat_mask: bool = False      # PATCH_V2_CFG: drop saturated object pixels of the target from the loss\n'
      '    sat_thr: float = 0.98\n'
      '    warmup_geom: int = 0        # first k iterations: geometry frozen, no densification\n'
      '    gc_eta: float = 0.2         # GC-ILA: EMA rate of the per-view coefficients\n'
      '    gc_gauge: bool = True       # GC-ILA: geometric mean of a = 1, mean of b = 0 over views\n'
      '    snapshots: tuple = ()       # iterations at which phase0 saves snap_XXXXX.pt\n', 'PATCH_V2_CFG')
patch(t, '        mode = getattr(self.cfg, "appearance", "none")   # PATCH_APP_MODES\n',
      '        mode = getattr(self.cfg, "appearance", "none")   # PATCH_APP_MODES\n'
      '        if mode == "gc_log":   # PATCH_GC_MODE\n'
      '            return _gc_appearance(self, img, cam)\n', 'PATCH_GC_MODE')
patch(t, '        loss = photometric_loss(self._appearance(out["image"], cam), cam.image, cfg.lambda_dssim)\n',
      '        loss = _loss_v2(self._appearance(out["image"], cam), cam.image, cfg, cam)   # PATCH_LOSS_V2\n', 'PATCH_LOSS_V2')
patch(t, '        self._set_means_lr(means_lr)\n',
      '        frozen = self.step < getattr(cfg, "warmup_geom", 0)   # PATCH_WARMUP\n'
      '        if frozen:\n'
      '            means_lr, densify = 0.0, False\n'
      '        for g in self.opt.param_groups:\n'
      '            if g["name"] in ("log_scales", "quats"):\n'
      '                g.setdefault("lr0", g["lr"])\n'
      '                g["lr"] = 0.0 if frozen else g["lr0"]\n'
      '        self._set_means_lr(means_lr)\n', 'PATCH_WARMUP')
patch(t, '            out["means2d"].retain_grad()\n',
      '            out.get("means2d_full", out["means2d"]).retain_grad()   # PATCH_GSPLAT_GRAD\n', 'PATCH_GSPLAT_GRAD')
patch(t, '                g = out["means2d"].grad\n',
      '                g = out["means2d"].grad   # PATCH_GSPLAT_GRAD2\n'
      '                if "means2d_full" in out and out["means2d_full"].grad is not None:\n'
      '                    g = out["means2d_full"].grad[0]\n', 'PATCH_GSPLAT_GRAD2')
append(t, '''

# PATCH_V2_FUNCS (runtime patch from colab_v2_mechanism.ipynb)
from .losses import ssim_map as _ssim_map
_MASK_CACHE = {}


def _object_mask(cam, device):
    from frecon.data import dtu as _dtu
    m = getattr(_dtu, "MASK_BY_NAME", {}).get(cam.name)
    if m is None:
        return None
    key = (cam.name, str(device))
    if key not in _MASK_CACHE:
        _MASK_CACHE[key] = torch.as_tensor(m, device=device).bool()
    return _MASK_CACHE[key]


def _loss_v2(pred, gt, cfg, cam):
    """Photometric loss; with sat_mask, object pixels whose target is saturated (any channel >= sat_thr) are
    excluded (the composited white background is kept)."""
    if not getattr(cfg, "sat_mask", False):
        return photometric_loss(pred, gt, cfg.lambda_dssim)
    m = _object_mask(cam, gt.device)
    sat = (gt >= getattr(cfg, "sat_thr", 0.98)).any(-1)
    w = (~(sat & m) if m is not None else torch.ones_like(sat)).float()
    den = w.sum().clamp_min(1.0)
    l1 = ((pred - gt).abs().mean(-1) * w).sum() / den
    ss = ((1 - _ssim_map(pred, gt)) * w).sum() / den
    return (1 - cfg.lambda_dssim) * l1 + cfg.lambda_dssim * ss


def _oracle_appearance(img, cam):   # V3: honours the clip flag of the perturbation (overrides V2)
    from frecon.data.photometric import ORACLE
    if cam.name not in ORACLE:
        return img
    e = ORACLE[cam.name]
    p, m, clip = e[0], e[1], (e[2] if len(e) > 2 else True)
    m = m.to(img.device)
    scale = p["gain"] * torch.tensor(p["wb"], dtype=img.dtype, device=img.device)
    v = (img * scale).clamp_min(1e-4)
    if clip:
        v = v.clamp(max=1.0)
    return torch.where(m[..., None], v ** p["gamma"], img)


def _gc_appearance(tr, img, target_cam):
    """GC-ILA: per-view, per-channel affine alignment in log intensity solved in closed form at every step
    (weighted LS, ridge towards slope 1, saturated / dark target pixels excluded), smoothed by an EMA, then
    projected on the gauge {geometric mean of a = 1, mean of b = 0 over the views}. No gradient through the
    coefficients."""
    cfg, target = tr.cfg, target_cam.image
    st = tr.__dict__.setdefault("gc", {})
    dev = img.device
    obj = (target < 0.99).any(-1)                       # object pixels (background composited white)
    x, y = img.clamp_min(1e-3).log(), target.clamp_min(1e-3).log()
    a_prev, b_prev = st.get(target_cam.name, (torch.ones(3, device=dev), torch.zeros(3, device=dev)))
    with torch.no_grad():
        w = (obj[..., None] & (target > 0.02) & (target < getattr(cfg, "sat_thr", 0.98))).float()
        n = w.sum((0, 1))
        if bool((n > 64).all()):
            xm, ym = (w * x).sum((0, 1)) / n, (w * y).sum((0, 1)) / n
            xc, yc = x - xm, y - ym
            cov, var = (w * xc * yc).sum((0, 1)) / n, (w * xc * xc).sum((0, 1)) / n
            lam = 1e-3
            a = ((cov + lam) / (var + lam)).clamp(0.5, 2.0)   # ridge towards a = 1
            b = ym - a * xm
            eta = getattr(cfg, "gc_eta", 0.2)
            st[target_cam.name] = ((1 - eta) * a_prev + eta * a, (1 - eta) * b_prev + eta * b)
        else:
            st[target_cam.name] = (a_prev, b_prev)
        if getattr(cfg, "gc_gauge", True) and len(st) > 1:
            A = torch.stack([v[0] for v in st.values()])
            B = torch.stack([v[1] for v in st.values()])
            ga, gb = A.log().mean(0).exp(), B.mean(0)
            for k in list(st):
                st[k] = (st[k][0] / ga, st[k][1] - gb)
        a, b = st[target_cam.name]
    return torch.where(obj[..., None], (a * x + b).exp(), img)
''', 'PATCH_V2_FUNCS')

# (d) gsplat backend: trả về tensor means2d gốc để retain_grad có gradient (view [0] không nằm trên đồ thị)
patch('frecon/render/gsplat_rast.py',
      '    out = {"image": colors[0], "alpha": alphas[0, ..., 0], "means2d": means2d,\n',
      '    out = {"means2d_full": meta["means2d"],   # PATCH_GSPLAT_FULL\n'
      '           "image": colors[0], "alpha": alphas[0, ..., 0], "means2d": means2d,\n', 'PATCH_GSPLAT_FULL')

# (e) phase0.py: snapshot theo vòng lặp, khởi tạo SfM (điểm triangulate sẵn, màu lấy từ ảnh huấn luyện)
ph0 = 'scripts/phase0.py'
patch(ph0, '    tr.fit(train_cams, seed=seed, log=lambda m: log(m))\n',
      '    snaps = {int(x) for x in (cfg["train"].get("snapshots") or [])}   # PATCH_SNAP\n'
      '    def _snap(it):\n'
      '        if it in snaps:\n'
      '            torch.save({"params": tr.model.state_dict(), "step": it, "extent": tr.extent},\n'
      '                       path.parent / f"snap_{it:05d}.pt")\n'
      '    tr.fit(train_cams, seed=seed, log=lambda m: log(m), callback=_snap if snaps else None)\n', 'PATCH_SNAP')
patch(ph0, '    model = (_hull_init(cfg, train_cams, gen, dev) if cfg["init"].get("type") == "hull" else   # PATCH_HULL\n',
      '    model = (_sfm_init(cfg, train_cams, dev) if cfg["init"].get("type") == "sfm" else   # PATCH_SFM\n'
      '             _hull_init(cfg, train_cams, gen, dev) if cfg["init"].get("type") == "hull" else   # PATCH_HULL\n',
      'PATCH_SFM')
patch(ph0, 'def _hull_init(cfg, cams, gen, dev):   # PATCH_HULL_FN\n',
      '''def _sfm_init(cfg, cams, dev):   # PATCH_SFM_FN
    """Points triangulated beforehand (scripts/sfm_tri.py, clean training views, known poses); colours = mean of
    the (possibly perturbed) training images over the views that see each point, as with COLMAP colours."""
    import numpy as np
    P = torch.from_numpy(np.load(cfg["init"]["points"])).float().to(dev)
    cols = torch.zeros_like(P)
    cnt = torch.zeros(len(P), 1, device=dev)
    for cam in cams:
        Xc = P @ cam.R.to(dev).T + cam.t.to(dev)
        z = Xc[:, 2]
        u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
        v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
        ok = (z > 1e-6) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
        cols[ok] += cam.image.to(dev)[v[ok], u[ok]]
        cnt[ok] += 1
    cols = torch.where(cnt > 0, cols / cnt.clamp_min(1), torch.full_like(cols, 0.5))
    log(f"sfm init: {len(P)} points")
    return GaussianModel.from_points(P, cols.clamp(0.02, 0.98))


def _hull_init(cfg, cams, gen, dev):   # PATCH_HULL_FN
''', 'PATCH_SFM_FN')

# (f) phase0.py: bỏ bước tính tín hiệu sau huấn luyện khi chỉ cần base model (v2 không dùng; với gsplat bước này lỗi
#     vì autograd.grad theo view means2d[0] không nằm trên đồ thị)
patch(ph0, '    sig = cached(sdir / "signals.pt", lambda: {k: v.cpu() for k, v in compute_signals(\n',
      '    if cfg.get("base_only") and cfg.get("skip_signals", False):   # PATCH_SKIP_SIGNALS\n'
      '        log(f"seed {seed}: base model ready (signals skipped)")\n'
      '        return None\n'
      '    sig = cached(sdir / "signals.pt", lambda: {k: v.cpu() for k, v in compute_signals(\n', 'PATCH_SKIP_SIGNALS')

# ---- tự kiểm tra (CPU, khoảng 1 phút) ----
import importlib, sys, tempfile, torch, numpy as np
import frecon.data.photometric as ph, frecon.data.dtu as dtu, frecon.train as T, frecon.gaussians as G
for mod in (ph, dtu, G, T):
    importlib.reload(mod)
sys.path.insert(0, 'scripts')
import phase0; importlib.reload(phase0)
from frecon.camera import Camera
from frecon.render import get_rasterizer
ras, bg = get_rasterizer('torch'), torch.ones(3)
def look(i, n=4):
    a = 2 * np.pi * i / n
    c = torch.tensor([3 * np.sin(a), 0.3 * (i % 2), 3 * np.cos(a)], dtype=torch.float32)
    f = -c / c.norm(); r = torch.cross(torch.tensor([0., 1., 0.]), f, dim=0); r = r / r.norm()
    R = torch.stack([r, torch.cross(f, r, dim=0), f]); return R, -R @ c
cams, masks = [], []
gimg = torch.Generator().manual_seed(1)
for i in range(4):
    R, t = look(i)
    img = torch.ones(32, 32, 3); img[8:24, 8:24] = torch.rand(16, 16, 3, generator=gimg) * 0.8 + 0.1
    m = np.zeros((32, 32), bool); m[8:24, 8:24] = True
    cams.append(Camera(R=R, t=t, fx=40., fy=40., cx=16., cy=16., W=32, H=32, image=img, name=f'v2c{i}')); masks.append(m)
dtu.MASK_BY_NAME.update({c.name: m for c, m in zip(cams, masks)})
base = dict(exposure_stops=1.0, wb=0.15, gamma=0.2, seed=0)
p_id, P0 = ph.perturb_views(cams, masks, base)
p_pm, P1 = ph.perturb_views(cams, masks, dict(base, perm=[2, 0, 3, 1]))
assert P1[0] == P0[2] and P1[3] == P0[1], 'perm'
nc, _ = ph.perturb_views(cams, masks, dict(base, clip=False, exposure_stops=2.0))
assert float(max(c.image.max() for c in nc)) > 1.0, 'noclip should exceed 1'
for c_clean, c_noisy in zip(cams, nc):
    assert torch.allclose(T._oracle_appearance(c_clean.image, c_noisy), c_noisy.image, atol=1e-4), 'oracle noclip'
pc, _ = ph.perturb_views(cams, masks, base)
assert torch.allclose(T._oracle_appearance(cams[0].image, pc[0]), pc[0].image, atol=1e-3), 'oracle clip'
# loss with saturation mask: background kept, saturated object pixels dropped
tc = T.TrainConfig(sat_mask=True)
pred = pc[0].image.clone(); pred[0, 0] = 0.0                                  # background error must count
assert float(T._loss_v2(pred, pc[0].image, tc, pc[0])) > 0
# GC-ILA, warm-up (geometry frozen), snapshots, SfM init through phase0.train_base
for app in ('gc_log', 'oracle', 'affine'):
    m0 = G.GaussianModel.random_in_box(300, 1.0, torch.Generator().manual_seed(0), 'cpu')
    tr = T.Trainer(m0, T.TrainConfig(iterations=24, densify_from=5, densify_until=20, densify_every=5, log_every=0,
                                     appearance=app, sat_mask=True, warmup_geom=10), 1.0, ras, bg)
    mu0 = tr.model.means.detach().clone(); n0 = tr.model.n
    tr.fit(pc, iterations=10, log=None)
    assert torch.equal(tr.model.means.detach(), mu0) and tr.model.n == n0, f'{app}: warm-up must freeze geometry'
    tr.fit(pc, iterations=14, log=None)
    assert not torch.equal(tr.model.means.detach()[:n0], mu0) or tr.model.n != n0, f'{app}: geometry must move after warm-up'
    if app == 'gc_log':
        A = torch.stack([v[0] for v in tr.gc.values()])
        assert abs(float(A.log().mean(0).abs().max())) < 1e-4, 'gauge'
    print(f'{app}: warm-up + {tr.model.n} Gaussians OK')
tmp = Path(tempfile.mkdtemp())
np.save(tmp / 'pts.npy', (np.random.default_rng(0).uniform(-0.3, 0.3, (500, 3))).astype(np.float32))
cfg = {'train': dict(iterations=20, densify_from=5, densify_until=15, densify_every=5, log_every=0, snapshots=[10]),
       'init': {'type': 'sfm', 'points': str(tmp / 'pts.npy'), 'n_points': 10, 'half_extent': 1.0}}
(tmp / 'seed_0').mkdir()
phase0.train_base(cfg, 0, pc, ras, bg, 'cpu', tmp / 'seed_0' / 'base.pt')
assert (tmp / 'seed_0' / 'snap_00010.pt').exists() and (tmp / 'seed_0' / 'base.pt').exists(), 'snapshot / base'
print('PATCH 5 OK (perm, noclip, oracle-clip, sat-mask, GC-ILA, warm-up, snapshot, SfM init, gsplat grad)')


# === PATCH 6 (v3): can thiệp nhân quả (E2-A, E2-B), fork từ checkpoint chung, thống kê Gaussian/gradient theo GT
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

def append(path, code, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    Path(path).write_text(s + code, encoding='utf-8'); print(f'{path}: patched ({tag})')

t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n'
      '    interv: str = "none"          # PATCH_V3_CFG: none | cap | pos | cap_pos (targeted, E2-B)\n'
      '    interv_target: str = "off"    # off (GT off-surface) | rand | near (matched controls)\n'
      '    interv_from: int = 0\n'
      '    interv_dynamic: bool = True   # re-label every stat_every steps (full topology)\n'
      '    interv_cap: str = "abs"       # abs: opacity <= interv_cap_value; ref: opacity <= value at interv_from\n'
      '    interv_cap_value: float = 0.1\n'
      '    interv_seed: int = 0\n'
      '    freeze: tuple = ()            # global attribute freeze (E2-A): opacity, means, scale, rot, color\n'
      '    freeze_from: int = 0\n'
      '    topo_freeze_from: int = -1    # >= 0: no clone / split / prune from this step (opacity reset kept)\n'
      '    gt_ref: str = ""              # GT points (.npy) for labels and statistics (oracle diagnostic only)\n'
      '    gt_tau: float = 0.1\n'
      '    gt_ref_max: int = 50000\n'
      '    stat_every: int = 100\n'
      '    save_full: tuple = ()         # steps at which phase0 saves a full checkpoint fork_XXXXX.pt\n', 'PATCH_V3_CFG')
patch(t, '        self.opt.step()\n        if self.app_opt is not None:   # PATCH_APP_OPT\n',
      '        _v3 = _v3_pre(self)   # PATCH_V3_STEP\n'
      '        self.opt.step()\n'
      '        _v3_post(self, _v3)\n'
      '        if self.app_opt is not None:   # PATCH_APP_OPT\n', 'PATCH_V3_STEP')
patch(t, '    def densify_and_prune(self, prune_big: bool):\n        cfg, m = self.cfg, self.model\n',
      '    def densify_and_prune(self, prune_big: bool):\n        cfg, m = self.cfg, self.model\n'
      '        if 0 <= getattr(cfg, "topo_freeze_from", -1) <= self.step:   # PATCH_TOPO_FREEZE\n'
      '            self._reset_stats()\n'
      '            return\n', 'PATCH_TOPO_FREEZE')
patch(t, '    def reset_opacity(self):\n        m = self.model\n',
      '    def reset_opacity(self):\n        m = self.model\n'
      '        if "opacity" in _active_freeze(self):   # PATCH_FREEZE_RESET\n'
      '            return\n', 'PATCH_FREEZE_RESET')
append(t, '''

# PATCH_V3_FUNCS (runtime patch from colab_v3_causal.ipynb)
from .gaussians import nearest_dist as _nearest_dist
_FREEZE_MAP = {"opacity": ("opacity_logits",), "means": ("means",), "scale": ("log_scales",), "rot": ("quats",),
               "color": ("color_logits", "sh_rest")}


def _active_freeze(tr):
    cfg = tr.cfg
    if not getattr(cfg, "freeze", ()) or tr.step < getattr(cfg, "freeze_from", 0):
        return set()
    return set(cfg.freeze)


def _relabel(tr):
    """GT distance of every Gaussian centre (subsampled GT points); oracle diagnostic, never a test-time signal."""
    cfg = tr.cfg
    if not hasattr(tr, "_gt"):
        import numpy as np
        P = torch.from_numpy(np.load(cfg.gt_ref)).float()
        if len(P) > cfg.gt_ref_max:
            P = P[torch.randperm(len(P), generator=torch.Generator().manual_seed(0))[:cfg.gt_ref_max]]
        tr._gt = P.to(tr.model.means.device)
    tr._d = _nearest_dist(tr.model.means.detach(), tr._gt)
    tr._off = tr._d > cfg.gt_tau


def _targets(tr):
    """Target set of the intervention: GT off-surface Gaussians, or a matched control with the same count and
    the same opacity distribution (deciles of the off-surface opacities), drawn from all Gaussians (rand) or
    from near-surface ones (near)."""
    cfg, off = tr.cfg, tr._off
    if cfg.interv_target == "off" or not bool(off.any()):
        return off.clone() if cfg.interv_target == "off" else torch.zeros_like(off)
    pool = torch.ones_like(off) if cfg.interv_target == "rand" else ~off
    op = tr.model.opacities.detach().reshape(-1)
    qs = torch.quantile(op[off].float(), torch.linspace(0, 1, 11, device=op.device))
    qs[0], qs[-1] = -1.0, 2.0
    g = torch.Generator().manual_seed(int(cfg.interv_seed) * 100003 + tr.step)
    sel = torch.zeros_like(off)
    for b in range(10):
        inb = (op > qs[b]) & (op <= qs[b + 1])
        need = int((off & inb).sum())
        cand = torch.nonzero(pool & inb & ~sel).flatten()
        if need and len(cand):
            pick = cand[torch.randperm(len(cand), generator=g)[:need].to(cand.device)]
            sel[pick] = True
    return sel


def _record(tr):
    m, off = tr.model, tr._off
    op = m.opacities.detach().reshape(-1)
    surf = ~off
    row = {"step": tr.step, "n": m.n, "n_off": int(off.sum()),
           "a_off": float(op[off].mean()) if off.any() else 0.0,
           "a_surf": float(op[surf].mean()) if surf.any() else 0.0,
           "mass_off": float(op[off].sum() / op.sum().clamp_min(1e-12)),
           "n_target": int(tr._tmask.sum()) if getattr(tr, "_tmask", None) is not None else 0}
    for k, p in m.params.items():
        if p.grad is not None:
            g2 = p.grad.detach().reshape(m.n, -1).pow(2).sum(1)
            row[f"g_{k}_off"] = float(g2[off].mean().sqrt()) if off.any() else 0.0
            row[f"g_{k}_surf"] = float(g2[surf].mean().sqrt()) if surf.any() else 0.0
    tr.__dict__.setdefault("stats", []).append(row)


def _v3_pre(tr):
    """Called after backward, before the optimizer step: GT labels and statistics, intervention targets, and the
    values that the post-step hook restores (hard freeze: parameters and Adam moments of frozen rows)."""
    cfg, out = tr.cfg, {}
    frz = _active_freeze(tr)
    active = getattr(cfg, "interv", "none") != "none" and tr.step >= cfg.interv_from
    if not (getattr(cfg, "gt_ref", "") or frz or active):
        return out
    n = tr.model.n
    if getattr(cfg, "gt_ref", ""):
        stale = getattr(tr, "_off", None) is None or tr._off.numel() != n
        if stale or tr.step % cfg.stat_every == 0:
            _relabel(tr)
            if tr.step % cfg.stat_every == 0:
                _record(tr)
        tm = getattr(tr, "_tmask", None)
        if active and (tm is None or tm.numel() != n or (cfg.interv_dynamic and tr.step % cfg.stat_every == 0)):
            tr._tmask = _targets(tr)
    P = tr.model.params
    for name in frz:
        for k in _FREEZE_MAP[name]:
            if k in P:
                out[k] = (None, P[k].detach().clone())
    if active:
        tm = tr._tmask
        if "pos" in cfg.interv and "means" not in out:
            out["means"] = (tm, P["means"].detach()[tm].clone())
        if "cap" in cfg.interv:
            if cfg.interv_cap == "ref" and (getattr(tr, "_op_ref", None) is None or tr._op_ref.numel() != n):
                tr._op_ref = P["opacity_logits"].detach().clone()
            out["_cap"] = tm
    return out


def _v3_post(tr, out):
    if not out:
        return
    cfg, P = tr.cfg, tr.model.params
    with torch.no_grad():
        for k, v in out.items():
            if k.startswith("_"):
                continue
            mask, old = v
            p = P[k]
            st = tr.opt.state.get(p, {})
            if mask is None:
                p.copy_(old)
                for s in ("exp_avg", "exp_avg_sq"):
                    if s in st:
                        st[s].zero_()
            elif bool(mask.any()):
                p[mask] = old
                for s in ("exp_avg", "exp_avg_sq"):
                    if s in st:
                        st[s][mask] = 0
        if "_cap" in out and bool(out["_cap"].any()):
            lg, mask = P["opacity_logits"], out["_cap"]
            if cfg.interv_cap == "ref":
                lim = tr._op_ref
            else:
                lim = torch.full_like(lg, float(inverse_sigmoid(torch.tensor(cfg.interv_cap_value))))
            lg[mask] = torch.minimum(lg[mask], lim[mask])
''', 'PATCH_V3_FUNCS')

ph0 = 'scripts/phase0.py'
patch(ph0, '    tr.save(path)\n    return tr\n',
      '    _write_stats(tr, path)   # PATCH_STATS\n    tr.save(path)\n    return tr\n', 'PATCH_STATS')
patch(ph0, '    snaps = {int(x) for x in (cfg["train"].get("snapshots") or [])}   # PATCH_SNAP\n    def _snap(it):\n',
      '    snaps = {int(x) for x in (cfg["train"].get("snapshots") or [])}   # PATCH_SNAP\n'
      '    fulls = {int(x) for x in (cfg["train"].get("save_full") or [])}   # PATCH_FULL\n'
      '    def _snap(it):\n'
      '        if it in fulls:\n'
      '            tr.save(path.parent / f"fork_{it:05d}.pt")\n', 'PATCH_FULL')
patch(ph0, 'callback=_snap if snaps else None)\n', 'callback=_snap if (snaps or fulls) else None)\n', 'callback=_snap if (snaps or fulls)')
patch(ph0, '        return Trainer.load(path, tcfg, rasterize, bg, dev)\n    set_seed(seed)\n',
      '''        return Trainer.load(path, tcfg, rasterize, bg, dev)
    if cfg["init"].get("type") == "fork":   # PATCH_FORK: continue a shared checkpoint (same state for every branch)
        ck = cfg["init"]["ckpt"].replace("{seed}", str(seed))
        tr = Trainer.load(ck, tcfg, rasterize, bg, dev)
        s0 = tr.step
        set_seed(seed * 7919 + s0)
        fsnaps = {int(x) for x in (cfg["train"].get("snapshots") or [])}
        def _fsnap(it):
            if tr.step in fsnaps:
                torch.save({"params": tr.model.state_dict(), "step": tr.step, "extent": tr.extent},
                           path.parent / f"snap_{tr.step:05d}.pt")
        log(f"seed {seed}: fork from step {s0} of {ck}")
        t0 = time.time()
        tr.fit(train_cams, iterations=tcfg.iterations - s0, seed=seed * 7919 + s0, log=lambda m: log(m),
               callback=_fsnap if fsnaps else None)
        log(f"seed {seed}: base done in {(time.time() - t0) / 60:.1f} min, {tr.model.n} Gaussians")
        _write_stats(tr, path)
        tr.save(path)
        return tr
    set_seed(seed)
''', 'PATCH_FORK')
patch(ph0, 'def _sfm_init(cfg, cams, dev):   # PATCH_SFM_FN\n',
      '''def _write_stats(tr, path):   # PATCH_STATS_FN
    rows = getattr(tr, "stats", None)
    if rows:
        import csv
        keys = sorted({k for r in rows for k in r})
        with open(path.parent / "gaussian_stats.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(rows)


def _sfm_init(cfg, cams, dev):   # PATCH_SFM_FN
''', 'PATCH_STATS_FN')

# ---- tự kiểm tra (CPU) ----
import importlib, sys, tempfile, torch, numpy as np
import frecon.train as T, frecon.gaussians as G, frecon.data.dtu as dtu, frecon.data.photometric as ph
for mod in (G, T, dtu, ph):
    importlib.reload(mod)
sys.path.insert(0, 'scripts')
import phase0; importlib.reload(phase0)
from frecon.camera import Camera
from frecon.render import get_rasterizer
ras, bg = get_rasterizer('torch'), torch.ones(3)
def look(i, n=4):
    a = 2 * np.pi * i / n
    c = torch.tensor([3 * np.sin(a), 0.3 * (i % 2), 3 * np.cos(a)], dtype=torch.float32)
    f = -c / c.norm(); r = torch.cross(torch.tensor([0., 1., 0.]), f, dim=0); r = r / r.norm()
    R = torch.stack([r, torch.cross(f, r, dim=0), f]); return R, -R @ c
cams = []
gimg = torch.Generator().manual_seed(1)
for i in range(4):
    R, t_ = look(i)
    img = torch.ones(32, 32, 3); img[8:24, 8:24] = torch.rand(16, 16, 3, generator=gimg) * 0.8 + 0.1
    cams.append(Camera(R=R, t=t_, fx=40., fy=40., cx=16., cy=16., W=32, H=32, image=img, name=f'v3c{i}'))
tmp = Path(tempfile.mkdtemp())
np.save(tmp / 'gt.npy', (np.random.default_rng(0).uniform(-0.15, 0.15, (3000, 3))).astype(np.float32))
np.save(tmp / 'init.npy', (np.random.default_rng(1).uniform(-0.5, 0.5, (600, 3))).astype(np.float32))
base_tr = dict(iterations=40, densify_from=5, densify_until=30, densify_every=5, opacity_reset_every=1000,
               log_every=0, gt_ref=str(tmp / 'gt.npy'), stat_every=5, save_full=[10])
def run(name, train_extra, init):
    d = tmp / name / 'seed_0'; d.mkdir(parents=True)
    cfg = {'train': dict(base_tr, **train_extra), 'init': init}
    return phase0.train_base(cfg, 0, cams, ras, bg, 'cpu', d / 'base.pt')
par = run('parent', {}, {'type': 'sfm', 'points': str(tmp / 'init.npy'), 'n_points': 10, 'half_extent': 1.0})
fk = str(tmp / 'parent' / 'seed_{seed}' / 'fork_00010.pt')
assert Path(fk.replace('{seed}', '0')).exists() and (tmp / 'parent' / 'seed_0' / 'gaussian_stats.csv').exists()
ck = T.Trainer.load(fk.replace('{seed}', '0'), T.TrainConfig(**base_tr), ras, bg, 'cpu')
n10 = ck.model.n
# E2-B fixed topology, cap at reference opacity on GT off-surface Gaussians + freeze their positions
b3 = run('b3', dict(interv='cap_pos', interv_from=10, interv_dynamic=False, interv_cap='ref', topo_freeze_from=10),
         {'type': 'fork', 'ckpt': fk})
assert b3.model.n == n10, 'topology must stay fixed'
T._relabel(ck); off0 = ck._off
assert torch.equal(b3.model.means.detach()[off0], ck.model.means.detach()[off0]), 'positions of targets frozen'
assert bool((b3.model.params['opacity_logits'].detach()[off0] <= ck.model.params['opacity_logits'].detach()[off0] + 1e-6).all()), 'cap'
# matched controls have the same size as the off-surface set
ck.cfg.interv_target = 'near'; ck._off = off0; sel = T._targets(ck)
assert int(sel.sum()) <= int(off0.sum()) and not bool((sel & off0).any()), 'near control'
# E2-A global freeze of opacity (hard: no Adam drift, no reset)
a1 = run('a1', dict(freeze=['opacity'], freeze_from=10, topo_freeze_from=10), {'type': 'fork', 'ckpt': fk})
assert torch.equal(a1.model.params['opacity_logits'].detach(), ck.model.params['opacity_logits'].detach()), 'opacity frozen'
# full topology with absolute cap and dynamic labels
b1 = run('b1', dict(interv='cap', interv_cap='abs', interv_cap_value=0.1), {'type': 'sfm', 'points': str(tmp / 'init.npy'),
                                                                            'n_points': 10, 'half_extent': 1.0})
import csv as _csv
rows = list(_csv.DictReader(open(tmp / 'b1' / 'seed_0' / 'gaussian_stats.csv')))
assert rows and max(int(r['n_target']) for r in rows) > 0, 'targets recorded'
print('PATCH 6 OK (fork, targeted cap/pos, matched controls, global freeze, topology freeze, GT statistics)')


# === PATCH 7 (v4): can thiệp chọn lọc lên kích thước / màu, xoá Gaussian theo GT, cửa sổ can thiệp (interv_until)
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

t = 'frecon/train.py'
patch(t, '    interv_from: int = 0\n',
      '    interv_from: int = 0\n    interv_until: int = -1        # PATCH_V4_CFG: >= 0: intervention only before this step\n', 'PATCH_V4_CFG')
patch(t, '    active = getattr(cfg, "interv", "none") != "none" and tr.step >= cfg.interv_from\n',
      '    active = (getattr(cfg, "interv", "none") != "none" and tr.step >= cfg.interv_from   # PATCH_V4_WINDOW\n'
      '              and (getattr(cfg, "interv_until", -1) < 0 or tr.step < cfg.interv_until))\n', 'PATCH_V4_WINDOW')
patch(t, '        if "pos" in cfg.interv and "means" not in out:\n            out["means"] = (tm, P["means"].detach()[tm].clone())\n',
      '        if "pos" in cfg.interv and "means" not in out:\n            out["means"] = (tm, P["means"].detach()[tm].clone())\n'
      '        if "scl" in cfg.interv and "log_scales" not in out:   # PATCH_V4_ATTR\n'
      '            out["log_scales"] = (tm, P["log_scales"].detach()[tm].clone())\n'
      '        if "col" in cfg.interv:\n'
      '            for k in ("color_logits", "sh_rest"):\n'
      '                if k in P and k not in out:\n'
      '                    out[k] = (tm, P[k].detach()[tm].clone())\n'
      '        if "del" in cfg.interv and tr.step % cfg.stat_every == 0:\n'
      '            out["_del"] = tm\n', 'PATCH_V4_ATTR')
patch(t, '            lg[mask] = torch.minimum(lg[mask], lim[mask])\n',
      '            lg[mask] = torch.minimum(lg[mask], lim[mask])\n'
      '    if "_del" in out and bool(out["_del"].any()) and int(out["_del"].sum()) < tr.model.n:   # PATCH_V4_DEL\n'
      '        tr.prune(~out["_del"])\n'
      '        tr._off, tr._tmask = None, None\n', 'PATCH_V4_DEL')

# ---- tự kiểm tra (CPU, vài giây) ----
import importlib, sys, tempfile, torch, numpy as np
import frecon.train as T, frecon.gaussians as G
importlib.reload(G); importlib.reload(T)
sys.path.insert(0, 'scripts')
import phase0; importlib.reload(phase0)
from frecon.camera import Camera
from frecon.render import get_rasterizer
ras, bg = get_rasterizer('torch'), torch.ones(3)
def look(i, n=4):
    a = 2 * np.pi * i / n
    c = torch.tensor([3 * np.sin(a), 0.3 * (i % 2), 3 * np.cos(a)], dtype=torch.float32)
    f = -c / c.norm(); r = torch.cross(torch.tensor([0., 1., 0.]), f, dim=0); r = r / r.norm()
    R = torch.stack([r, torch.cross(f, r, dim=0), f]); return R, -R @ c
cams = []
gimg = torch.Generator().manual_seed(1)
for i in range(4):
    R, t_ = look(i)
    img = torch.ones(32, 32, 3); img[8:24, 8:24] = torch.rand(16, 16, 3, generator=gimg) * 0.8 + 0.1
    cams.append(Camera(R=R, t=t_, fx=40., fy=40., cx=16., cy=16., W=32, H=32, image=img, name=f'v4c{i}'))
tmp = Path(tempfile.mkdtemp())
np.save(tmp / 'gt.npy', (np.random.default_rng(0).uniform(-0.15, 0.15, (3000, 3))).astype(np.float32))
np.save(tmp / 'init.npy', (np.random.default_rng(1).uniform(-0.5, 0.5, (600, 3))).astype(np.float32))
init = {'type': 'sfm', 'points': str(tmp / 'init.npy'), 'n_points': 10, 'half_extent': 1.0}
base = dict(iterations=30, densify_from=5, densify_until=25, densify_every=5, opacity_reset_every=1000, log_every=0,
            gt_ref=str(tmp / 'gt.npy'), stat_every=5, interv_from=0, interv_dynamic=True, interv_cap='abs', interv_cap_value=0.1)
def run(name, extra):
    d = tmp / name / 'seed_0'; d.mkdir(parents=True)
    return phase0.train_base({'train': dict(base, **extra), 'init': init}, 0, cams, ras, bg, 'cpu', d / 'base.pt')
tr = run('del', dict(interv='del', interv_target='off'))
T._relabel(tr)
print('del: Gaussian ngoài bề mặt còn lại', int(tr._off.sum()), '/', tr.model.n)
tr = run('all', dict(interv='cap_scl_col', interv_target='off'))
tr = run('win', dict(interv='cap', interv_target='off', interv_until=10))
print('PATCH 7 OK (scale / colour targeted freeze, GT deletion, intervention window)')


# === PATCH 8 (v4/v5): CAGS hooks (probe, risk, gate / att / mod), lineage of Gaussians, P-B / P-C perturbations
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

def append(path, code, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    Path(path).write_text(s + code, encoding='utf-8'); print(f'{path}: patched ({tag})')

t = 'frecon/train.py'
patch(t, '    save_full: tuple = ()         # steps at which phase0 saves a full checkpoint fork_XXXXX.pt\n',
      '    save_full: tuple = ()         # steps at which phase0 saves a full checkpoint fork_XXXXX.pt\n'
      '    cags_mode: str = "none"       # PATCH_V5_CFG: none | log | oracle_geo | oracle_cmp | det | rule\n'
      '    cags_actions: str = "full"    # full | gate | att | mod | gate+att ...\n'
      '    cags_from: int = 250          # first step of the actions (a probe is run at that step)\n'
      '    cags_every: int = 100         # probe period (risk update)\n'
      '    cags_ema: float = 0.5\n'
      '    cags_tau_s: float = 0.5       # suppression strength m = clip((r - tau_s) / (1 - tau_s), 0, 1)\n'
      '    cags_tau_gate: float = 0.5\n'
      '    cags_gamma: float = 2.0       # opacity cap (1 - m) ** gamma\n'
      '    cags_budget: float = 1.0      # at most this fraction of the Gaussians with m > 0\n'
      '    cags_model: str = ""          # detector checkpoint (mode det)\n'
      '    cags_log_steps: tuple = ()    # steps at which features + GT labels are logged\n'
      '    cags_log_max: int = 20000\n', 'PATCH_V5_CFG')
patch(t, '        rng = random.Random(seed)\n        iterations = self.cfg.iterations if iterations is None else iterations\n',
      '        rng = random.Random(seed)\n        self._cams = cams   # PATCH_V5_CAMS\n'
      '        iterations = self.cfg.iterations if iterations is None else iterations\n', 'PATCH_V5_CAMS')
patch(t, '        _v3 = _v3_pre(self)   # PATCH_V3_STEP\n        self.opt.step()\n        _v3_post(self, _v3)\n',
      '        _v3 = _v3_pre(self)   # PATCH_V3_STEP\n        _cg = _cags.pre_step(self)   # PATCH_V5_STEP\n'
      '        self.opt.step()\n        _v3_post(self, _v3)\n        _cags.post_step(self, _cg)\n', 'PATCH_V5_STEP')
patch(t, '        if (densify and getattr(cfg, "cap_scale_after_densify", False)   # PATCH_CAP_SCALE\n',
      '        _cags.tick(self)   # PATCH_V5_TICK\n'
      '        if (densify and getattr(cfg, "cap_scale_after_densify", False)   # PATCH_CAP_SCALE\n', 'PATCH_V5_TICK')
patch(t, '            split = (grads >= cfg.densify_grad_thresh) & big\n',
      '            split = (grads >= cfg.densify_grad_thresh) & big\n'
      '            _g = _cags.gate(self)   # PATCH_V5_GATE\n'
      '            if _g is not None and _g.numel() == clone.numel():\n'
      '                clone &= ~_g\n'
      '                split &= ~_g\n', 'PATCH_V5_GATE')
patch(t, '                self._append({k: m.params[k].detach()[clone] for k in m.params})   # PATCH_SH_CLONE\n',
      '                self._par = torch.nonzero(clone).flatten()   # PATCH_V5_PAR\n'
      '                self._append({k: m.params[k].detach()[clone] for k in m.params})   # PATCH_SH_CLONE\n', 'PATCH_V5_PAR')
patch(t, '                self._append(new)\n',
      '                self._par = torch.nonzero(split).flatten().repeat(2)   # PATCH_V5_PAR2\n'
      '                self._append(new)\n', 'PATCH_V5_PAR2')
patch(t, '        self.grad_denom = torch.cat([self.grad_denom, torch.zeros(n_new, device=dev)])\n',
      '        self.grad_denom = torch.cat([self.grad_denom, torch.zeros(n_new, device=dev)])\n'
      '        _cags.on_append(self, n_new)   # PATCH_V5_APPEND\n', 'PATCH_V5_APPEND')
patch(t, '        self.grad_accum = self.grad_accum[keep]\n        self.grad_denom = self.grad_denom[keep]\n',
      '        self.grad_accum = self.grad_accum[keep]\n        self.grad_denom = self.grad_denom[keep]\n'
      '        _cags.on_prune(self, keep)   # PATCH_V5_PRUNE\n', 'PATCH_V5_PRUNE')
patch(t, '        torch.save({"params": self.model.state_dict(), "opt": opt_state, "step": self.step,\n'
         '                    "extent": self.extent}, path)\n',
      '        torch.save({"params": self.model.state_dict(), "opt": opt_state, "step": self.step,\n'
      '                    "extent": self.extent, "cags_aux": getattr(self, "_aux", None)}, path)   # PATCH_V5_SAVE\n',
      'PATCH_V5_SAVE')
patch(t, '        tr.step = ck["step"]\n        return tr\n',
      '        tr.step = ck["step"]\n'
      '        if ck.get("cags_aux") is not None:   # PATCH_V5_LOAD\n'
      '            tr._aux = {k: v.to(device) for k, v in ck["cags_aux"].items()}\n'
      '        return tr\n', 'PATCH_V5_LOAD')
append(t, '\n\nfrom . import cags as _cags   # PATCH_V5_IMPORT\n', 'PATCH_V5_IMPORT')

patch('scripts/phase0.py', 'def _write_stats(tr, path):   # PATCH_STATS_FN\n',
      'def _write_stats(tr, path):   # PATCH_STATS_FN\n'
      '    from frecon.cags import write_log as _cags_write   # PATCH_V5_LOG\n'
      '    _cags_write(tr, path)\n', 'PATCH_V5_LOG')

append('frecon/data/photometric.py', '''

# PATCH_MODES_V5 (runtime patch from colab_v4_v5_cags.ipynb): spatially varying synthetic perturbations.
# P-B  field: poly   smooth zero-mean quadratic field + vignetting, slight per-channel scaling (log2 units)
# P-C  field: blobs  3-6 local Gaussian blobs of +/- amplitude (brighter / darker spots), per-channel tint
# The image becomes clip(I * 2 ** F, 0, 1) inside the object mask. Both are synthetic stress tests of spatially
# varying inconsistency, not physical models of shading or reflections. CLEAN keeps the unperturbed training
# images; it is read only to build labels (log phi = log I' - log I), never by a GT-free method.
CLEAN = {}
_perturb_v3 = perturb_views


def _field(kind, H, W, mask, rng, stops, cfg):
    yy, xx = np.meshgrid(np.linspace(-1, 1, H), np.linspace(-1, 1, W), indexing="ij")
    if kind == "poly":
        c = rng.uniform(-1, 1, 5)
        F = c[0] * xx + c[1] * yy + c[2] * xx ** 2 + c[3] * xx * yy + c[4] * yy ** 2
        F = F - F[mask].mean()
        F = F / (np.abs(F[mask]).max() + 1e-8) * stops * rng.uniform(0.5, 1.0)
        r2 = xx ** 2 + yy ** 2
        F = F - rng.uniform(0, cfg.get("vignette", 0.5)) * stops * (r2 - r2[mask].mean())
        return F[..., None] * (1 + rng.uniform(-0.15, 0.15, 3))
    if kind == "blobs":
        F = np.zeros((H, W, 3))
        ys, xs = np.nonzero(mask)
        if len(ys) == 0:
            return F
        Y, X = np.arange(H)[:, None], np.arange(W)[None]
        for _ in range(int(rng.integers(cfg.get("n_min", 3), cfg.get("n_max", 6) + 1))):
            k = int(rng.integers(len(ys)))
            s = rng.uniform(0.04, 0.15) * max(H, W)
            amp = rng.uniform(0.5, 1.0) * stops * (1 if rng.uniform() < 0.5 else -1)
            g = np.exp(-((Y - ys[k]) ** 2 + (X - xs[k]) ** 2) / (2 * s * s))
            F += g[..., None] * amp * (1 + rng.uniform(-0.3, 0.3, 3))
        return F
    raise ValueError(kind)


def perturb_views(cams, masks, cfg: dict):
    kind = cfg.get("field")
    clean = [cam.image for cam in cams]
    isp = any(cfg.get(k, 0.0) for k in ("exposure_stops", "wb", "gamma"))
    if not kind:
        out, params = _perturb_v3(cams, masks, cfg)
    else:   # field alone, or global ISP (P-A) first and the field on top (P-AC)
        base, _ = _perturb_v3(cams, masks, cfg) if isp else (cams, None)
        out, params = [], []
        off = {"poly": 101, "blobs": 202}[kind]
        for k, (cam, m) in enumerate(zip(base, masks)):
            m = np.asarray(m).astype(bool)
            rng = np.random.default_rng(int(cfg.get("seed", 0)) * 1000 + off + k)
            H, W = cam.image.shape[:2]
            F = torch.from_numpy(_field(kind, H, W, m, rng, float(cfg.get("stops", 1.0)), cfg)).to(cam.image)
            mt = torch.as_tensor(m, device=cam.image.device)
            img = torch.where(mt[..., None], (cam.image * 2.0 ** F).clamp(0, 1), cam.image)
            out.append(replace(cam, image=img))
            params.append({"field": kind, "std_log2": float(F[torch.as_tensor(m)].std())})
    for cam, c in zip(out, clean):
        CLEAN[cam.name] = c.detach().cpu()
    return out, params
''', 'PATCH_MODES_V5')

# ---- self test (CPU, about a minute) ----
import importlib, sys, tempfile, torch, numpy as np
import frecon.gaussians as G, frecon.data.photometric as ph
importlib.reload(G); importlib.reload(ph)
import frecon.cags as C; importlib.reload(C)
import frecon.train as T; importlib.reload(T)
sys.path.insert(0, 'scripts')
import phase0; importlib.reload(phase0)
from frecon.camera import Camera
from frecon.render import get_rasterizer
ras, bg = get_rasterizer('torch'), torch.ones(3)
def look(i, n=6):
    a = 2 * np.pi * i / n
    c = torch.tensor([3 * np.sin(a), 0.3 * (i % 2), 3 * np.cos(a)], dtype=torch.float32)
    f = -c / c.norm(); r = torch.cross(torch.tensor([0., 1., 0.]), f, dim=0); r = r / r.norm()
    R = torch.stack([r, torch.cross(f, r, dim=0), f]); return R, -R @ c
cams, masks = [], []
gimg = torch.Generator().manual_seed(1)
for i in range(6):
    R, t_ = look(i)
    img = torch.ones(32, 32, 3); img[8:24, 8:24] = torch.rand(16, 16, 3, generator=gimg) * 0.6 + 0.2
    m = np.zeros((32, 32), bool); m[8:24, 8:24] = True
    cams.append(Camera(R=R, t=t_, fx=40., fy=40., cx=16., cy=16., W=32, H=32, image=img, name=f'v5c{i}')); masks.append(m)
for kind in ('poly', 'blobs'):
    pc, prm = ph.perturb_views(cams, masks, {'field': kind, 'stops': 1.0, 'seed': 0})
    assert all(torch.equal(pc[k].image[~torch.as_tensor(masks[k])], cams[k].image[~torch.as_tensor(masks[k])]) for k in range(6))
    assert prm[0]['std_log2'] > 0.05, prm[0]
pac, _ = ph.perturb_views(cams, masks, {'field': 'blobs', 'stops': 1.0, 'seed': 0, 'exposure_stops': 1.0, 'wb': 0.15, 'gamma': 0.2})
pa_, _ = ph.perturb_views(cams, masks, {'exposure_stops': 1.0, 'wb': 0.15, 'gamma': 0.2, 'seed': 0})
pc, _ = ph.perturb_views(cams, masks, {'field': 'blobs', 'stops': 1.0, 'seed': 0})
assert not torch.equal(pac[0].image, pc[0].image) and not torch.equal(pac[0].image, pa_[0].image), 'P-AC = P-A then P-C'
assert torch.equal(ph.CLEAN[pac[0].name], cams[0].image), 'clean image kept'
tmp = Path(tempfile.mkdtemp())
np.save(tmp / 'gt.npy', (np.random.default_rng(0).uniform(-0.15, 0.15, (3000, 3))).astype(np.float32))
np.save(tmp / 'init.npy', (np.random.default_rng(1).uniform(-0.5, 0.5, (600, 3))).astype(np.float32))
init = {'type': 'sfm', 'points': str(tmp / 'init.npy'), 'n_points': 10, 'half_extent': 1.0}
base = dict(iterations=60, densify_from=5, densify_until=50, densify_every=5, opacity_reset_every=1000, log_every=0,
            gt_ref=str(tmp / 'gt.npy'), stat_every=10, sh_degree=1)
def run(name, extra, ini=init):
    d = tmp / name / 'seed_0'; d.mkdir(parents=True)
    cfg = {'train': dict(base, **extra), 'init': ini}
    tr = phase0.train_base(cfg, 0, pc, ras, bg, 'cpu', d / 'base.pt')
    return tr, d
# 1) logging run (vanilla trajectory), saves a fork checkpoint with the lineage state
par, d = run('log', dict(cags_mode='log', cags_log_steps=[10, 30, 60], save_full=[20]))
z = np.load(d / 'cags_log.npz')
assert z['F'].shape[1] == len(C.FEATS) and len(z['y_geo']) > 0 and np.isfinite(z['F'].astype(np.float32)).all()
assert (z['delta'] >= 0).all() and z['delta'].max() > 0, 'compensation label'
assert par._aux['birth'].numel() == par.model.n and par._aux['gen'].max() >= 1, 'lineage'
print('logged rows', len(z['y_geo']), '| off-surface', int((z['y_geo'] == 1).sum()), '| joint', int(z['y_joint'].sum()))
# 2) forks from the same checkpoint: oracle with each action
fk = {'type': 'fork', 'ckpt': str(d / 'fork_00020.pt')}
ck = T.Trainer.load(str(d / 'fork_00020.pt'), T.TrainConfig(**base), ras, bg, 'cpu')
assert ck._aux is not None and ck._aux['birth'].numel() == ck.model.n
res = {}
for act in ('gate', 'att', 'mod', 'full'):
    tr, dd = run(f'or_{act}', dict(cags_mode='oracle_geo', cags_actions=act, cags_from=20, cags_every=10), fk)
    h = list(__import__('csv').DictReader(open(dd / 'cags_hist.csv')))
    res[act] = (tr.model.n, max(int(r['n_active']) for r in h))
    assert tr._aux['risk'].numel() == tr.model.n
print('oracle forks (n Gaussians, max active):', res)
ctl, _ = run('ctl', dict(cags_mode='log', cags_log_steps=[]), fk)
# 3) opacity cap: after an att run, Gaussians with full strength have opacity <= cap
tr, _ = run('att2', dict(cags_mode='oracle_geo', cags_actions='att', cags_from=20, cags_every=10, cags_tau_s=0.0), fk)
C.post_step(tr, None)   # the last probe may have changed the risk after the final step
m_ = C._strength(tr); sel = m_ > 0
assert bool(sel.any())
assert bool((tr.model.opacities.detach().reshape(-1)[sel] <= (1 - m_[sel]) ** 2 + 1e-4).all()), 'opacity cap'
# 4) detector mode with a random (untrained) detector, and the rule baseline
det = C.Detector(list(range(len(C.FEATS))))
torch.save({'idx': list(range(len(C.FEATS))), 'state': det.state_dict()}, tmp / 'det.pt')
run('det', dict(cags_mode='det', cags_model=str(tmp / 'det.pt'), cags_from=20, cags_every=10, cags_budget=0.1), fk)
run('rule', dict(cags_mode='rule', cags_from=20, cags_every=10), fk)
print('PATCH 8 OK (CAGS probe / labels / lineage, oracle + detector + rule modes, gate / att / mod, P-B / P-C)')

## 3. Dữ liệu, renderer, kiểm tra probe trên GPU, hình minh hoạ P-B/P-C

In [ ]:
import yaml, json, time, glob, re, torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
DTU15 = (24, 37, 40, 55, 63, 65, 69, 83, 97, 105, 106, 110, 114, 118, 122)
FOLDS = [DTU15[i::5] for i in range(5)]            # 5 fold × 3 scan, cố định
FOLD_OF = {s: i for i, f in enumerate(FOLDS) for s in f}
SB = (63, 83)                                      # oracle pilot (v4)
SE = (63, 83, 65, 114)                             # ablation
def ready(s):
    r = Path(f'data/dtu/scan{s}')
    return (r / 'sfm_v8.npy').exists() and (r / 'ref_points_strict80.npy').exists()
AVAIL = [s for s in DTU15 if ready(s)]
print('Đủ dữ liệu:', AVAIL, '| thiếu:', [s for s in DTU15 if s not in AVAIL])
print('Fold:', FOLDS)
GSPLAT_OK = False
try:
    from frecon.render import get_rasterizer
    from frecon.train import TrainConfig, Trainer
    from floater_metrics import load_eval_scene
    tras, gras, bg0 = get_rasterizer('torch'), get_rasterizer('gsplat'), torch.ones(3, device='cuda')
    rd0 = 'runs/v2_none_std_dtu_scan63_v8'
    cfg0 = yaml.safe_load(open(f'{rd0}/config.yaml', encoding='utf-8'))
    _, tc0, _ = load_eval_scene(cfg0, 'cuda', 'strict', 0.8, 16)
    m0 = Trainer.load(f'{rd0}/seed_0/base.pt', TrainConfig(**cfg0['train']), tras, bg0, 'cuda').model
    with torch.no_grad():
        a, b = m0.render(tras, tc0[0], bg0)['image'], m0.render(gras, tc0[0], bg0)['image']
    ps = float(-10 * torch.log10(((a - b) ** 2).mean().clamp_min(1e-12)))
    GSPLAT_OK = ps > 30
    print(f'gsplat: PSNR giữa hai renderer {ps:.1f} dB')
except Exception as e:
    print('gsplat lỗi:', repr(e)[:300])
STD_RENDERER = 'gsplat' if GSPLAT_OK else 'torch'
print('Renderer huấn luyện:', STD_RENDERER)

In [ ]:
# Probe trên mô hình thật (scan63, run v2 gốc) với ảnh P-B: thời gian, đặc trưng hữu hạn, nhãn; và hình P-B / P-C
import frecon.cags as C
from frecon.data import load_scene
import frecon.data.photometric as ph
ras = get_rasterizer(STD_RENDERER)
sc = dict(cfg0['scene'], n_train_views=8)
figs = []
for lvl, pcfg in (('pb', dict(field='poly', stops=1.0, vignette=0.5, seed=0)), ('pc', dict(field='blobs', stops=1.0, seed=0))):
    trc, _, _, info = load_scene(dict(sc, photometric=pcfg), ras, bg0, 'cuda')
    figs.append((lvl, trc, info))
trc = figs[0][1]
tr0 = Trainer.load(f'{rd0}/seed_0/base.pt', TrainConfig(**dict(cfg0['train'], cags_mode='log',
                   gt_ref='data/dtu/scan63/ref_points_strict80.npy')), ras, bg0, 'cuda')
C.ensure_aux(tr0)
torch.cuda.synchronize(); t0 = time.time()
Fp, raw, lab = C.probe(tr0, trc, with_labels=True)
torch.cuda.synchronize()
print(f'probe: {tr0.model.n} Gaussian, 8 view, {time.time() - t0:.2f} s; đặc trưng hữu hạn: {bool(torch.isfinite(Fp).all())}')
print(f"y_geo: off {int((lab['y_geo'] == 1).sum())}, on {int((lab['y_geo'] == 0).sum())}, bất định {int((lab['y_geo'] < 0).sum())}; "
      f"tau_delta {lab['tau_delta']:.3f}; y_joint {int(lab['y_joint'].sum())}")
fig, axs = plt.subplots(3, 4, figsize=(14, 8))
for c in range(4):
    k = c * 2
    axs[0, c].imshow(ph.CLEAN[trc[k].name].numpy()); axs[0, c].set_title(f'sạch, view {k}')
    for r, (lvl, cams_, info) in enumerate(figs, 1):
        axs[r, c].imshow(cams_[k].image.cpu().numpy()); axs[r, c].set_title(f"{lvl}: std log2 {info['photometric'][k]['std_log2']:.2f}")
for ax in axs.flat: ax.axis('off')
fig.tight_layout(); fig.savefig(f'{RES}/perturbations_scan63.png', dpi=110); plt.show()
del tr0, Fp, raw, lab; torch.cuda.empty_cache()

## 4. Cấu hình
Tên run: `runs/v5_<mức>_std_<biến thể>_dtu_scan<s>_v8`. Các run có `gt_ref` ghi `gaussian_stats.csv` (khối lượng độ mờ ngoài
bề mặt theo thời gian) và `cags_hist.csv`; run có `cags_log_steps` ghi `cags_log.npz`. GT chỉ dùng để ghi log, gán nhãn và
cho oracle.

In [ ]:
SEEDS = (0, 1)
LV = {'none': None,
      'pa': dict(exposure_stops=1.0, wb=0.15, gamma=0.20, seed=0),
      'pb': dict(field='poly', stops=1.0, vignette=0.5, seed=0),
      'pc': dict(field='blobs', stops=1.0, seed=0),
      'pac': dict(field='blobs', stops=1.0, seed=0, exposure_stops=1.0, wb=0.15, gamma=0.20)}
template = Path('configs/dtu_scan24_v8.yaml').read_text(encoding='utf-8')
STD = dict(iterations=10000, densify_from=500, densify_until=5000, densify_every=100, opacity_reset_every=3000,
           sh_degree=3, max_gaussians=300000)
LOGS = [100, 250, 500, 1000, 2000, 2900, 4000, 5000, 7000, 10000]

def make(s, level, tag, train=None, init=None, seeds=SEEDS):
    base = yaml.safe_load(template.replace('scan24', f'scan{s}').replace('stl024', f'stl{s:03d}'))
    cfg = dict(base, out_dir=f'runs/v5_{level}_std_{tag}_dtu_scan{s}_v8', seeds=list(seeds), skip_signals=True,
               renderer=STD_RENDERER)
    cfg['scene'] = dict(base['scene'], n_train_views=8)
    if LV[level]:
        cfg['scene']['photometric'] = dict(LV[level])
    cfg['init'] = init or dict(base['init'], type='sfm', points=f'data/dtu/scan{s}/sfm_v8.npy')
    cfg['train'] = dict(base['train'], **STD, **(train or {}))
    path = f"configs/{Path(cfg['out_dir']).name}.yaml"
    yaml.safe_dump(cfg, open(path, 'w', encoding='utf-8'), sort_keys=False)
    return path

def gt(s): return dict(gt_ref=f'data/dtu/scan{s}/ref_points_strict80.npy', gt_tau=0.1, stat_every=100)
def fork(s, level, t0): return dict(type='fork', ckpt=f'runs/v5_{level}_std_van_dtu_scan{s}_v8/seed_{{seed}}/fork_{t0:05d}.pt')
def J(paths, seeds=SEEDS): return [(p, k) for p in paths for k in seeds]
def cfg_of(p): return yaml.safe_load(open(p, encoding='utf-8'))
def out_dir(p): return cfg_of(p)['out_dir']
def is_done(p, k): return os.path.exists(f'{out_dir(p)}/seed_{k}/base.pt')
def dep_ok(p, k):
    c = cfg_of(p)
    ini, mdl = c['init'], c['train'].get('cags_model', '')
    return ((ini.get('type') != 'fork' or os.path.exists(ini['ckpt'].replace('{seed}', str(k))))
            and (not mdl or os.path.exists(mdl)))

# Phần A (v5a): 3DGS gốc + ghi đặc trưng/nhãn. scan63, 83 lưu checkpoint fork cho phần B.
EXPS = {'A_van': J([make(s, l, 'van', dict(gt(s), cags_mode='log', cags_log_steps=LOGS, cags_log_max=10000,
                                          save_full=[100, 250, 500] if s in SB else []))
                    for s in sorted(AVAIL, key=lambda s: s not in SB) for l in LV])}
# Phần B (v4): oracle, fork tại t0 từ run gốc cùng mức, seed 0
ORV = {'ctl': dict(cags_mode='log', cags_log_steps=[]),
       **{f'geo_{a}': dict(cags_mode='oracle_geo', cags_actions=a) for a in ('gate', 'att', 'mod', 'full')},
       'cmp_full': dict(cags_mode='oracle_cmp', cags_actions='full')}
EXPS['B_oracle'] = J([make(s, l, f'or{t0}_{v}', dict(gt(s), cags_from=t0, cags_every=100, **kw), fork(s, l, t0), seeds=(0,))
                      for s in SB for l in ('none', 'pb', 'pc') for t0 in (100, 250, 500) for v, kw in ORV.items()], seeds=(0,))
# B2: oracle với nhãn sửa (d > 0.1 là ngoài bề mặt bất kể d/σ_n), fork tại 100
EXPS['B2_oracle'] = J([make(s, l, f'or100_{v}2_full', dict(gt(s), cags_from=100, cags_every=100, cags_mode=f'oracle_{v}',
                                                         cags_actions='full'), fork(s, l, 100), seeds=(0,))
                       for s in SB for l in ('none', 'pc') for v in ('geo', 'cmp')], seeds=(0,))
for e, jobs in EXPS.items():
    print(f'{e:10s} {sum(is_done(*j) for j in jobs):3d}/{len(jobs)} đã có')

### 4b. Chốt tham số CAGS cho phần D/E/F
Đặt sau khi xem 7a (oracle). Theo v4 (2 scan, pc): oracle_cmp giảm ΔW 63% khi bật từ vòng 100, 46% từ 250, 17% từ 500, nên
mặc định T_ON = 100 (bật sớm và liên tục, nhất quán với E2-B); cả ba hành động, probe mỗi 100 vòng, ngân sách 20%. Chạy lại cell này sau khi đổi.

In [ ]:
T_ON, ACTIONS, K_PROBE, BUDGET = 100, 'full', 100, 0.2   # T_ON = 100 theo kết quả v4 (7a)
DET = f'{RES}/det'
def det_path(s, tag='r0', fs='AGPD'): return f'{DET}/{tag}_{fs}_fold{FOLD_OF[s]}.pt'
def cg(s, mode='det', tag='r0', fs='AGPD', **kw):
    d = dict(gt(s), cags_mode=mode, cags_from=T_ON, cags_every=K_PROBE, cags_actions=ACTIONS, cags_budget=BUDGET,
             cags_log_steps=LOGS, cags_log_max=5000)
    if mode == 'det':
        d['cags_model'] = det_path(s, tag, fs)
    d.update(kw)
    return d
GC = dict(appearance='gc_log', sat_mask=True)
MAIN_LV = ('none', 'pc', 'pac')
EXPS['D_main'] = J([make(s, l, t, tr) for s in AVAIL for l in MAIN_LV for t, tr in (
    ('gc', dict(gt(s), **GC)), ('cags', cg(s)), ('gccags', dict(cg(s), **GC)),
    ('orcmp', cg(s, 'oracle_cmp', cags_budget=1.0)), ('gcorcmp', dict(cg(s, 'oracle_cmp', cags_budget=1.0), **GC)))])
EXPS['E_abl'] = J([make(s, l, t, tr) for s in SE for l in MAIN_LV for t, tr in (
    ('cags_gate', cg(s, cags_actions='gate')), ('cags_att', cg(s, cags_actions='att')), ('cags_mod', cg(s, cags_actions='mod')),
    ('cags_geo', cg(s, fs='AG')), ('cags_nocmp', cg(s, fs='AGPDnocmp')), ('rule', cg(s, 'rule')),
    ('orgeo', cg(s, 'oracle_geo', cags_budget=1.0)), ('cags_k500', cg(s, cags_every=500)))]
                  + [make(s, 'pa', t, tr) for s in SE for t, tr in (
    ('gc', dict(gt(s), **GC)), ('cags', cg(s)), ('gccags', dict(cg(s), **GC)), ('orcmp', cg(s, 'oracle_cmp', cags_budget=1.0)))])
EXPS['F_dagger'] = J([make(s, l, t, tr) for s in AVAIL for l in MAIN_LV for t, tr in (
    ('cagsr1', cg(s, tag='r1')), ('gccagsr1', dict(cg(s, tag='r1'), **GC)))])
for e, jobs in EXPS.items():
    print(f'{e:10s} {sum(is_done(*j) for j in jobs):3d}/{len(jobs)} đã có')

## 5. Huấn luyện song song (chạy lại được)
Mỗi phần một cell. Hàng đợi chỉ khởi chạy lượt fork khi checkpoint cha đã có, và lượt `det` khi file detector đã có.

In [ ]:
free_gb = torch.cuda.mem_get_info()[0] / 2**30
WORKERS = max(1, min(int(free_gb // 10), (os.cpu_count() or 8) - 1, 16))
print(f'GPU trống {free_gb:.0f} GB, CPU {os.cpu_count()} -> WORKERS = {WORKERS}')

def run_jobs(jobs, workers):
    env = dict(os.environ, PYTHONIOENCODING='utf-8', OMP_NUM_THREADS='2', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    pending = [j for j in jobs if not is_done(*j)]
    print(f'{len(pending)} lượt cần chạy, {len(jobs) - len(pending)} đã có', flush=True)
    running, failed, t0 = [], [], time.time()
    while pending or running:
        while len(running) < workers:
            nxt = next((j for j in pending if dep_ok(*j)), None)
            if nxt is None:
                break
            pending.remove(nxt); p, k = nxt
            d = f'{out_dir(p)}/seed_{k}'; os.makedirs(d, exist_ok=True)
            log = open(f'{d}/train.log', 'a')
            pr = subprocess.Popen([sys.executable, '-W', 'ignore', 'scripts/phase0.py', '--config', p, '--seeds', str(k),
                                   '--base_only'], stdout=log, stderr=subprocess.STDOUT, env=env)
            running.append((pr, p, k, log, time.time()))
        if not running:
            print('Không còn lượt nào chạy được (thiếu checkpoint cha hoặc detector):', pending[:5]); break
        time.sleep(15)
        for item in list(running):
            pr, p, k, log, ts = item
            if pr.poll() is None:
                continue
            log.close(); running.remove(item)
            ok = pr.returncode == 0 and is_done(p, k)
            if not ok:
                failed.append((p, k))
            print(f'[{(time.time() - t0) / 60:6.1f} phút] {"OK  " if ok else "LỖI"} {Path(p).stem} seed {k} '
                  f'({(time.time() - ts) / 60:.1f} phút) | chờ {len(pending)}, đang chạy {len(running)}', flush=True)
    print(f'xong sau {(time.time() - t0) / 60:.0f} phút; lỗi: {failed or "không"}')
    if failed:
        print('Xem log:', f'{out_dir(failed[0][0])}/seed_{failed[0][1]}/train.log')
    return failed

In [ ]:
# 5A – 3DGS gốc có ghi đặc trưng (v5a). scan63, 83 được xếp trước để phần B có checkpoint sớm.
failed = run_jobs(EXPS['A_van'], WORKERS)

In [ ]:
# 5B – oracle can thiệp (v4); B2: chạy lại oracle với nhãn đã sửa (LABEL_V2), mốc 100, mức none và pc
failed = run_jobs(EXPS['B_oracle'] + EXPS['B2_oracle'], WORKERS)

In [ ]:
# 5C – huấn luyện detector (v5b): 5 fold leave-scan-out; ablation nhóm đặc trưng; heuristic; cross-corruption
logs = 'runs/v5_*_std_van_dtu_scan*_v8/seed_*/cags_log.npz'
print(len(glob.glob(logs)), 'file log')
fold_args = [','.join(map(str, f)) for f in FOLDS]
!python -W ignore scripts/cags_train.py --logs "{logs}" --out {DET} --folds {' '.join(fold_args)} --sets A AG AGP AGPD --tag r0 --baselines
!python -W ignore scripts/cags_train.py --logs "{logs}" --out {DET} --folds {' '.join(fold_args)} --sets AGPD --train_levels none pa pb --test_levels pc pac --tag xc
!python -W ignore scripts/cags_train.py --logs "{logs}" --out {DET} --folds {' '.join(fold_args)} --sets AGPD --train_levels none pa pb pc --test_levels pac --tag xac
# biến thể bỏ đầu bù trừ (q = p_geo) dùng cho ablation E
for f in range(5):
    ck = torch.load(f'{DET}/r0_AGPD_fold{f}.pt', weights_only=False); ck['use_cmp'] = False
    torch.save(ck, f'{DET}/r0_AGPDnocmp_fold{f}.pt')

In [ ]:
# 5C-cal – hiệu chỉnh ngưỡng rủi ro của detector (không huấn luyện lại, vài phút).
# Pilot đầu: rủi ro trung bình ~0.02, chỉ < 0.02% Gaussian vượt tau_s = 0.5 -> CAGS gần như không tác động.
# tau = ngưỡng thấp nhất mà Gaussian bị gắn cờ trên dữ liệu huấn luyện có precision (ngoài bề mặt) >= 0.8;
# khi chạy, q được đổi thành 0.5 q / tau (q = tau -> bắt đầu tác động, q = 2 tau -> tác động tối đa).
logs = 'runs/v5_*_std_van_dtu_scan*_v8/seed_*/cags_log.npz'
fold_args = [','.join(map(str, f)) for f in FOLDS]
for f in range(5):
    ck = torch.load(f'{DET}/r0_AGPD_fold{f}.pt', weights_only=False); ck['use_cmp'] = False
    torch.save(ck, f'{DET}/r0_AGPDnocmp_fold{f}.pt')
!python -W ignore scripts/cags_train.py --logs "{logs}" --out {DET} --folds {' '.join(fold_args)} --sets AG AGPD AGPDnocmp --tag r0 --calibrate_only --cal_prec 0.8

In [ ]:
# 5D-pilot – scan63, 83, seed 0 và 1 (dao động giữa hai seed của van lên tới 3 lần, một seed không đủ để kết luận).
# Run cags / gccags của pilot đầu (detector chưa hiệu chỉnh) bị xoá một lần cùng kết quả đánh giá của chúng, rồi chạy lại.
MARK = f'{RES}/.pilot_cal_redo'
if not os.path.exists(MARK):
    gone = [f'v5_{l}_std_{t}_dtu_scan{s}_v8' for s in SB for l in MAIN_LV for t in ('cags', 'gccags')]
    for g in gone:
        if os.path.exists(f'runs/{g}'): shutil.rmtree(f'runs/{g}'); print('xoá runs/' + g)
    fe = f'{RES}/v5_final.csv'
    if os.path.exists(fe):
        ev = pd.read_csv(fe); ev[~ev.run.isin(gone)].to_csv(fe, index=False)
    open(MARK, 'w').write('done')
pilot = [(p, k) for p, k in EXPS['D_main'] if any(f'scan{s}_' in p for s in SB)]
failed = run_jobs(pilot, WORKERS)

In [ ]:
# 5D – CAGS và đối chứng trên 15 scan (dùng tham số ở 4b)
failed = run_jobs(EXPS['D_main'], WORKERS)

In [ ]:
# 5E – ablation trên 4 scan
failed = run_jobs(EXPS['E_abl'], WORKERS)

In [ ]:
# 5F – DAgger: huấn luyện lại detector với trạng thái thu từ run CAGS (phần D), rồi chạy lại CAGS
logs1 = 'runs/v5_*_std_cags_dtu_scan*_v8/seed_*/cags_log.npz'
print(len(glob.glob(logs)), '+', len(glob.glob(logs1)), 'file log')
!python -W ignore scripts/cags_train.py --logs "{logs}" "{logs1}" --out {DET} --folds {' '.join(fold_args)} --sets AGPD --tag r1
failed = run_jobs(EXPS['F_dagger'], WORKERS)

In [ ]:
for e, jobs in EXPS.items():
    print(f'{e:10s} {sum(is_done(*j) for j in jobs):3d}/{len(jobs)}')
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

## 6. Đánh giá (renderer PyTorch, bề mặt lọc chặt, 16 view kiểm định chung)
`fm3`: W_float, PSNR trực tiếp và căn chỉnh, Chamfer/F-score cho mọi run v5 có `base.pt` (chỉ đánh giá run mới, kết quả cũ
được giữ trong `v5_final.csv`). Gộp `gaussian_stats.csv` và `cags_hist.csv`.

In [ ]:
def eval_parallel(dirs, tag, script, extra, workers, seeds=('0', '1')):
    groups = {}
    for d in dirs:
        sc = cfg_of(f'{d}/config.yaml')['scene']; groups.setdefault((sc['root'], sc['n_train_views']), []).append(d)
    buckets = [[] for _ in range(min(workers, len(groups)))]
    for k in sorted(groups, key=lambda k: -len(groups[k])):
        min(buckets, key=len).extend(groups[k])
    procs = []
    for i, b in enumerate(buckets):
        out = f'{RES}/{tag}_part{i}.csv'; log = open(f'{RES}/{tag}_part{i}.log', 'w')
        procs.append((subprocess.Popen([sys.executable, '-W', 'ignore', script, '--run_dirs', *b, '--seeds', *seeds,
                                        '--out', out, *extra], stdout=log, stderr=subprocess.STDOUT), out, log))
    t0 = time.time()
    for pr, out, log in procs:
        pr.wait(); log.close(); print(f'{out}: exit {pr.returncode} ({(time.time() - t0) / 60:.1f} phút)', flush=True)
    parts = [pd.read_csv(o) for _, o, _ in procs if os.path.exists(o)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

EW = max(1, min(WORKERS, 12))
has_ck = lambda d: bool(glob.glob(f'{d}/seed_*/base.pt'))
dirs = sorted(d for d in glob.glob('runs/v5_*_dtu_scan*_v8') if has_ck(d))
old = pd.read_csv(f'{RES}/v5_final.csv') if os.path.exists(f'{RES}/v5_final.csv') else pd.DataFrame(columns=['run', 'seed'])
old = old[old['wfloat_01_cov'].notna()] if 'wfloat_01_cov' in old else old.iloc[0:0]   # đánh giá lại run chưa có W_cov
done = set(zip(old.run, old.seed.astype(int))) if len(old) else set()
todo = [d for d in dirs if any((Path(d).name, int(Path(s).name.split('_')[1])) not in done
                               for s in glob.glob(f'{d}/seed_*') if os.path.exists(f'{s}/base.pt'))]
print(len(dirs), 'run có checkpoint;', len(todo), 'run cần đánh giá')
if todo:
    new = eval_parallel(todo, 'v5_new', 'scripts/fm3.py', [], EW)
    ev = pd.concat([old[~old.run.isin([Path(d).name for d in todo])], new], ignore_index=True)
    ev.to_csv(f'{RES}/v5_final.csv', index=False)
for name, pat in (('v5_gstats', 'gaussian_stats.csv'), ('v5_cags_hist', 'cags_hist.csv')):
    rows = []
    for f in glob.glob(f'runs/v5_*/seed_*/{pat}'):
        g = pd.read_csv(f); g['run'] = Path(f).parts[-3]; g['seed'] = int(Path(f).parts[-2].split('_')[1]); rows.append(g)
    if rows:
        pd.concat(rows, ignore_index=True).to_csv(f'{RES}/{name}.csv', index=False)
print('đã ghi', [f for f in os.listdir(RES) if f.endswith('.csv')])

## 7. Phân tích và tiêu chí quyết định
`Δ` = giá trị mức nhiễu trừ mức none **cùng scan, seed và cùng biến thể**. Mức giảm = 1 − ΔW / ΔW_đối chứng.

In [ ]:
from fm3 import parse
from scipy.stats import wilcoxon
df = pd.read_csv(f'{RES}/v5_final.csv')
df = pd.concat([df.drop(columns=[c for c in ('prefix', 'level', 'tags') if c in df]), df.run.apply(parse).apply(pd.Series)], axis=1)
df['tags'] = df['tags'].fillna('').str.replace('_std_', '', regex=False)
df['scan'] = df.run.str.extract(r'scan(\d+)')[0].astype(int)
def add_delta(col):
    none = df[df.level == 'none'].groupby(['tags', 'scan', 'seed'])[col].mean()
    return df.apply(lambda r: r[col] - none.get((r.tags, r.scan, r.seed), np.nan), axis=1)
for c, n in (('wfloat_01', 'dW'), ('wfloat_01_cov', 'dWc'), ('chamfer', 'dCh'), ('fscore_010', 'dF'), ('psnr', 'dPSNR')):
    if c in df: df[n] = add_delta(c)
df.to_csv(f'{RES}/v5_with_delta.csv', index=False)
pd.set_option('display.width', 230); pd.set_option('display.max_rows', 500)
def summ(sub, by):
    extra = dict(dWc=('dWc', 'mean')) if 'dWc' in sub else {}
    return sub.groupby(by).agg(**extra, W=('wfloat_01', 'mean'), dW=('dW', 'mean'), up=('dW', lambda x: f'{(x > 0).sum()}/{x.notna().sum()}'),
                               Ch=('chamfer', 'mean'), dCh=('dCh', 'mean'), F=('fscore_010', 'mean'), dF=('dF', 'mean'),
                               psnr=('psnr', 'mean'), nG=('n_gaussians', 'mean')).round(4)
print(df.groupby(['tags', 'level']).size().unstack(fill_value=0).to_string())

In [ ]:
# 7a – oracle (v4): mức giảm ΔW so với ctl cùng mốc fork; chọn mốc và hành động cho 4b
b = df[df.tags.str.startswith('or') & df.tags.str.contains('_')].copy()
b['t0'] = b.tags.str.extract(r'or(\d+)_')[0].astype(int); b['var'] = b.tags.str.replace(r'or\d+_', '', regex=True)
print(summ(b, ['t0', 'var', 'level']).to_string())
pert = b[b.level != 'none']
print('\nΔW theo scan:\n', pert.pivot_table(index=['level', 't0', 'var'], columns='scan', values='dW').round(4).to_string())
wc = [c for c in ('wfloat_005', 'wfloat_01') if c in b]
print('\nW_float ở hai ngưỡng (mức nhiễu − none cũng biến thể):')
for c in wc:
    nn = b[b.level == 'none'].set_index(['scan', 't0', 'var'])[c]
    d_ = pert.set_index(['scan', 't0', 'var', 'level'])[c] - nn.reindex(pert.set_index(['scan', 't0', 'var']).index).values
    print(c, '\n', d_.groupby(['level', 't0', 'var']).mean().unstack('var').round(4).to_string())
# mức giảm = 1 − mean(ΔW) / mean(ΔW_ctl) (tỉ số của trung bình, không phải trung bình của tỉ số);
# không tính khi ΔW_ctl < 0.02 (nhiễu không tạo đủ floater để đo)
agg = pert.groupby(['level', 't0', 'var']).dW.mean().unstack('var')
red = 1 - agg.drop(columns='ctl').div(agg['ctl'], axis=0)
red[agg['ctl'] < 0.02] = np.nan
print('\nMức giảm ΔW so với ctl (tỉ số của trung bình; NaN = ΔW_ctl < 0.02):\n', red.round(2).to_string())
fnone = b[b.level == 'none'].pivot_table(index=['scan', 't0'], columns='var', values='fscore_010')
print('\nF-score mức none, chênh so với ctl:\n', fnone.drop(columns='ctl').sub(fnone['ctl'], axis=0).groupby('t0').mean().round(3).to_string())
fp = pert.groupby(['level', 't0', 'var']).fscore_010.mean().unstack('var')
dF = fp.drop(columns='ctl').sub(fp['ctl'], axis=0)
print('\nF-score mức nhiễu, chênh so với ctl:\n', dF.round(3).to_string())
lv_ok = [l for l in ('pb', 'pc') if l in red.index.get_level_values(0) and red.xs(l)['cmp_full'].notna().any()]
print('Mức nhiễu đo được (ΔW_ctl ≥ 0.02):', lv_ok or 'không có')
if lv_ok:
    best = red.loc[lv_ok, 'cmp_full'].groupby('t0').mean().idxmax()
    ok = all(red.loc[(l, best), 'cmp_full'] >= 0.3 and dF.loc[(l, best), 'cmp_full'] >= -0.02 for l in lv_ok)
    print(f'Mốc tốt nhất của oracle_cmp: {best}; ' + ', '.join(f'{l}: giảm {red.loc[(l, best), "cmp_full"]:.2f}, '
          f'dF {dF.loc[(l, best), "cmp_full"]:+.3f}' for l in lv_ok))
    print('=> ĐẠT trên các mức đo được' if ok else '=> CHƯA ĐẠT', '(chỉ 2 scan × 1 seed: kết luận sơ bộ)')
gst = pd.read_csv(f'{RES}/v5_gstats.csv'); gst = gst[gst.run.str.contains('_or|_van_') & gst.run.str.contains('scan63|scan83')]
gst = pd.concat([gst, gst.run.apply(parse).apply(pd.Series)], axis=1)
fig, axs = plt.subplots(1, 3, figsize=(16, 4))
for ax, lvl in zip(axs, ('none', 'pb', 'pc')):
    for v, c in (('_std_van', 'k'), ('_std_or250_ctl', 'C0'), ('_std_or250_geo_full', 'C1'), ('_std_or250_cmp_full', 'C3'),
                 ('_std_or100_cmp_full', 'C4'), ('_std_or500_cmp_full', 'C2')):
        g = gst[(gst.tags == v) & (gst.level == lvl)].groupby('step').mass_off.mean()
        if len(g): ax.plot(g.index, g.values, color=c, label=v.replace('_std_', ''))
    ax.set_title(f'{lvl}: tỉ lệ khối lượng độ mờ ngoài bề mặt'); ax.grid(alpha=0.3); ax.set_xlabel('vòng lặp')
axs[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(f'{RES}/v4_oracle_mass.png', dpi=140); plt.show()

In [ ]:
# 7a-bis – chẩn đoán: W_float(0.1) của run oracle phân theo nhãn GT của Gaussian (đóng góp trên 16 view kiểm định).
# Kiểm tra giả thuyết: khi oracle_geo chặn Gaussian có nhãn 1, phần bù chuyển sang Gaussian lớn có tâm xa bề mặt
# nhưng d/σ_n < 1 (nhãn 0) hoặc bất định, nên không bị chặn.
import fm2, frecon.cags as C
from floater_metrics import load_eval_scene
rows = []
for s in SB:
    for lvl in ('pb', 'pc'):
        ev = None
        for t0 in (100, 500):
            for v in ('ctl', 'geo_full', 'cmp_full'):
                rd = f'runs/v5_{lvl}_std_or{t0}_{v}_dtu_scan{s}_v8'
                if not os.path.exists(f'{rd}/seed_0/base.pt'): continue
                cfg = cfg_of(f'{rd}/config.yaml')
                if ev is None: ev = load_eval_scene(cfg, 'cuda', 'strict', 0.8, 16)
                tr = Trainer.load(f'{rd}/seed_0/base.pt', TrainConfig(**cfg['train']), tras, bg0, 'cuda')
                d, dn, y = C.gt_labels(tr)
                cb = fm2.contrib(tr.model, ev[1], tras, bg0); tot = cb.sum()
                far = d > 0.1; smax = tr.model.scales.detach().max(1).values / tr.extent
                rows.append(dict(scan=s, level=lvl, t0=t0, var=v, n=tr.model.n, W=float(cb[far].sum() / tot),
                                 W_nhan1=float(cb[far & (y == 1)].sum() / tot), W_nhan0=float(cb[far & (y == 0)].sum() / tot),
                                 W_batdinh=float(cb[far & (y < 0)].sum() / tot),
                                 scale_far_p90=float(smax[far].quantile(0.9)) if far.any() else np.nan))
                del tr; torch.cuda.empty_cache()
diag = pd.DataFrame(rows); diag.to_csv(f'{RES}/v4_diag_labels.csv', index=False)
print(diag.round(4).to_string())

In [ ]:
# 7v – trực quan hoá: ảnh kết xuất, sai số, bản đồ floater, độ sâu, rủi ro CAGS, đám mây Gaussian
# Dùng được cho mọi run v5 (van, or…, gc, cags, …). Hình lưu ở {RES}/vis.
import frecon.cags as C
from floater_metrics import load_eval_scene
from frecon.gaussians import nearest_dist
VIS = f'{RES}/vis'; os.makedirs(VIS, exist_ok=True)
_EV = {}

def _scene(cfg):
    key = cfg['scene']['root']
    if key not in _EV:
        _EV.clear(); _EV[key] = load_eval_scene(cfg, 'cuda', 'strict', 0.8, 16)
    return _EV[key]

def _load(s, level, tag, seed=0):
    rd = f'runs/v5_{level}_std_{tag}_dtu_scan{s}_v8'
    if not os.path.exists(f'{rd}/seed_{seed}/base.pt'):
        return None
    cfg = cfg_of(f'{rd}/config.yaml')
    tr = Trainer.load(f'{rd}/seed_{seed}/base.pt', TrainConfig(**cfg['train']), tras, bg0, 'cuda')
    return cfg, tr

@torch.no_grad()
def _maps(tr, cam, ref):
    m = tr.model
    d = nearest_dist(m.means.detach(), ref)
    far = (d > 0.1).float()
    z = (m.means.detach() @ cam.R.T + cam.t)[:, 2]
    a = getattr(tr, '_aux', None)
    risk = a['risk'] if a is not None and a['risk'].numel() == m.n else torch.zeros(m.n, device=z.device)
    rgb = m.render(tras, cam, bg0)['image'].clamp(0, 1)
    out = m.render(tras, cam, torch.zeros(3, device=z.device), features=torch.stack([far, z, risk.float()], 1))
    A = out['alpha']
    return dict(rgb=rgb, far=out['image'][..., 0], depth=out['image'][..., 1] / A.clamp_min(1e-6),
                risk=out['image'][..., 2] / A.clamp_min(1e-6), alpha=A, d=d)

def show(s, level, tags, view=None, seed=0, save=True):
    """Hàng = biến thể; cột = ảnh thật, kết xuất, |sai số|, bản đồ floater (tỉ lệ trọng số của Gaussian có d > 0.1),
    độ sâu, rủi ro CAGS (oracle: nhãn GT; det: đầu ra detector). view=None: view kiểm định có nhiều floater nhất ở biến thể đầu."""
    runs = [(t, _load(s, level, t, seed)) for t in tags]
    runs = [(t, r) for t, r in runs if r is not None]
    if not runs:
        print('không có run nào'); return
    views, test_cams, ref = _scene(runs[0][1][0])
    if view is None:
        tr0, wv = runs[0][1][1], []
        for c in test_cams:
            mm = _maps(tr0, c, ref); wv.append(float(mm['far'].sum() / mm['alpha'].sum().clamp_min(1e-6)))
        view = int(np.argmax(wv))
    cam = test_cams[view]
    M = [(t, _maps(r[1], cam, ref)) for t, r in runs]
    msk = torch.stack([mm['alpha'] > 0.5 for _, mm in M]).any(0)
    dv = torch.cat([mm['depth'][mm['alpha'] > 0.5] for _, mm in M])
    dlo, dhi = (float(torch.quantile(dv, 0.02)), float(torch.quantile(dv, 0.98))) if len(dv) else (0, 1)
    cols = ['ảnh thật (sạch)', 'kết xuất', '|sai số|', 'bản đồ floater', 'độ sâu', 'rủi ro CAGS']
    fig, axs = plt.subplots(len(M), 6, figsize=(19, 3.0 * len(M)), squeeze=False)
    for i, (t, mm) in enumerate(M):
        gt_ = cam.image.cpu().numpy(); rgb = mm['rgb'].cpu().numpy()
        w = float(mm['far'].sum() / mm['alpha'].sum().clamp_min(1e-6))
        ps = float(-10 * np.log10(((rgb - gt_) ** 2).mean() + 1e-12))
        axs[i, 0].imshow(gt_); axs[i, 1].imshow(rgb)
        axs[i, 2].imshow(np.abs(rgb - gt_).mean(-1), cmap='magma', vmin=0, vmax=0.3)
        im = axs[i, 3].imshow(mm['far'].cpu().numpy(), cmap='inferno', vmin=0, vmax=1)
        dep = mm['depth'].cpu().numpy().copy(); dep[mm['alpha'].cpu().numpy() < 0.5] = np.nan
        axs[i, 4].imshow(dep, cmap='turbo', vmin=dlo, vmax=dhi)
        axs[i, 5].imshow(mm['risk'].cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
        axs[i, 0].set_ylabel(t, fontsize=10)
        axs[i, 1].set_title(f'PSNR {ps:.2f} dB', fontsize=9); axs[i, 3].set_title(f'W_float view = {w:.3f}', fontsize=9)
        for j in range(6):
            axs[i, j].set_xticks([]); axs[i, j].set_yticks([])
            if i == 0: axs[i, j].set_title(cols[j] + ('' if j not in (1, 3) else f'\n{axs[i, j].get_title()}'), fontsize=9)
    fig.suptitle(f'scan{s}, mức {level}, view kiểm định {view}, seed {seed}', fontsize=12)
    fig.tight_layout()
    if save: fig.savefig(f'{VIS}/maps_scan{s}_{level}_v{view}.png', dpi=110)
    plt.show()
    return view

@torch.no_grad()
def cloud(s, level, tags, seed=0, min_op=0.05, save=True):
    """Tâm Gaussian chiếu lên hai trục chính của bề mặt tham chiếu: xám = gần bề mặt, đỏ = d > 0.1
    (kích thước điểm ~ độ mờ). Nhìn từ trên xuống (trục 1–2) và từ bên (trục 1–3)."""
    runs = [(t, _load(s, level, t, seed)) for t in tags]
    runs = [(t, r) for t, r in runs if r is not None]
    if not runs:
        print('không có run nào'); return
    _, _, ref = _scene(runs[0][1][0])
    c0 = ref.mean(0); _, _, V = torch.linalg.svd(ref - c0, full_matrices=False)
    P = lambda X: ((X - c0) @ V.T).cpu().numpy()
    rp = P(ref[torch.randperm(len(ref), device=ref.device)[:20000]])
    fig, axs = plt.subplots(2, len(runs), figsize=(4.2 * len(runs), 8), squeeze=False)
    for j, (t, (cfg, tr)) in enumerate(runs):
        m = tr.model; op = m.opacities.detach().reshape(-1); keep = op > min_op
        d = nearest_dist(m.means.detach(), ref); X = P(m.means.detach()[keep])
        far = (d[keep] > 0.1).cpu().numpy(); sz = (op[keep].cpu().numpy() * 6) ** 1.5
        for r, (a, b) in enumerate(((0, 1), (0, 2))):
            ax = axs[r, j]
            ax.scatter(rp[:, a], rp[:, b], s=0.2, c='#bbbbbb', alpha=0.4, linewidths=0)
            ax.scatter(X[~far, a], X[~far, b], s=sz[~far], c='#4a6fa5', alpha=0.35, linewidths=0)
            ax.scatter(X[far, a], X[far, b], s=sz[far], c='#d62728', alpha=0.7, linewidths=0)
            ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        mass = float(op[d > 0.1].sum() / op.sum())
        axs[0, j].set_title(f'{t}\n{int((d > 0.1).sum())}/{m.n} Gaussian xa, khối lượng độ mờ xa {mass:.2f}', fontsize=9)
    axs[0, 0].set_ylabel('trên xuống (trục 1–2)'); axs[1, 0].set_ylabel('bên cạnh (trục 1–3)')
    fig.suptitle(f'scan{s}, mức {level}: xám = STL, xanh = Gaussian gần bề mặt, đỏ = Gaussian có d > 0.1', fontsize=11)
    fig.tight_layout()
    if save: fig.savefig(f'{VIS}/cloud_scan{s}_{level}.png', dpi=110)
    plt.show()

# Mặc định: các biến thể D nếu đã chạy (5D-pilot), nếu chưa thì oracle v4. Đổi scan / mức / danh sách tuỳ ý.
if os.path.exists('runs/v5_pc_std_cags_dtu_scan63_v8/seed_0/base.pt'):
    TAGS = ['van', 'gc', 'cags', 'gccags', 'orcmp', 'gcorcmp']
    v = show(63, 'pc', TAGS); show(63, 'pac', TAGS, view=v)
    cloud(63, 'pac', TAGS)
else:
    TAGS = ['van', 'or100_ctl', 'or100_geo_full', 'or100_geo2_full', 'or100_cmp_full', 'or100_cmp2_full']
    v = show(63, 'pc', TAGS)
    show(63, 'none', ['van', 'or100_ctl', 'or100_geo2_full'], view=v)
    cloud(63, 'pc', ['or100_ctl', 'or100_geo_full', 'or100_geo2_full', 'or100_cmp2_full'])

In [ ]:
# 7b – detector (v5b): AUROC/AP theo fold, ablation nhóm đặc trưng, heuristic, cross-corruption
m0 = pd.read_csv(f'{DET}/r0_metrics.csv')
cols = [c for c in ('auroc_geo', 'ap_geo', 'ap_geo_w', 'ap_joint', 'ap_joint_w', 'spearman_delta') if c in m0]
print(m0.groupby(['level', 'model'])[cols].mean().round(3).to_string())
for t in ('xc', 'xac'):
    if os.path.exists(f'{DET}/{t}_metrics.csv'):
        x = pd.read_csv(f'{DET}/{t}_metrics.csv')
        print(f'\nCross-corruption {t} (train không có mức kiểm tra):\n', x[x.level != 'all'].groupby('model')[cols].mean().round(3).to_string())
g = m0[m0.level.isin(['pc', 'pac'])].groupby(['level', 'model']).ap_joint.mean().unstack()
ok = all(g.loc[l, 'AGPD'] > max(g.loc[l, 'AG'], g.loc[l, 'rule']) for l in ('pc', 'pac') if l in g.index)
print('\nAP_joint pc/pac:\n', g.round(3).to_string())
if 'joint_rate' in m0:
    print('\nTỉ lệ mẫu dương y_joint (AP của bộ dự đoán ngẫu nhiên):\n', m0[m0.model == 'AGPD'].groupby('level').joint_rate.mean().round(4).to_string())
fd = m0[m0.level.isin(['pc', 'pac']) & m0.model.isin(['AG', 'AGPD'])].pivot_table(index=['level', 'fold'], columns='model', values=['ap_joint', 'ap_geo'])
dd = pd.DataFrame({k: fd[(k, 'AGPD')] - fd[(k, 'AG')] for k in ('ap_joint', 'ap_geo')})
print('\nAGPD − AG theo fold (dương ở mọi fold mới coi là ổn định):\n', dd.round(4).to_string())
print('=> ĐẠT: đặc trưng trắc quang + động học thêm thông tin so với hình học và heuristic' if ok else '=> CHƯA ĐẠT tiêu chí C')

In [ ]:
# 7c – CAGS trên 15 scan (D): ΔW, hình học, PSNR; tiêu chí T1, T2, T4
VD = ['van', 'gc', 'cags', 'gccags', 'orcmp', 'gcorcmp'] + (['cagsr1', 'gccagsr1'] if (df.tags == 'cagsr1').any() else [])
SD = sorted(df[df.tags == 'cags'].scan.unique())   # chỉ các scan đã có run D (pilot: 63, 83) cho mọi biến thể
d = df[df.tags.isin(VD) & df.scan.isin(SD)]
print('Scan:', SD)
print(summ(d, ['tags', 'level']).to_string())
DWCOL = 'dWc' if 'dWc' in d else 'dW'   # chỉ số chính: ΔW trên vùng có tham chiếu (STL của DTU có lỗ)
print('Chỉ số chính:', DWCOL)
P = d[d.level != 'none'].pivot_table(index=['level', 'scan'], columns='tags', values=DWCOL)
N = d[d.level == 'none'].pivot_table(index='scan', columns='tags', values=['psnr', 'fscore_010'])
for lvl in ('pc', 'pac'):
    if lvl not in P.index.get_level_values(0): continue
    p = P.xs(lvl)
    r_c = (1 - p['cags'] / p['van']); r_o = (1 - p['orcmp'] / p['van'])
    t1 = (r_c / r_o.where(r_o > 0)).median()
    q4 = p[['gccags', 'gc', 'cags']].dropna()
    t4 = ((q4['gccags'] < q4['gc']) & (q4['gccags'] < q4['cags'])).mean()
    print(f'\n[{lvl}] giảm ΔW trung vị: CAGS {r_c.median():.2f}, oracle_cmp {r_o.median():.2f}, GC-ILA {(1 - p["gc"] / p["van"]).median():.2f}, '
          f'GC+CAGS {(1 - p["gccags"] / p["van"]).median():.2f}')
    print(f'   T1 CAGS / oracle (trung vị) = {t1:.2f} -> {"ĐẠT" if t1 >= 0.5 else "chưa đạt"}; '
          f'T4 GC+CAGS tốt hơn cả hai ở {t4:.0%} scan (n={len(q4)})')
    if 'gcorcmp' in p:
        g_c, g_o = (1 - p['gccags'] / p['gc']), (1 - p['gcorcmp'] / p['gc'])
        print(f'   Trên nền GC: GC+CAGS giảm thêm {g_c.median():.2f}, GC+oracle_cmp {g_o.median():.2f}; '
              f'tỉ số (trung vị) {(g_c / g_o.where(g_o > 0)).median():.2f}')
    for a_, b_ in (('cags', 'van'), ('gccags', 'gc')):
        x = p[[a_, b_]].dropna()
        if len(x) >= 6: print(f'   Wilcoxon ΔW {a_} < {b_}: p = {wilcoxon(x[a_], x[b_], alternative="less").pvalue:.4f} (n={len(x)})')
dps = (N[('psnr', 'cags')] - N[('psnr', 'van')]).median()
dfs = (N[('fscore_010', 'cags')] - N[('fscore_010', 'van')]).median()
print(f'\nT2 mức none: PSNR CAGS − gốc trung vị {dps:+.2f} dB -> {"ĐẠT" if dps >= -0.3 else "chưa đạt"}; F-score {dfs:+.3f}')
h = pd.read_csv(f'{RES}/v5_cags_hist.csv')
if 'act_off' in h:
    h = h[h.run.str.contains('_cags_') & h.act_off.notna()].copy()
    h['fp'] = h.act_on / (h.act_on + h.act_off).clip(lower=1)
    print('\nTỉ lệ Gaussian bị can thiệp mà thực ra gần bề mặt (chẩn đoán, theo nhãn GT) theo vòng lặp:\n',
          h.groupby('step').fp.mean().round(3).to_string())

In [ ]:
# 7c-bis – CAGS có thực sự can thiệp không? (1) rủi ro và tỉ lệ Gaussian bị tác động theo vòng lặp;
# (2) can thiệp đúng / sai theo nhãn GT; (3) ΔW theo scan và seed: chênh lệch giữa 2 seed của van = mức dao động ngẫu nhiên
h = pd.read_csv(f'{RES}/v5_cags_hist.csv')
h['tag'] = h.run.str.extract(r'_std_(.+?)_dtu')[0]
h['level'] = h.run.str.extract(r'v5_(none|pac|pa|pb|pc)_std')[0]
a = h[h.n_active.notna() & h.tag.isin(['cags', 'gccags', 'orcmp', 'gcorcmp', 'cagsr1', 'gccagsr1'])].copy()
a['act%'] = 100 * a.n_active / a.n; a['gate%'] = 100 * a.n_gate / a.n
ST = [100, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000]
print('(1) rủi ro trung bình, % Gaussian bị điều biến / chặn nhân bản (trung bình qua scan):')
print(a[a.step.isin(ST)].pivot_table(index=['tag', 'level'], columns='step', values=['risk_mean', 'act%', 'gate%']).round(3).to_string())
g = h[(h['diag'] == 1) & h.tag.isin(a.tag.unique())] if 'diag' in h else h.iloc[0:0]
if len(g):
    s_ = g.groupby(['tag', 'level'])[['n', 'n_off', 'n_joint', 'act_off', 'act_on', 'act_joint']].sum()
    s_['recall_off'] = s_.act_off / s_.n_off.clip(lower=1); s_['recall_joint'] = s_.act_joint / s_.n_joint.clip(lower=1)
    s_['prec_off'] = s_.act_off / (s_.act_off + s_.act_on).clip(lower=1)
    print('\n(2) tổng qua các bước ghi log: Gaussian bị tác động thuộc nhãn ngoài bề mặt (act_off), gần bề mặt (act_on), bù trừ (act_joint):')
    print(s_.round(3).to_string())
x = df[df.scan.isin(SD) & df.tags.isin(VD) & df.level.isin(['pc', 'pac'])]
print(f'\n(3) {DWCOL} theo scan, seed:')
print(x.pivot_table(index=['level', 'scan'], columns=['tags', 'seed'], values=DWCOL).round(4).to_string())

In [ ]:
# 7d – ablation (E): từng hành động, detector chỉ hình học, bỏ đầu bù trừ, heuristic, oracle_geo, probe 500; mức P-A
VE = ['van', 'cags', 'cags_gate', 'cags_att', 'cags_mod', 'cags_geo', 'cags_nocmp', 'rule', 'orgeo', 'orcmp', 'cags_k500', 'gc', 'gccags']
e = df[df.tags.isin(VE) & df.scan.isin(SE)]
print(summ(e, ['tags', 'level']).to_string())
P = e[e.level != 'none'].pivot_table(index=['level', 'scan', 'seed'], columns='tags', values='dW')
print('\nMức giảm ΔW so với gốc (trung bình theo scan, seed):\n', P.drop(columns='van').div(P['van'], axis=0).rsub(1).groupby('level').mean().round(2).T.to_string())
for b_ in ('cags_geo', 'rule', 'cags_nocmp'):
    x = P[['cags', b_]].dropna()
    if len(x) >= 6:
        print(f'T3 Wilcoxon ΔW cags < {b_}: p = {wilcoxon(x["cags"], x[b_], alternative="less").pvalue:.4f} (n={len(x)})')

In [ ]:
!ls -la {RES} {DET}